<!-- dd:dd-lesson-ar-02 -->

# ARENA 0.2 — Neural networks and ResNets

*PyTorch · `ar-02`*

Work through this with the **Delta Drills** side panel open. It picks what you practise, sends you to the cell, and records how it went — you do not need to read this notebook in order.


In [ ]:
# === Delta Drills ===
# Which lesson this notebook is, for the side panel. Nothing to run.
DD_LESSON_ID = "ar-02"


In [ ]:
#@title 🔧 Delta Drills checker — run me first { display-mode: "form" }
# Delta Drills — problem checker. Generated; see scripts/colab_grader.py.
#
# `dd_check(<problem id>)` runs your `solve` against the same cases the tutor
# grades with, and tells you which ones failed. It reads `solve` out of the
# notebook, so define it (run your cell) before you check.
import base64
import json
import sys
import zlib

import numpy as np

# Filled in by the generated cell that follows this source: {qid: {fn, cases}}.
_DD_TESTS = {}

# Where the ARENA digits fixture is fetched from, also filled in by that cell.
_DD_FIXTURE_URL = ""
_DD_FIXTURE_PATH = "/delta_numbers.npy"

_DD_RTOL = 1e-5
_DD_ATOL = 1e-6


def _dd_install_fixtures():
    """Make `np.load('/delta_numbers.npy')` work here the way it does in the app.

    24 of the einops drills are written against the ARENA digits image, and the
    bank refers to it by an absolute path the backend rewrites at grade time
    (`code_runner.CODE_PREAMBLE`). Nothing rewrote it in a notebook, so those
    problems could not run at all in Colab — not the checker, not the starter
    code the learner was sent there to fill in. Downloaded on first use, so the
    six notebooks that never touch it never pay for it.
    """
    import os
    import urllib.request

    original = np.load
    if getattr(original, "_dd_patched", False):
        return

    def _load(file, *args, **kwargs):
        if str(file) == _DD_FIXTURE_PATH and not os.path.exists(_DD_FIXTURE_PATH):
            if not _DD_FIXTURE_URL:
                raise FileNotFoundError(
                    "This drill needs the ARENA digits fixture and no source was "
                    "compiled into this notebook — regenerate it."
                )
            urllib.request.urlretrieve(_DD_FIXTURE_URL, _DD_FIXTURE_PATH)
        return original(file, *args, **kwargs)

    _load._dd_patched = True
    np.load = _load


def _dd_load(blob):
    """The test payload, deflated and base64'd.

    Not encryption and not pretending to be — it is one `zlib.decompress` away.
    It is compressed because the payload for a 84-problem notebook is ~80 KB of
    JSON, and out of sight because an expanded grader cell would otherwise sit
    in the notebook spelling out the expected answer to every problem below it.
    """
    return json.loads(zlib.decompress(base64.b64decode(blob)).decode("utf-8"))


def _dd_preflight_torch():
    """Import torch once, here, where a failure can still be explained.

    Every drill cell opens with `import torch as t`, so the learner meets a
    broken torch install as a traceback through torch's own internals — the one
    reported was `AttributeError: partially initialized module 'torch' has no
    attribute 'fx'` from `torch/_export/utils.py`, raised while evaluating a
    function's annotations. That message names neither the cause nor the cure,
    and it is not even the real error: it is what a LATER import sees after an
    earlier one died partway and left the half-built module in `sys.modules`.
    Python does unwind a failed import normally, but a torch that was swapped
    on disk under a running kernel (a `pip install` mid-session) or shadowed by
    a stray `torch.py` gets far enough in to be cached before it falls over.

    So: purge the wreckage and retry ONCE, which is the whole fix whenever the
    first failure was transient, and report what actually broke when it is not.
    Importing torch in this cell rather than lazily is safe now in a way the
    `_dd_tensor` comment below still guards against for the per-comparison
    path — the bank is 448/448 torch and every notebook imports it a few cells
    down, so there is no numpy-only notebook left to charge for it.

    Never raises: a checker that refuses to load over this would take the
    lesson down with the runtime.
    """

    def _purge():
        # Submodules too, and that is the whole point. Python drops only the
        # module that raised, so `torch` goes and a `torch._export` imported
        # seconds earlier STAYS — and the next `import torch` re-runs
        # `torch/__init__.py` straight back into that stale submodule, which
        # reaches for a `torch.fx` the half-built parent has not bound yet.
        # Leaving one behind reproduces the bug instead of clearing it.
        for name in [n for n in sys.modules if n == "torch" or n.startswith("torch.")]:
            del sys.modules[name]

    def _usable(mod):
        # `import torch` does NOT re-execute a module already in sys.modules,
        # so a corpse left by a failed import is imported "successfully" and
        # the error surfaces later, from the learner's own cell. Judge the
        # object, not the statement: a torch that finished has both of these.
        return hasattr(mod, "fx") and hasattr(mod, "__version__")

    cached = sys.modules.get("torch")
    if cached is not None and not _usable(cached):
        _purge()

    for attempt in (1, 2):
        try:
            import torch
            if not _usable(torch):
                raise ImportError(
                    "torch imported but is only partially initialised "
                    "(no .fx) — an earlier import in this session died partway"
                )
            return True
        except Exception as exc:
            if attempt == 1:
                _purge()
                continue
            print(
                "⚠️  This runtime cannot import PyTorch, so no drill in this "
                "notebook will run.\n"
                "    %s: %s\n"
                "    Fix: Runtime ▸ Disconnect and delete runtime, then reopen "
                "this notebook and run\n"
                "    this cell first. If it comes back, check for a file named "
                "torch.py in /content,\n"
                "    and re-run any pip install BEFORE anything imports torch."
                % (type(exc).__name__, exc)
            )
    return False


def _dd_tensor(value):
    # torch only if something already imported it. numpy-only notebooks must
    # not pay a torch import to compare two lists of ints.
    torch = sys.modules.get("torch")
    return torch is not None and isinstance(value, torch.Tensor)


def _dd_close(a, b):
    """Tolerance compare, but ONLY when a float or complex is involved.

    torch defaults to float32 where numpy defaults to float64 and honest
    answers differ in reduction order, so exact equality fails correct work.
    Integer and boolean results stay exact — an index answer (argmax, nonzero,
    searchsorted) must never be fudged by a tolerance. Returns None to mean
    "not a float comparison, use exact equality".
    """
    try:
        floaty = any(
            np.issubdtype(x.dtype, np.floating) or np.issubdtype(x.dtype, np.complexfloating)
            for x in (a, b)
        )
        if not floaty:
            return None
        if a.shape != b.shape:
            return False
        return bool(np.allclose(a, b, rtol=_DD_RTOL, atol=_DD_ATOL, equal_nan=True))
    except Exception:
        return None


def _dd_array_equal(a, b):
    close = _dd_close(a, b)
    if close is not None:
        return close
    return bool(np.array_equal(a, b))


def _dd_equal(a, b):
    if _dd_tensor(a) or _dd_tensor(b):
        try:
            a2 = a.detach().cpu().numpy() if _dd_tensor(a) else np.asarray(a)
            b2 = b.detach().cpu().numpy() if _dd_tensor(b) else np.asarray(b)
            return _dd_array_equal(a2, b2)
        except Exception:
            # dtypes numpy cannot hold (bfloat16, conj views): equal tensors
            # must not grade as unequal — ask torch itself.
            torch = sys.modules.get("torch")
            if torch is not None and isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor):
                try:
                    return bool(torch.equal(a.detach().cpu().resolve_conj(),
                                            b.detach().cpu().resolve_conj()))
                except Exception:
                    return False
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        return _dd_array_equal(np.asarray(a), np.asarray(b))
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        if len(a) != len(b):
            return False
        return all(_dd_equal(x, y) for x, y in zip(a, b))
    close = _dd_close(np.asarray(a), np.asarray(b))
    if close is not None:
        return close
    return bool(a == b)


def _dd_seed():
    # The same seed before the actual-side and the expected-side setup runs, for
    # BOTH rngs: setup executes twice, so an unseeded torch.rand in a fixture
    # would hand the two sides different data and fail an honest answer.
    np.random.seed(0)
    torch = sys.modules.get("torch")
    if torch is not None:
        torch.manual_seed(0)


def _dd_show(value, limit=320):
    try:
        text = repr(value)
    except Exception as exc:
        text = "<unrepresentable: %s>" % type(exc).__name__
    text = " ".join(text.split()) if len(text) > limit else text
    if len(text) > limit:
        text = text[: limit - 1] + "…"
    return text


def dd_check(question_id, verbose=True):
    """Grade the `solve` you just defined against this problem's cases.

    Returns True when every case passes. Prints which ones did not, with the
    fixture, what was expected and what came back — a failing grade should be
    evidence you can act on, not a verdict.
    """
    qid = str(question_id)
    entry = _DD_TESTS.get(qid)
    if entry is None:
        print("No checker for problem %s in this notebook." % qid)
        return False

    # The learner's namespace, not this function's: `solve` lives in the cell
    # they ran, and in Colab that is the caller's globals.
    try:
        env = sys._getframe(1).f_globals
    except Exception:
        env = globals()

    fn_name = entry.get("fn") or "solve"
    if fn_name not in env:
        print("❌ `%s` is not defined yet — run your solution cell first." % fn_name)
        return False

    cases = entry.get("cases") or []
    failures = []
    for i, case in enumerate(cases, 1):
        # A fresh copy per case: fixtures are exec'd, and exec'ing them into the
        # notebook's own globals would quietly overwrite whatever the learner
        # named `x` two cells ago.
        ns = dict(env)
        try:
            if case.get("setup_code"):
                _dd_seed()
                exec(case["setup_code"], ns)
            actual = eval(case["call"], ns)
            if case.get("assert_code"):
                exec(case["assert_code"], dict(ns, result=actual))
            expected_setup = case.get("expected_setup_code") or case.get("setup_code")
            if expected_setup:
                _dd_seed()
                exec(expected_setup, ns)
            expected = eval(case["expected_expr"], ns)
            if not _dd_equal(actual, expected):
                failures.append((i, case, _dd_show(expected), _dd_show(actual), ""))
        except Exception as exc:
            failures.append((i, case, "", "", "%s: %s" % (type(exc).__name__, exc)))

    total = len(cases)
    if not failures:
        print("✅ Problem %s — %d/%d cases passed." % (qid, total, total))
        return True

    print("❌ Problem %s — %d of %d cases failed." % (qid, len(failures), total))
    if verbose:
        for i, case, expected, actual, error in failures:
            print("\n  case %d" % i)
            if case.get("setup_code"):
                for line in case["setup_code"].strip().splitlines():
                    print("    given     %s" % line)
            print("    called    %s" % _dd_show_source(case.get("call", "")))
            if error:
                print("    raised    %s" % error)
            else:
                print("    expected  %s" % expected)
                print("    you got   %s" % actual)
    return False


def _dd_show_source(text, limit=160):
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

_DD_FIXTURE_URL = "https://raw.githubusercontent.com/AkiraTheSquid/arena-book-colab/main/ARENA_5.0/ch-1-foundations/numbers.npy"
_dd_preflight_torch()
_dd_install_fixtures()
_DD_TESTS = _dd_load(
    "..."
)
print("Delta Drills checker ready — 180 problems. Run dd_check(<problem number>) under any of them.")


<!-- dd:dd-kp-cnn-stride-views -->

## Strides and as_strided views

`cnn.stride-views`


<!-- dd:dd-seg-cnn-stride-views-0 -->

### An axis step is a storage step


A tensor's values live in one flat storage; its shape and strides say how to find them. Shape counts the positions along each axis, and `x.stride()` returns, per axis, how many storage slots to move when that axis's index goes up by one. A fresh `(2, 3)` matrix has strides `(3, 1)`: the next row is three slots on, the next column is one. Its transpose shares the same storage and simply swaps the strides to `(1, 3)` — no value moves.

`x.as_strided(shape, strides)` builds a new view on the same storage from a shape and strides you choose. The rule for choosing them is to derive every stride from the *source's* strides, never from an assumed contiguous layout, because the source may itself be a transposed or sliced view whose strides are not `(n, 1)`. A stride of zero is legal and useful: the index on that axis advances but the address does not, so one row is seen many times without being copied. Every index the new view can reach must land inside the source's storage; `as_strided` does not check for you.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.arange(6).reshape(2,3)\nprint(x.stride(), x.T.stride())\n# Hidden checks\nassert x.stride()==(3,1) and x.T.stride()==(1,3)\n', globals()), end='')


We build a view that shows a vector as two identical rows. The column stride is the vector's own stride, read from `v.stride(0)`; the row stride is zero, so both rows start at the same address.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nv=t.tensor([2.,5.,8.])\nrows=v.as_strided((2,3),(0,v.stride(0)))\nprint(rows)\n# Hidden checks\nassert rows.tolist()==[[2.,5.,8.],[2.,5.,8.]]\n', globals()), end='')




Because the view shares storage, writing to the source shows up in every apparent copy at once — there is only one `5.` in memory.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('v[1]=-1.\nprint(rows)\n# Hidden checks\nassert rows.tolist()==[[2.,-1.,8.],[2.,-1.,8.]]\n', globals()), end='')


<!-- dd:dd-q1217 -->

### Problem 1217 · faded — your turn

Return a transpose view using as_strided, shape (n,m). x: float (m,n). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1217)


In [ ]:
#@title 💡 Solution — Problem 1217
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.as_strided((x.shape[1],x.shape[0]),(x.stride(1),x.stride(0)))


We build a view that shows a vector as two identical rows. The column stride is the vector's own stride, read from `v.stride(0)`; the row stride is zero, so both rows start at the same address.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nv=t.tensor([2.,5.,8.])\nrows=v.as_strided((2,3),(0,v.stride(0)))\nprint(rows)\n# Hidden checks\nassert rows.tolist()==[[2.,5.,8.],[2.,5.,8.]]\n', globals()), end='')




Because the view shares storage, writing to the source shows up in every apparent copy at once — there is only one `5.` in memory.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('v[1]=-1.\nprint(rows)\n# Hidden checks\nassert rows.tolist()==[[2.,-1.,8.],[2.,-1.,8.]]\n', globals()), end='')


<!-- dd:dd-q1218 -->

### Problem 1218 · faded — your turn

Return the main diagonal as a view of length min(m,n), using as_strided. x: float (m,n). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1218)


In [ ]:
#@title 💡 Solution — Problem 1218
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.as_strided((min(x.shape),),(x.stride(0)+x.stride(1),))


<!-- dd:dd-seg-cnn-stride-views-1 -->

### Build the view, then compute


Many operations are "select some addresses, then reduce". The view does the selecting: a diagonal, for instance, advances one row *and* one column per step, so its single stride is the sum of the two source strides, and its length is the shorter side. The reduction then does the arithmetic on whatever the view exposes. Keeping the two steps separate means the view can be checked by printing it before any number is computed.

The pattern extends to products. A matrix-vector product uses the same vector against every row of the matrix; a zero-stride view of the vector with the matrix's shape expresses that reuse, and `(x * vv).sum(dim=1)` then multiplies and reduces along the feature axis. The reason to reduce over `dim=1` is that the feature axis is the one whose entries were paired; the row axis must survive as one result per row. Treat overlapping views as read-only: a write through the expanded vector would land at one address and change several apparent positions.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.arange(1.,10.).reshape(3,3).T\ndiag=x.as_strided((3,),(x.stride(0)+x.stride(1),))\nprint(diag, diag.sum())\n# Hidden checks\nassert diag.tolist()==[1.,5.,9.] and diag.sum().item()==15\n', globals()), end='')


We compute a matrix-vector product with no `@`. First the view: the vector expanded to the matrix's shape with a zero row stride, so every row of the view is the same vector.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[1.,2.],[3.,4.]])\nv=t.tensor([2.,-1.])\nexpanded=v.as_strided(x.shape,(0,v.stride(0)))\nprint(expanded)\n# Hidden checks\nassert expanded.tolist()==[[2.,-1.],[2.,-1.]]\n', globals()), end='')




Then the arithmetic: multiply elementwise and sum across the feature axis. The result matches `x @ v`, one number per row.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print((x*expanded).sum(dim=1), x@v)\n# Hidden checks\nassert (x*expanded).sum(dim=1).tolist()==[0.,2.] and t.equal((x*expanded).sum(dim=1),x@v)\n', globals()), end='')


<!-- dd:dd-q1219 -->

### Problem 1219 · faded — your turn

Return m copies of v as a view, shape (m,n), using as_strided. x: float (m,n); v: float (n,). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x,v):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1219)


In [ ]:
#@title 💡 Solution — Problem 1219
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,v):
    return v.as_strided(x.shape,(0,v.stride(0)))


We compute a matrix-vector product with no `@`. First the view: the vector expanded to the matrix's shape with a zero row stride, so every row of the view is the same vector.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[1.,2.],[3.,4.]])\nv=t.tensor([2.,-1.])\nexpanded=v.as_strided(x.shape,(0,v.stride(0)))\nprint(expanded)\n# Hidden checks\nassert expanded.tolist()==[[2.,-1.],[2.,-1.]]\n', globals()), end='')




Then the arithmetic: multiply elementwise and sum across the feature axis. The result matches `x @ v`, one number per row.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print((x*expanded).sum(dim=1), x@v)\n# Hidden checks\nassert (x*expanded).sum(dim=1).tolist()==[0.,2.] and t.equal((x*expanded).sum(dim=1),x@v)\n', globals()), end='')


<!-- dd:dd-q1220 -->

### Problem 1220 · faded — your turn

Return x multiplied by v, shape (m,), using only as_strided, elementwise arithmetic and sum. x: float (m,n); v: float (n,). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x,v):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1220)


In [ ]:
#@title 💡 Solution — Problem 1220
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,v):
    vv=v.as_strided(x.shape,(0,v.stride(0)))
    return (x*vv).sum(dim=1)


<!-- dd:dd-q1221 -->

### Problem 1221 · independent

Return the diagonal one step above the main diagonal as a view of length min(m,n-1), using as_strided. x: float (m,n). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1221)


In [ ]:
#@title 💡 Solution — Problem 1221
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    y=x[:,1:]
    return y.as_strided((min(y.shape),),(y.stride(0)+y.stride(1),))


<!-- dd:dd-q1222 -->

### Problem 1222 · independent

Return the outer product of v with itself, shape (n,n), using only as_strided views (stride zero) and elementwise multiplication. v: float (n,). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(v):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1222)


In [ ]:
#@title 💡 Solution — Problem 1222
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(v):
    n=v.shape[0]
    rows=v.as_strided((n,n),(v.stride(0),0))
    cols=v.as_strided((n,n),(0,v.stride(0)))
    return rows*cols


<!-- dd:dd-q1223 -->

### Problem 1223 · independent

Return every other column of x as a view, starting at column zero; use as_strided. x: float (m,n). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1223)


In [ ]:
#@title 💡 Solution — Problem 1223
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.as_strided((x.shape[0],(x.shape[1]+1)//2),(x.stride(0),2*x.stride(1)))


<!-- dd:dd-q1224 -->

### Problem 1224 · independent

Return a view of shape (m,n,2) whose last axis repeats each source value twice; use as_strided. x: float (m,n). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1224)


In [ ]:
#@title 💡 Solution — Problem 1224
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.as_strided((x.shape[0],x.shape[1],2),(x.stride(0),x.stride(1),0))


<!-- dd:dd-q1225 -->

### Problem 1225 · independent

Return the sum of the main diagonal as a scalar tensor, using as_strided and sum. x: float (m,n). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1225)


In [ ]:
#@title 💡 Solution — Problem 1225
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.as_strided((min(x.shape),),(x.stride(0)+x.stride(1),)).sum()


<!-- dd:dd-q1226 -->

### Problem 1226 · independent

Return the row Gram matrix, shape (m,m), using only as_strided, elementwise arithmetic and sum; do not use matmul or einsum. x: float (m,n). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1226)


In [ ]:
#@title 💡 Solution — Problem 1226
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    m,n=x.shape
    a=x.as_strided((m,m,n),(x.stride(0),0,x.stride(1)))
    b=x.as_strided((m,m,n),(0,x.stride(0),x.stride(1)))
    return (a*b).sum(dim=2)


<!-- dd:dd-q1227 -->

### Problem 1227 · independent

Return sums of consecutive width-two windows from each row, shape (m,n-1). n is at least two. Allowed view operation: as_strided. x: float (m,n). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1227)


In [ ]:
#@title 💡 Solution — Problem 1227
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    m,n=x.shape
    w=x.as_strided((m,n-1,2),(x.stride(0),x.stride(1),x.stride(1)))
    return w.sum(dim=-1)


<!-- dd:dd-q1228 -->

### Problem 1228 · independent

Return the feature Gram matrix, shape (n,n), using only as_strided, elementwise arithmetic and sum; do not use matmul or einsum. x: float (m,n). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1228)


In [ ]:
#@title 💡 Solution — Problem 1228
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    m,n=x.shape
    a=x.as_strided((m,n,n),(x.stride(0),x.stride(1),0))
    b=x.as_strided((m,n,n),(x.stride(0),0,x.stride(1)))
    return (a*b).sum(dim=0)


<!-- dd:dd-q1229 -->

### Problem 1229 · independent

Return scalar squared length of x multiplied by v, using only as_strided, elementwise arithmetic and sum; do not use matmul or einsum. x: float (m,n); v: float (n,). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x,v):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1229)


In [ ]:
#@title 💡 Solution — Problem 1229
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,v):
    vv=v.as_strided(x.shape,(0,v.stride(0)))
    y=(x*vv).sum(dim=1)
    return (y*y).sum()


#### Common mistakes

- **A `(m, n)` tensor always has strides `(n, 1)`.** Only when contiguous; a transpose or slice has different strides, and a view built from `(n, 1)` reads the wrong values.
- **A stride of zero copies the data.** It re-reads one address; nothing is allocated.
- **`as_strided` checks that the view fits.** It does not; an out-of-range view reads garbage or crashes.
- **The diagonal stride is `n + 1`.** It is `stride(0) + stride(1)`, which is `n + 1` only for a contiguous matrix.


<!-- dd:dd-kp-cnn-module-state -->

## Modules, parameters and buffers

`cnn.module-state`


<!-- dd:dd-seg-cnn-module-state-0 -->

### A module is a callable computation


PyTorch packages a computation as a class that inherits from `t.nn.Module`. You define one method, `forward(self, x)`, saying how an input becomes an output, and then you call the instance itself — `m(x)` — rather than `m.forward(x)`. The general shape is: `class Name(t.nn.Module):`, then `def forward(self, x):` indented inside it, then `return` the result. `self` is the instance the method was called on; every method of a class receives it first.

The reason to call `m(x)` and not `forward` directly is that the call goes through `Module.__call__`, which runs framework hooks around `forward` (checks, tracing, later on device placement). Calling `forward` skips all of that, so it works today and breaks the day a hook matters. The reason to make even a stateless operation a module is uniformity: a network is a tree of modules, and a tree whose leaves are all the same kind of thing can be printed, moved and saved with one call.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass Positive(t.nn.Module):\n    def forward(self,x):\n        return x.clamp(min=0)\nm=Positive()\nprint(m(t.tensor([-3.,2.])))\n# Hidden checks\nassert m(t.tensor([-3.,2.])).tolist()==[0.,2.]\n', globals()), end='')




This is ReLU: positive entries pass through, negative entries become zero. It has no state, which makes it the smallest possible module.


We build a module that halves its input. It needs no `__init__`, because nothing is stored: `forward` is the whole definition. Predict the output before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass Half(t.nn.Module):\n    def forward(self,x):\n        return x/2\nm=Half()\nprint(m(t.tensor([3.,-1.])))\n# Hidden checks\nassert m(t.tensor([3.,-1.])).tolist()==[1.5,-0.5]\n', globals()), end='')




The same instance works on any shape, because `forward` only used elementwise arithmetic. Constructing a second instance gives an independent object of the same class; with no state, the two are indistinguishable.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(Half()(t.tensor([[8.,0.]])))\n# Hidden checks\nassert Half()(t.tensor([[8.,0.]])).tolist()==[[4.,0.]]\n', globals()), end='')


<!-- dd:dd-q1169 -->

### Problem 1169 · faded — your turn

Return a parameter-free ReLU module. For any float tensor x, preserve positive entries and replace negative entries with zero.


In [ ]:
import torch as t

def solve():
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1169)


In [ ]:
#@title 💡 Solution — Problem 1169
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve():
    class ReLU(t.nn.Module):
        def forward(self,x):
            return x.clamp(min=0)
    return ReLU()


<!-- dd:dd-seg-cnn-module-state-1 -->

### Parameters are the state an optimizer learns


State that training should change is stored as a `t.nn.Parameter`, assigned to an attribute of `self` inside `__init__`. The procedure has three fixed lines: `def __init__(self):`, then `super().__init__()` as the first statement, then `self.weight = t.nn.Parameter(tensor)`. `forward` reads `self.weight` like any attribute.

`super().__init__()` runs the parent class's constructor — `Module`'s — which creates the bookkeeping that registration relies on. It must come first because the very next line, assigning a `Parameter` to `self`, is intercepted by that bookkeeping: `Module` notices a `Parameter` being attached and records it, so `m.parameters()` can later hand every trainable tensor to an optimizer. Assign before calling `super().__init__()` and there is nothing to record into; the assignment raises. A plain tensor attribute is NOT recorded — it is state the module has but the optimizer cannot see. Wrapping in `Parameter` is what makes the difference, and it also sets `requires_grad=True`, so gradients flow to it.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass Scale(t.nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.weight=t.nn.Parameter(t.tensor(1.5))\n    def forward(self,x):\n        return x*self.weight\nm=Scale()\nprint(m(t.tensor([2.,-4.])))\n# Hidden checks\nassert m(t.tensor([2.,-4.])).detach().tolist()==[3.,-6.]\n', globals()), end='')


We give a module a trainable offset and check that registration actually happened. First the module: the constructor stores one scalar `Parameter` named `bias`, and `forward` adds it.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass Shift(t.nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.bias=t.nn.Parameter(t.tensor(0.25))\n    def forward(self,x):\n        return x+self.bias\nm=Shift()\nprint(m(t.tensor([1.,2.])))\n# Hidden checks\nassert m(t.tensor([1.,2.])).detach().tolist()==[1.25,2.25]\n', globals()), end='')




Now the check that matters for training: `named_parameters()` lists what an optimizer would receive. The name is the attribute name, and the tensor requires a gradient because `Parameter` set that for us.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('names=[name for name,_ in m.named_parameters()]\nprint(names, m.bias.requires_grad)\n# Hidden checks\nassert names==["bias"] and m.bias.requires_grad\n', globals()), end='')


<!-- dd:dd-q1170 -->

### Problem 1170 · faded — your turn

Implement learned scaling: output x multiplied by weight. w: scalar float tensor. Return a new module; register its learnable scalar as weight. Its forward accepts float tensors x of arbitrary shape.


In [ ]:
import torch as t

def solve(w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1170)


In [ ]:
#@title 💡 Solution — Problem 1170
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w):
    class Scale(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.weight=t.nn.Parameter(w.clone())
        def forward(self,x):
            return x*self.weight
    return Scale()


We give a module a trainable offset and check that registration actually happened. First the module: the constructor stores one scalar `Parameter` named `bias`, and `forward` adds it.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass Shift(t.nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.bias=t.nn.Parameter(t.tensor(0.25))\n    def forward(self,x):\n        return x+self.bias\nm=Shift()\nprint(m(t.tensor([1.,2.])))\n# Hidden checks\nassert m(t.tensor([1.,2.])).detach().tolist()==[1.25,2.25]\n', globals()), end='')




Now the check that matters for training: `named_parameters()` lists what an optimizer would receive. The name is the attribute name, and the tensor requires a gradient because `Parameter` set that for us.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('names=[name for name,_ in m.named_parameters()]\nprint(names, m.bias.requires_grad)\n# Hidden checks\nassert names==["bias"] and m.bias.requires_grad\n', globals()), end='')


<!-- dd:dd-q1172 -->

### Problem 1172 · faded — your turn

Return an affine scalar module with optional bias. w is a scalar float tensor; b is a scalar float tensor or None. Register weight; register bias only when supplied.


In [ ]:
import torch as t

def solve(w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1172)


In [ ]:
#@title 💡 Solution — Problem 1172
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w,b):
    class Affine(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.weight=t.nn.Parameter(w.clone())
            self.bias=t.nn.Parameter(b.clone()) if b is not None else None
        def forward(self,x):
            y=x*self.weight
            return y if self.bias is None else y+self.bias
    return Affine()


<!-- dd:dd-seg-cnn-module-state-2 -->

### Buffers are state that is saved but not trained


Some state belongs to a model without being learned: a calibration offset, a fixed mask, and later the running statistics of BatchNorm. Register it with `self.register_buffer("name", tensor)` inside `__init__`, after `super().__init__()`, and read it as `self.name`. The rule for choosing: if an optimizer should update it, `Parameter`; if it must travel with the model but never be trained, buffer; if neither, it is not model state at all.

The reason a buffer is more than a plain attribute is that `Module` tracks it: `m.to(dtype)` and `m.to(device)` convert buffers along with parameters, and `state_dict()` saves them, so a reloaded model has the same offset it was trained with. A plain tensor attribute is left behind by all three. And unlike a `Parameter`, a buffer is not returned by `parameters()`, so the optimizer never touches it — which is exactly the contract.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass Offset(t.nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.register_buffer("offset",t.tensor(3.))\n    def forward(self,x):\n        return x+self.offset\nm=Offset()\nprint(m(t.tensor([1.,-1.])), list(m.parameters()))\n# Hidden checks\nassert m(t.tensor([1.,-1.])).tolist()==[4.,2.] and list(m.parameters())==[]\n', globals()), end='')


We confirm the two things a buffer promises: it follows the module through a dtype change, and it appears in the saved state. First the dtype: `m.to(t.float64)` converts the buffer even though it is not a parameter.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('m=Offset().to(t.float64)\nprint(m.offset.dtype)\n# Hidden checks\nassert m.offset.dtype==t.float64\n', globals()), end='')




Then the state dict, which is what `save` writes and `load_state_dict` reads. The buffer is in it under its registered name, and there is no parameter entry because the module has none.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('keys=list(m.state_dict())\nprint(keys)\n# Hidden checks\nassert keys==["offset"]\n', globals()), end='')


<!-- dd:dd-q1171 -->

### Problem 1171 · faded — your turn

Return a calibrated module: subtract fixed scalar b, then multiply by learned scalar w. Store weight as a Parameter and offset as a buffer; both inputs are scalar float tensors.


In [ ]:
import torch as t

def solve(w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1171)


In [ ]:
#@title 💡 Solution — Problem 1171
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w,b):
    class Calibrated(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.weight=t.nn.Parameter(w.clone())
            self.register_buffer("offset",b.clone())
        def forward(self,x):
            return (x-self.offset)*self.weight
    return Calibrated()


<!-- dd:dd-q1173 -->

### Problem 1173 · independent

Return a parameter-free module whose forward clips any float tensor x into [0,1]: entries below zero become zero, entries above one become one.


In [ ]:
import torch as t

def solve():
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1173)


In [ ]:
#@title 💡 Solution — Problem 1173
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve():
    class Clip(t.nn.Module):
        def forward(self,x):
            return x.clamp(min=0,max=1)
    return Clip()


<!-- dd:dd-q1174 -->

### Problem 1174 · independent

Return a module that multiplies x by a FIXED scalar taken from w: store it as a buffer named scale, so the module has no parameters.


In [ ]:
import torch as t

def solve(w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1174)


In [ ]:
#@title 💡 Solution — Problem 1174
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w):
    class FixedScale(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.register_buffer("scale",w.clone())
        def forward(self,x):
            return x*self.scale
    return FixedScale()


<!-- dd:dd-q1175 -->

### Problem 1175 · independent

Return a module that adds a trainable scalar offset to x: register the scalar from w as a parameter named bias.


In [ ]:
import torch as t

def solve(w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1175)


In [ ]:
#@title 💡 Solution — Problem 1175
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w):
    class Shift(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.bias=t.nn.Parameter(w.clone())
        def forward(self,x):
            return x+self.bias
    return Shift()


<!-- dd:dd-q1176 -->

### Problem 1176 · independent

Return a module that counts its forward calls in an integer buffer named calls (starting at zero) and returns x unchanged.


In [ ]:
import torch as t

def solve():
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1176)


In [ ]:
#@title 💡 Solution — Problem 1176
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve():
    class Counter(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.register_buffer("calls",t.tensor(0))
        def forward(self,x):
            self.calls += 1
            return x
    return Counter()


<!-- dd:dd-q1177 -->

### Problem 1177 · independent

Return a model containing one child module named scale, whose sole parameter weight starts at scalar tensor w. Output is the positive part of x times that weight. Parent parameter discovery must find scale.weight.


In [ ]:
import torch as t

def solve(w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1177)


In [ ]:
#@title 💡 Solution — Problem 1177
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w):
    class Scale(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.weight=t.nn.Parameter(w.clone())
        def forward(self,x):
            return x*self.weight
    class Model(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.scale=Scale()
        def forward(self,x):
            return self.scale(x).clamp(min=0)
    return Model()


<!-- dd:dd-q1178 -->

### Problem 1178 · independent

Return a module with a trainable vector parameter weight initialized from w (n,), whose forward returns the weighted sum of x (...,n) along its last axis, shape (...).


In [ ]:
import torch as t

def solve(w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1178)


In [ ]:
#@title 💡 Solution — Problem 1178
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w):
    class WeightedSum(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.weight=t.nn.Parameter(w.clone())
        def forward(self,x):
            return (x*self.weight).sum(dim=-1)
    return WeightedSum()


<!-- dd:dd-q1179 -->

### Problem 1179 · independent

Return a learned scaling module whose parameter weight (from w) is FROZEN: it stays a registered parameter but requires no gradient. Forward returns x times weight.


In [ ]:
import torch as t

def solve(w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1179)


In [ ]:
#@title 💡 Solution — Problem 1179
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w):
    class Frozen(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.weight=t.nn.Parameter(w.clone(),requires_grad=False)
        def forward(self,x):
            return x*self.weight
    return Frozen()


<!-- dd:dd-q1180 -->

### Problem 1180 · independent

Return a module using the same trainable scalar on both sides of a ReLU: positive part of x times weight, then multiplied by weight again. Register weight only once. w: scalar float tensor.


In [ ]:
import torch as t

def solve(w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1180)


In [ ]:
#@title 💡 Solution — Problem 1180
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w):
    class Scale(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.weight=t.nn.Parameter(w.clone())
        def forward(self,x):
            return (x*self.weight).clamp(min=0)*self.weight
    return Scale()


<!-- dd:dd-q1181 -->

### Problem 1181 · independent

Return a learned scaling module (parameter weight from w, forward x times weight) whose extra_repr reports the current weight as the string "weight=<value>", using the tensor's item().


In [ ]:
import torch as t

def solve(w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1181)


In [ ]:
#@title 💡 Solution — Problem 1181
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w):
    class Scale(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.weight=t.nn.Parameter(w.clone())
        def forward(self,x):
            return x*self.weight
        def extra_repr(self):
            return "weight="+str(self.weight.item())
    return Scale()


#### Common mistakes

- **Any tensor attribute is trainable.** Only a `t.nn.Parameter` is handed to the optimizer; a plain tensor on `self` is invisible to `parameters()`, `to()` and `state_dict()`.
- **`super().__init__()` is boilerplate that can go anywhere.** It creates the registries; a `Parameter` or buffer assigned before it has nowhere to go and raises.
- **Fixed state can be a plain attribute.** If it must move with the model or be saved, it is a buffer; otherwise it is lost on `to()` and on reload.
- **Using a parameter twice makes two copies.** Reading `self.weight` in two places shares one tensor, and both uses accumulate gradient into it; a second, independent value needs a second `Parameter`.


<!-- dd:dd-kp-cnn-linear-layer -->

## The linear layer

`cnn.linear-layer`


<!-- dd:dd-seg-cnn-linear-layer-0 -->

### Each output owns a weight row


A linear layer turns `in_features` numbers into `out_features` numbers. It stores one weight row per output, so the weight has shape `(out_features, in_features)`, and output `j` is the dot product of row `j` with the input, plus a bias `b[j]`. For a batch `x` of shape `(batch, in_features)` the whole thing is `x @ w.T + b`: transposing makes the shapes `(batch, in) @ (in, out)`, so the feature axes meet and the result is `(batch, out)`.

The reason the weight is stored as `(out, in)` rather than `(in, out)` is convention — PyTorch's `nn.Linear` does it, and every checkpoint you load will — and the transpose in the forward is the price. The reason the bias is `(out_features,)` and simply added is broadcasting: it lines up with the last axis of the product, so each output column gets its own offset on every example. Test shapes with a non-square weight; a square one lets an accidental missing transpose run silently with the wrong numbers.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[1.,2.]])\nw=t.tensor([[2.,0.],[0.,3.],[1.,-1.]])\nb=t.tensor([1.,-1.,0.])\ny=x@w.T+b\nprint(y)\n# Hidden checks\nassert y.tolist()==[[3.,5.,-1.]]\n', globals()), end='')


We run a layer with three inputs and two outputs on a batch of two examples. Row `0` of the weight sums the inputs, row `1` takes the first minus the last, and the bias shifts each output. Predict the `(2, 2)` result before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[1.,1.,1.],[2.,0.,-2.]])\nw=t.tensor([[1.,1.,1.],[1.,0.,-1.]])\nb=t.tensor([0.,10.])\ny=x@w.T+b\nprint(y)\n# Hidden checks\nassert y.tolist()==[[3.,10.],[0.,14.]]\n', globals()), end='')




Drop the transpose and the shapes do not meet: `(2, 3) @ (2, 3)` has no shared inner axis. Catching this at the shape level is the point of a non-square weight.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('try:\n    x@w\nexcept RuntimeError as e:\n    print("shape mismatch:", type(e).__name__)\n# Hidden checks\nassert (x@w.T).shape==(2,2)\n', globals()), end='')


<!-- dd:dd-q1185 -->

### Problem 1185 · faded — your turn

Return the affine output, shape (...,out_features). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1185)


In [ ]:
#@title 💡 Solution — Problem 1185
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return y


We run a layer with three inputs and two outputs on a batch of two examples. Row `0` of the weight sums the inputs, row `1` takes the first minus the last, and the bias shifts each output. Predict the `(2, 2)` result before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[1.,1.,1.],[2.,0.,-2.]])\nw=t.tensor([[1.,1.,1.],[1.,0.,-1.]])\nb=t.tensor([0.,10.])\ny=x@w.T+b\nprint(y)\n# Hidden checks\nassert y.tolist()==[[3.,10.],[0.,14.]]\n', globals()), end='')




Drop the transpose and the shapes do not meet: `(2, 3) @ (2, 3)` has no shared inner axis. Catching this at the shape level is the point of a non-square weight.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('try:\n    x@w\nexcept RuntimeError as e:\n    print("shape mismatch:", type(e).__name__)\n# Hidden checks\nassert (x@w.T).shape==(2,2)\n', globals()), end='')


<!-- dd:dd-q1186 -->

### Problem 1186 · faded — your turn

Return the affine output with zero bias, shape (...,out_features). x: float (...,in_features); w: float (out_features,in_features).


In [ ]:
import torch as t

def solve(x,w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1186)


In [ ]:
#@title 💡 Solution — Problem 1186
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w):
    return x@w.T


<!-- dd:dd-seg-cnn-linear-layer-1 -->

### Leading axes survive; only the last axis is transformed


The layer treats the last axis as features and every axis before it as "which example". A `(batch, seq, in_features)` input — a batch of token sequences, say — goes through the same `x @ w.T + b` and comes out `(batch, seq, out_features)`: the matrix product only touches the last axis of `x`, and the bias broadcasts along it. The general rule is that a linear layer never needs to know how many leading axes there are.

The reason this matters is that it is the same weights for every example and every position. Batching does not make a weight per example; every row of every leading axis is evidence about one shared map, which is what makes the parameters learnable from many examples at once. The weight is the layer's state, created once and reused on every call; `nn.Linear` registers it as a parameter and, when `bias=False`, stores no bias at all rather than a zero one — absent state, not a broadcast constant.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.ones(2,3,4); w=t.ones(5,4)\nprint((x@w.T).shape)\n# Hidden checks\nassert (x@w.T).shape==(2,3,5)\n', globals()), end='')


We push a single vector and a batch of vectors through the same layer and check that the batch result is just the vector results stacked. First the batch: two examples, two inputs, one output.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[1.,0.],[0.,1.]])\nw=t.tensor([[2.,3.]]); b=t.tensor([4.])\nprint(x@w.T+b)\n# Hidden checks\nassert (x@w.T+b).tolist()==[[6.],[7.]]\n', globals()), end='')




Now one example on its own, as a plain `(2,)` vector. The product is `(1,)` — the leading batch axis was never required — and it equals the first row of the batched answer.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(x[0]@w.T+b)\n# Hidden checks\nassert (x[0]@w.T+b).tolist()==[6.]\n', globals()), end='')


<!-- dd:dd-q1187 -->

### Problem 1187 · faded — your turn

Return positive parts of affine outputs, shape (...,out_features). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1187)


In [ ]:
#@title 💡 Solution — Problem 1187
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return y.clamp(min=0)


We push a single vector and a batch of vectors through the same layer and check that the batch result is just the vector results stacked. First the batch: two examples, two inputs, one output.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[1.,0.],[0.,1.]])\nw=t.tensor([[2.,3.]]); b=t.tensor([4.])\nprint(x@w.T+b)\n# Hidden checks\nassert (x@w.T+b).tolist()==[[6.],[7.]]\n', globals()), end='')




Now one example on its own, as a plain `(2,)` vector. The product is `(1,)` — the leading batch axis was never required — and it equals the first row of the batched answer.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(x[0]@w.T+b)\n# Hidden checks\nassert (x[0]@w.T+b).tolist()==[6.]\n', globals()), end='')


<!-- dd:dd-q1188 -->

### Problem 1188 · faded — your turn

Return the affine output averaged across output features, shape (...). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1188)


In [ ]:
#@title 💡 Solution — Problem 1188
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return y.mean(dim=-1)


<!-- dd:dd-q1189 -->

### Problem 1189 · independent

Return the affine output of x after centring each example: subtract every example's own mean over in_features first, shape (...,out_features). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1189)


In [ ]:
#@title 💡 Solution — Problem 1189
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    centred=x-x.mean(dim=-1,keepdim=True)
    return centred@w.T+b


<!-- dd:dd-q1190 -->

### Problem 1190 · independent

Return the affine outputs centred to zero mean within each example, shape (...,out_features). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1190)


In [ ]:
#@title 💡 Solution — Problem 1190
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return y-y.mean(dim=-1,keepdim=True)


<!-- dd:dd-q1191 -->

### Problem 1191 · independent

Return index of the largest affine output for every example, shape (...); ties choose first. x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1191)


In [ ]:
#@title 💡 Solution — Problem 1191
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return y.argmax(dim=-1)


<!-- dd:dd-q1192 -->

### Problem 1192 · independent

Return the affine output with all leading example axes combined, shape (number_of_examples,out_features). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1192)


In [ ]:
#@title 💡 Solution — Problem 1192
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return y.reshape(-1,w.shape[0])


<!-- dd:dd-q1193 -->

### Problem 1193 · independent

Return the mean affine output across all examples and positions, shape (out_features,). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1193)


In [ ]:
#@title 💡 Solution — Problem 1193
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return y.reshape(-1,w.shape[0]).mean(dim=0)


<!-- dd:dd-q1194 -->

### Problem 1194 · independent

Return the tied-weight autoencoder output, shape (...,in_features). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1194)


In [ ]:
#@title 💡 Solution — Problem 1194
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return y.clamp(min=0)@w


<!-- dd:dd-q1195 -->

### Problem 1195 · independent

Return total squared affine activation per example, shape (...). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1195)


In [ ]:
#@title 💡 Solution — Problem 1195
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return (y*y).sum(dim=-1)


<!-- dd:dd-q1196 -->

### Problem 1196 · independent

Return the residual tied-weight autoencoder output: reconstruction plus original input, shape (...,in_features). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1196)


In [ ]:
#@title 💡 Solution — Problem 1196
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return x+y.clamp(min=0)@w


<!-- dd:dd-q1197 -->

### Problem 1197 · independent

Return the difference between affine outputs on x and on zero input, shape (...,out_features). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1197)


In [ ]:
#@title 💡 Solution — Problem 1197
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return y-b


#### Common mistakes

- **The weight is `(in, out)` so that `x @ w` works.** PyTorch stores `(out, in)`; the transpose belongs in the forward.
- **The bias has one entry per example.** It has one per output feature and broadcasts over every leading axis.
- **A batch needs a loop over examples.** One matrix product handles every example and every leading position with the same weights.
- **`bias=False` means a bias of zeros.** It means no bias tensor exists; the state is absent, not constant.


<!-- dd:dd-kp-cnn-mlp -->

## A two-layer MLP for images

`cnn.mlp`


<!-- dd:dd-seg-cnn-mlp-0 -->

### Flatten each image, not the batch


An MLP reads a vector of features per example, but an image arrives as `(channels, height, width)`. The first step is to flatten each image into one vector while keeping examples apart: `x.reshape(x.shape[0], -1)` keeps the batch axis and merges everything after it into a single feature axis of length `channels * height * width`. The `-1` asks reshape to work out that length. The first weight matrix then has shape `(hidden, pixels)`, with `pixels` equal to the number of values in one image, and the hidden pre-activations are `flat @ w1.T + b1`, shape `(batch, hidden)`.

The reason to keep axis `0` is that the batch is not part of any image; flattening it too would glue neighbouring examples into one long vector and make the layer's input width depend on the batch size. The reason the flattening order does not matter for correctness is that the first layer learns a weight per feature position — whatever order the pixels arrive in, it is the same order every time.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.arange(12.).reshape(2,2,3)\nflat=x.reshape(x.shape[0],-1)\nprint(flat)\n# Hidden checks\nassert flat.tolist()==[[0.,1.,2.,3.,4.,5.],[6.,7.,8.,9.,10.,11.]]\n', globals()), end='')


We flatten a batch of three one-channel `2×2` images and feed them to a first layer with five hidden units. Each image has four values, so `pixels = 4` and the weight is `(5, 4)`. Predict the flattened shape before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.ones(3,1,2,2); w1=t.ones(5,4)\nflat=x.reshape(3,-1)\nprint(flat.shape)\n# Hidden checks\nassert flat.shape==(3,4)\n', globals()), end='')




The product keeps the batch of three and produces five hidden values per image. Every input was `1` and every weight was `1`, so each hidden value is the sum of four ones.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('h=flat@w1.T\nprint(h.shape, h[0])\n# Hidden checks\nassert h.shape==(3,5) and h[0].tolist()==[4.,4.,4.,4.,4.]\n', globals()), end='')


<!-- dd:dd-q1201 -->

### Problem 1201 · faded — your turn

Return image features, shape (b,pixels). x: images (b,...), float.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1201)


In [ ]:
#@title 💡 Solution — Problem 1201
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.reshape(x.shape[0],-1)


We flatten a batch of three one-channel `2×2` images and feed them to a first layer with five hidden units. Each image has four values, so `pixels = 4` and the weight is `(5, 4)`. Predict the flattened shape before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.ones(3,1,2,2); w1=t.ones(5,4)\nflat=x.reshape(3,-1)\nprint(flat.shape)\n# Hidden checks\nassert flat.shape==(3,4)\n', globals()), end='')




The product keeps the batch of three and produces five hidden values per image. Every input was `1` and every weight was `1`, so each hidden value is the sum of four ones.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('h=flat@w1.T\nprint(h.shape, h[0])\n# Hidden checks\nassert h.shape==(3,5) and h[0].tolist()==[4.,4.,4.,4.,4.]\n', globals()), end='')


<!-- dd:dd-q1202 -->

### Problem 1202 · faded — your turn

Return hidden pre-activation values, shape (b,hidden). x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1202)


In [ ]:
#@title 💡 Solution — Problem 1202
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1):
    return x.reshape(x.shape[0],-1)@w1.T+b1


<!-- dd:dd-seg-cnn-mlp-1 -->

### A nonlinearity between the layers makes a new representation


Two linear layers in a row, `w2 @ (w1 @ x)`, are together just one linear map, `(w2 @ w1) @ x`; stacking them adds parameters but no expressive power. An activation between them changes that. ReLU, `clamp(min=0)`, zeroes every negative hidden value, so different inputs switch different hidden units on and off and the second layer sees a different linear map in different regions of input space. The full forward is: flatten, first affine, ReLU, second affine.

The reason the last layer has no activation is that its outputs are logits — unnormalized class scores that the loss function consumes directly. A ReLU there would forbid negative scores and a softmax there would double-normalize once the loss applied its own. So the hidden layer gets the nonlinearity and the output layer stays affine. Inspect the hidden activations when debugging: a unit that is zero for every example is a dead unit, and the shape `(batch, hidden)` tells you at once whether the flattening went right.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[-1.,2.],[3.,-2.]])\nw=t.tensor([[1.,1.],[-1.,1.]])\nh=(x@w.T).clamp(min=0)\nprint(h)\n# Hidden checks\nassert h.tolist()==[[1.,3.],[1.,0.]]\n', globals()), end='')


We finish the network on those hidden activations: a second layer with two outputs turns `(batch, hidden)` into `(batch, classes)`. Note that the zeroed hidden unit in the second row contributes nothing, whichever weight multiplies it.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nh=t.tensor([[1.,3.],[1.,0.]])\nw2=t.tensor([[1.,-1.],[2.,1.]])\nlogits=h@w2.T\nprint(logits)\n# Hidden checks\nassert logits.tolist()==[[-2.,5.],[1.,2.]]\n', globals()), end='')




Now the same head applied to the hidden values *before* ReLU, to see what the nonlinearity changed. The second row differs, because its negative hidden unit now leaks through.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('raw=t.tensor([[1.,3.],[1.,-5.]])\nprint(raw@w2.T)\n# Hidden checks\nassert (raw@w2.T).tolist()==[[-2.,5.],[6.,-3.]]\n', globals()), end='')


<!-- dd:dd-q1203 -->

### Problem 1203 · faded — your turn

Return hidden activations after the positive-part nonlinearity, shape (b,hidden). x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1203)


In [ ]:
#@title 💡 Solution — Problem 1203
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    return h


We finish the network on those hidden activations: a second layer with two outputs turns `(batch, hidden)` into `(batch, classes)`. Note that the zeroed hidden unit in the second row contributes nothing, whichever weight multiplies it.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nh=t.tensor([[1.,3.],[1.,0.]])\nw2=t.tensor([[1.,-1.],[2.,1.]])\nlogits=h@w2.T\nprint(logits)\n# Hidden checks\nassert logits.tolist()==[[-2.,5.],[1.,2.]]\n', globals()), end='')




Now the same head applied to the hidden values *before* ReLU, to see what the nonlinearity changed. The second row differs, because its negative hidden unit now leaks through.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('raw=t.tensor([[1.,3.],[1.,-5.]])\nprint(raw@w2.T)\n# Hidden checks\nassert (raw@w2.T).tolist()==[[-2.,5.],[6.,-3.]]\n', globals()), end='')


<!-- dd:dd-q1204 -->

### Problem 1204 · faded — your turn

Return classifier logits, shape (b,classes). x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,); w2: (classes,hidden); b2: (classes,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1,w2,b2):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1204)


In [ ]:
#@title 💡 Solution — Problem 1204
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1,w2,b2):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    y=h@w2.T+b2
    return y


<!-- dd:dd-q1205 -->

### Problem 1205 · independent

Return the logits of the two-layer image classifier averaged over the batch, shape (classes,). x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,); w2: (classes,hidden); b2: (classes,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1,w2,b2):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1205)


In [ ]:
#@title 💡 Solution — Problem 1205
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1,w2,b2):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    return (h@w2.T+b2).mean(dim=0)


<!-- dd:dd-q1206 -->

### Problem 1206 · independent

Return predicted class indices, shape (b,); ties choose first. x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,); w2: (classes,hidden); b2: (classes,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1,w2,b2):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1206)


In [ ]:
#@title 💡 Solution — Problem 1206
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1,w2,b2):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    y=h@w2.T+b2
    return y.argmax(dim=1)


<!-- dd:dd-q1207 -->

### Problem 1207 · independent

Return mean hidden activation over the batch, shape (hidden,). x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1207)


In [ ]:
#@title 💡 Solution — Problem 1207
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    return h.mean(dim=0)


<!-- dd:dd-q1208 -->

### Problem 1208 · independent

Return number of active hidden features per example, shape (b,); positive values are active. x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1208)


In [ ]:
#@title 💡 Solution — Problem 1208
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    return (h>0).sum(dim=1)


<!-- dd:dd-q1209 -->

### Problem 1209 · independent

Return logits with the first hidden feature silenced, shape (b,classes). x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,); w2: (classes,hidden); b2: (classes,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1,w2,b2):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1209)


In [ ]:
#@title 💡 Solution — Problem 1209
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1,w2,b2):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    h[:,0]=0
    return h@w2.T+b2


<!-- dd:dd-q1210 -->

### Problem 1210 · independent

Return class probabilities, shape (b,classes). x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,); w2: (classes,hidden); b2: (classes,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1,w2,b2):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1210)


In [ ]:
#@title 💡 Solution — Problem 1210
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1,w2,b2):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    y=h@w2.T+b2
    z=y-y.max(dim=1,keepdim=True)[0]
    e=z.exp()
    return e/e.sum(dim=1,keepdim=True)


<!-- dd:dd-q1211 -->

### Problem 1211 · independent

Return the change in logits caused by removing the hidden nonlinearity, shape (b,classes): linear-only minus normal model. x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,); w2: (classes,hidden); b2: (classes,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1,w2,b2):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1211)


In [ ]:
#@title 💡 Solution — Problem 1211
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1,w2,b2):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    y=h@w2.T+b2
    plain=(flat@w1.T+b1)@w2.T+b2
    return plain-y


<!-- dd:dd-q1212 -->

### Problem 1212 · independent

Return the first hidden feature’s contribution to every logit, shape (b,classes), excluding output bias. x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,); w2: (classes,hidden). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1,w2):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1212)


In [ ]:
#@title 💡 Solution — Problem 1212
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1,w2):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    return h[:,0,None]*w2[None,:,0]


<!-- dd:dd-q1213 -->

### Problem 1213 · independent

Return logits relative to each example’s first class score, shape (b,classes). x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,); w2: (classes,hidden); b2: (classes,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1,w2,b2):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1213)


In [ ]:
#@title 💡 Solution — Problem 1213
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1,w2,b2):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    y=h@w2.T+b2
    return y-y[:,0,None]


#### Common mistakes

- **`x.reshape(-1)` flattens the images.** It flattens the whole batch into one vector; the batch axis must be kept.
- **Two linear layers learn more than one.** Without a nonlinearity between them they compose into a single linear map.
- **The output layer needs ReLU or softmax.** Logits are consumed raw by the loss; an activation there changes the model.
- **`pixels` is `height * width`.** It is every value in one image, channels included.


<!-- dd:dd-kp-cnn-convolution-1d -->

## 1-D convolution from strided windows

`cnn.convolution-1d`


<!-- dd:dd-seg-cnn-convolution-1d-0 -->

### One filter, many windows


A convolution scores every local patch of a signal with the same small filter. For a 1-D signal of width `W` and a kernel of length `K`, there are `W - K + 1` valid windows — every run of `K` consecutive values — and output `p` is the dot product of window `p` with the filter. The procedure is: build a view whose rows are the windows, multiply each row by the filter, sum along the kernel axis. The view is `x.as_strided((W-K+1, K), (stride, stride))`: moving to the next window and moving to the next value inside a window are both one storage step, so the two strides are equal, and no window is ever copied.

With channels and a batch the same idea gains axes. Input `x` is `(batch, in_channels, width)`, the filter bank `w` is `(out_channels, in_channels, kernel)`, and the window view is `(batch, in_channels, positions, kernel)`. Each output filter reads every input channel, so the einsum sums over the channel *and* kernel axes and keeps batch, output channel and position: `"b c p k, o c k -> b o p"`. The reason PyTorch's "convolution" does not flip the filter is that a learned filter can be learned flipped; the first weight meets the first value of each window, which is what the strided view gives for free.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([1.,2.,4.,8.])\nw=t.tensor([1.,-1.])\nwindows=x.as_strided((3,2),(1,1))\nprint((windows*w).sum(dim=1))\n# Hidden checks\nassert (windows*w).sum(dim=1).tolist()==[-1.,-2.,-4.]\n', globals()), end='')


We convolve one example with two input channels and a single output filter. First the window view: width `3` and kernel `2` give two positions, and the strides come from `x.stride()` rather than from assumptions.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nimport einops\nx=t.tensor([[[1.,2.,3.],[4.,5.,6.]]])\nw=t.tensor([[[1.,0.],[0.,1.]]])\nbs,cs,ws=x.stride()\nwindows=x.as_strided((1,2,2,2),(bs,cs,ws,ws))\nprint(windows[0,0], windows[0,1])\n# Hidden checks\nassert windows[0,0].tolist()==[[1.,2.],[2.,3.]] and windows[0,1].tolist()==[[4.,5.],[5.,6.]]\n', globals()), end='')




The filter takes the first value of channel `0` and the second value of channel `1`, so position `0` gives `1 + 5` and position `1` gives `2 + 6`. The einsum sums over `c` and `k` and keeps `b o p`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('y=einops.einsum(windows,w,"b c p k, o c k -> b o p")\nprint(y)\n# Hidden checks\nassert y.tolist()==[[[6.,8.]]]\n', globals()), end='')


<!-- dd:dd-q1233 -->

### Problem 1233 · faded — your turn

Return valid convolution with stride one and no padding. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel).


In [ ]:
import torch as t
import einops

def solve(x,w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1233)


In [ ]:
#@title 💡 Solution — Problem 1233
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w):
    b,c,width=x.shape
    o,_,k=w.shape
    win=x.as_strided((b,c,width-k+1,k),(x.stride(0),x.stride(1),x.stride(2),x.stride(2)))
    y=einops.einsum(win,w,"b c p k, o c k -> b o p")
    return y


We convolve one example with two input channels and a single output filter. First the window view: width `3` and kernel `2` give two positions, and the strides come from `x.stride()` rather than from assumptions.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nimport einops\nx=t.tensor([[[1.,2.,3.],[4.,5.,6.]]])\nw=t.tensor([[[1.,0.],[0.,1.]]])\nbs,cs,ws=x.stride()\nwindows=x.as_strided((1,2,2,2),(bs,cs,ws,ws))\nprint(windows[0,0], windows[0,1])\n# Hidden checks\nassert windows[0,0].tolist()==[[1.,2.],[2.,3.]] and windows[0,1].tolist()==[[4.,5.],[5.,6.]]\n', globals()), end='')




The filter takes the first value of channel `0` and the second value of channel `1`, so position `0` gives `1 + 5` and position `1` gives `2 + 6`. The einsum sums over `c` and `k` and keeps `b o p`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('y=einops.einsum(windows,w,"b c p k, o c k -> b o p")\nprint(y)\n# Hidden checks\nassert y.tolist()==[[[6.,8.]]]\n', globals()), end='')


<!-- dd:dd-q1234 -->

### Problem 1234 · faded — your turn

Return each filter’s valid output averaged over positions. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel).


In [ ]:
import torch as t
import einops

def solve(x,w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1234)


In [ ]:
#@title 💡 Solution — Problem 1234
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w):
    b,c,width=x.shape
    o,_,k=w.shape
    win=x.as_strided((b,c,width-k+1,k),(x.stride(0),x.stride(1),x.stride(2),x.stride(2)))
    y=einops.einsum(win,w,"b c p k, o c k -> b o p")
    return y.mean(dim=2)


<!-- dd:dd-seg-cnn-convolution-1d-1 -->

### Stride chooses positions; padding extends the signal


Two knobs change which windows exist. A stride `S` makes consecutive windows start `S` values apart instead of one, so the view's position stride becomes `S * stride` while the kernel stride stays `stride`. Padding `P` adds `P` known values — zeros, for convolution — at each end of the signal before any window is taken, so windows can be centred on the edges. With both, the output width is `1 + (W + 2P - K) // S`: the number of window starts that fit.

The reason padding must be a real, allocated tensor is that a view cannot address storage that does not exist; so make a zero tensor of the padded shape, write `x` into its middle with a slice assignment, and take the strides from the *padded* tensor, since that is what the windows read. The reason zero is the right padding value here is that it is neutral for the weighted sum: a zero times any weight adds nothing. It is not neutral for every operation — a maximum over negative values would be wrongly beaten by a zero — which the pooling lesson handles differently.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([1.,2.,3.])\npadded=t.zeros(5)\npadded[1:4]=x\nprint(padded)\n# Hidden checks\nassert padded.tolist()==[0.,1.,2.,3.,0.]\n', globals()), end='')


We take stride-2 windows of kernel `3` over that padded signal. The formula gives `1 + (5 - 3) // 2 = 2` windows, and the position stride is `2 * 1`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\npadded=t.tensor([0.,1.,2.,3.,0.])\nwindows=padded.as_strided((2,3),(2*padded.stride(0),padded.stride(0)))\nprint(windows)\n# Hidden checks\nassert windows.tolist()==[[0.,1.,2.],[2.,3.,0.]]\n', globals()), end='')




Both windows include a padding zero, which contributes nothing to the sum. With a filter of all ones, each output is the sum of the real values it covers.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('w=t.ones(3)\nprint((windows*w).sum(dim=1))\n# Hidden checks\nassert (windows*w).sum(dim=1).tolist()==[3.,5.]\n', globals()), end='')


<!-- dd:dd-q1235 -->

### Problem 1235 · faded — your turn

Return x with p zeros added at both ends of its width axis, preserving batch and channels. x: float (batch,in_channels,width); p: zeros added at each end.


In [ ]:
import torch as t

def solve(x,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1235)


In [ ]:
#@title 💡 Solution — Problem 1235
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,p):
    b,c,width=x.shape
    y=t.zeros((b,c,width+2*p),dtype=x.dtype)
    y[:,:,p:p+width]=x
    return y


We take stride-2 windows of kernel `3` over that padded signal. The formula gives `1 + (5 - 3) // 2 = 2` windows, and the position stride is `2 * 1`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\npadded=t.tensor([0.,1.,2.,3.,0.])\nwindows=padded.as_strided((2,3),(2*padded.stride(0),padded.stride(0)))\nprint(windows)\n# Hidden checks\nassert windows.tolist()==[[0.,1.,2.],[2.,3.,0.]]\n', globals()), end='')




Both windows include a padding zero, which contributes nothing to the sum. With a filter of all ones, each output is the sum of the real values it covers.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('w=t.ones(3)\nprint((windows*w).sum(dim=1))\n# Hidden checks\nassert (windows*w).sum(dim=1).tolist()==[3.,5.]\n', globals()), end='')


<!-- dd:dd-q1236 -->

### Problem 1236 · faded — your turn

Return convolution with stride s and symmetric zero padding p. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel); s: stride ≥ 1; p: zeros added at each end. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1236)


In [ ]:
#@title 💡 Solution — Problem 1236
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,width=x.shape
    o,_,k=w.shape
    xx=t.zeros((b,c,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+width]=x
    step=xx.stride()
    n=1+(xx.shape[2]-k)//s
    win=xx.as_strided((b,c,n,k),(step[0],step[1],s*step[2],step[2]))
    y=einops.einsum(win,w,"b c p k, o c k -> b o p")
    return y


<!-- dd:dd-q1237 -->

### Problem 1237 · independent

Return the valid convolution (stride one, no padding) summed over output channels, shape (batch,out_width). x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel).


In [ ]:
import torch as t
import einops

def solve(x,w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1237)


In [ ]:
#@title 💡 Solution — Problem 1237
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w):
    b,c,width=x.shape
    o,_,k=w.shape
    n=width-k+1
    step=x.stride()
    win=x.as_strided((b,c,n,k),(step[0],step[1],step[2],step[2]))
    return einops.einsum(win,w,"b c p k, o c k -> b p")


<!-- dd:dd-q1238 -->

### Problem 1238 · independent

Return valid convolution followed by positive-part activation. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel).


In [ ]:
import torch as t
import einops

def solve(x,w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1238)


In [ ]:
#@title 💡 Solution — Problem 1238
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w):
    b,c,width=x.shape
    o,_,k=w.shape
    win=x.as_strided((b,c,width-k+1,k),(x.stride(0),x.stride(1),x.stride(2),x.stride(2)))
    y=einops.einsum(win,w,"b c p k, o c k -> b o p")
    return y.clamp(min=0)


<!-- dd:dd-q1239 -->

### Problem 1239 · independent

Return each filter’s maximum valid activation over positions. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel).


In [ ]:
import torch as t
import einops

def solve(x,w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1239)


In [ ]:
#@title 💡 Solution — Problem 1239
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w):
    b,c,width=x.shape
    o,_,k=w.shape
    win=x.as_strided((b,c,width-k+1,k),(x.stride(0),x.stride(1),x.stride(2),x.stride(2)))
    y=einops.einsum(win,w,"b c p k, o c k -> b o p")
    return y.max(dim=2)[0]


<!-- dd:dd-q1240 -->

### Problem 1240 · independent

Return valid convolution with stride s and no padding. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel); s: stride ≥ 1. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1240)


In [ ]:
#@title 💡 Solution — Problem 1240
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s):
    b,c,width=x.shape
    o,_,k=w.shape
    win=x.as_strided((b,c,width-k+1,k),(x.stride(0),x.stride(1),x.stride(2),x.stride(2)))
    y=einops.einsum(win,w,"b c p k, o c k -> b o p")
    return y[:,:,::s]


<!-- dd:dd-q1241 -->

### Problem 1241 · independent

Return the output width of the convolution with stride s and padding p, as an integer. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel); s: stride ≥ 1; p: zeros added at each end. The kernel fits the padded input.


In [ ]:
import torch as t

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1241)


In [ ]:
#@title 💡 Solution — Problem 1241
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,s,p):
    return 1+(x.shape[2]+2*p-w.shape[2])//s


<!-- dd:dd-q1242 -->

### Problem 1242 · independent

Return convolution responses to channel-centred filters: each kernel position has zero mean across input channels. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel); s: stride ≥ 1; p: zeros added at each end. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1242)


In [ ]:
#@title 💡 Solution — Problem 1242
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    w=w-w.mean(dim=1,keepdim=True)
    b,c,width=x.shape
    o,_,k=w.shape
    xx=t.zeros((b,c,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+width]=x
    step=xx.stride()
    n=1+(xx.shape[2]-k)//s
    win=xx.as_strided((b,c,n,k),(step[0],step[1],s*step[2],step[2]))
    y=einops.einsum(win,w,"b c p k, o c k -> b o p")
    return y


<!-- dd:dd-q1243 -->

### Problem 1243 · independent

Return each filter’s activation averaged across both batch and positions, using stride s and padding p. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel); s: stride ≥ 1; p: zeros added at each end. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1243)


In [ ]:
#@title 💡 Solution — Problem 1243
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,width=x.shape
    o,_,k=w.shape
    xx=t.zeros((b,c,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+width]=x
    step=xx.stride()
    n=1+(xx.shape[2]-k)//s
    win=xx.as_strided((b,c,n,k),(step[0],step[1],s*step[2],step[2]))
    y=einops.einsum(win,w,"b c p k, o c k -> b o p")
    return y.mean(dim=(0,2))


<!-- dd:dd-q1244 -->

### Problem 1244 · independent

Return the convolution response summed over the batch, shape (out_channels,out_width), with stride s and padding p. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel); s: stride ≥ 1; p: zeros added at each end. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1244)


In [ ]:
#@title 💡 Solution — Problem 1244
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,width=x.shape
    o,_,k=w.shape
    xx=t.zeros((b,c,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+width]=x
    step=xx.stride()
    n=1+(xx.shape[2]-k)//s
    win=xx.as_strided((b,c,n,k),(step[0],step[1],s*step[2],step[2]))
    return einops.einsum(win,w,"b c p k, o c k -> o p")


<!-- dd:dd-q1245 -->

### Problem 1245 · independent

Return filters ranked by descending mean squared activation over batch and positions. Use stride s and padding p; scores are distinct. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel); s: stride ≥ 1; p: zeros added at each end. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1245)


In [ ]:
#@title 💡 Solution — Problem 1245
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,width=x.shape
    o,_,k=w.shape
    xx=t.zeros((b,c,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+width]=x
    step=xx.stride()
    n=1+(xx.shape[2]-k)//s
    win=xx.as_strided((b,c,n,k),(step[0],step[1],s*step[2],step[2]))
    y=einops.einsum(win,w,"b c p k, o c k -> b o p")
    return t.argsort((y*y).mean(dim=(0,2)),descending=True)


#### Common mistakes

- **The filter is reversed before multiplying.** Not in PyTorch; weight `0` meets the first value of each window.
- **Each output channel reads one input channel.** It reads all of them; the sum runs over channel and kernel axes together.
- **Padding is a view.** Padding allocates a new tensor; strides must be read from that tensor, not from `x`.
- **Stride `S` skips values inside a window.** It skips window *starts*; each window still holds `K` consecutive values.


<!-- dd:dd-kp-cnn-convolution-2d -->

## 2-D convolution

`cnn.convolution-2d`


<!-- dd:dd-seg-cnn-convolution-2d-0 -->

### An image window has two spatial axes


A 2-D convolution is the 1-D one with a second spatial axis everywhere. The input is `(batch, in_channels, height, width)`, the filter bank is `(out_channels, in_channels, kh, kw)`, and the window view has six axes: batch, input channel, output row, output column, kernel row, kernel column. Output rows number `1 + (H + 2P - kh) // S` and output columns `1 + (W + 2P - kw) // S`. The view's strides are read from the (padded) input: `(bs, cs, S*hs, S*ws, hs, ws)` — moving one output row is `S` image rows, moving one kernel row is one image row, and likewise for columns. The einsum sums over input channel and both kernel axes: `"b c h w i j, o c i j -> b o h w"`.

The reason to keep six named axes rather than flattening is that every mistake in a convolution is an axis mistake — rows and columns swapped, stride applied to the kernel axis, channels summed at the wrong place — and names make each one visible. The reason to test with rectangular images and rectangular kernels is the same: a `3×3` kernel on a square image hides a row/column swap; a `2×3` kernel on a `3×5` image does not. Padding, as in 1-D, is an allocated zero tensor with `x` written into its interior on both spatial axes.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nimport einops\nx=t.arange(1.,10.).reshape(1,1,3,3)\nwindows=x.as_strided((1,1,2,2,2,2),(9,9,3,1,3,1))\nw=t.ones(1,1,2,2)\ny=einops.einsum(windows,w,"b c h w i j, o c i j -> b o h w")\nprint(y)\n# Hidden checks\nassert y.tolist()==[[[[12.,16.],[24.,28.]]]]\n', globals()), end='')


We build the window view for a rectangular image and a rectangular kernel, reading strides from the tensor. A `3×5` image with a `2×3` kernel at stride `1` gives `2` output rows and `3` output columns.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.arange(1.,16.).reshape(1,1,3,5)\nbs,cs,hs,ws=x.stride()\nwin=x.as_strided((1,1,2,3,2,3),(bs,cs,hs,ws,hs,ws))\nprint(win.shape, win[0,0,1,2])\n# Hidden checks\nassert win.shape==(1,1,2,3,2,3) and win[0,0,1,2].tolist()==[[8.,9.,10.],[13.,14.,15.]]\n', globals()), end='')




Window `(1, 2)` is the bottom-right patch, and a kernel of ones sums it. The full output has one such sum per window position; predict the corner values before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import einops\ny=einops.einsum(win,t.ones(1,1,2,3),"b c h w i j, o c i j -> b o h w")\nprint(y)\n# Hidden checks\nassert y.tolist()==[[[[27.,33.,39.],[57.,63.,69.]]]]\n', globals()), end='')


<!-- dd:dd-q1251 -->

### Problem 1251 · faded — your turn

Return x with p zeros added on every side of both spatial axes. x: float (batch,in_channels,height,width); p: zero padding on every side.


In [ ]:
import torch as t

def solve(x,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1251)


In [ ]:
#@title 💡 Solution — Problem 1251
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,p):
    b,c,h,width=x.shape
    y=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    y[:,:,p:p+h,p:p+width]=x
    return y


We build the window view for a rectangular image and a rectangular kernel, reading strides from the tensor. A `3×5` image with a `2×3` kernel at stride `1` gives `2` output rows and `3` output columns.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.arange(1.,16.).reshape(1,1,3,5)\nbs,cs,hs,ws=x.stride()\nwin=x.as_strided((1,1,2,3,2,3),(bs,cs,hs,ws,hs,ws))\nprint(win.shape, win[0,0,1,2])\n# Hidden checks\nassert win.shape==(1,1,2,3,2,3) and win[0,0,1,2].tolist()==[[8.,9.,10.],[13.,14.,15.]]\n', globals()), end='')




Window `(1, 2)` is the bottom-right patch, and a kernel of ones sums it. The full output has one such sum per window position; predict the corner values before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import einops\ny=einops.einsum(win,t.ones(1,1,2,3),"b c h w i j, o c i j -> b o h w")\nprint(y)\n# Hidden checks\nassert y.tolist()==[[[[27.,33.,39.],[57.,63.,69.]]]]\n', globals()), end='')


<!-- dd:dd-q1249 -->

### Problem 1249 · faded — your turn

Return the image convolution with stride s and padding p. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1249)


In [ ]:
#@title 💡 Solution — Problem 1249
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    return y


<!-- dd:dd-seg-cnn-convolution-2d-1 -->

### The library function performs the same convolution


`torch.nn.functional.conv2d(x, w, stride=s, padding=p)` — imported as `F.conv2d` — computes exactly the windowed einsum: same axis conventions, same output size formula, zero padding on every side, no filter flip. It takes the weight as an argument on every call, which is what makes it a *function*: it owns nothing. A `Conv2d` module is the thin wrapper that stores the weight as a trainable `Parameter` and calls the function with it; ARENA's version omits the bias because a BatchNorm that follows supplies its own offset.

The reason to write the strided version first and then switch to `F.conv2d` is verification: the library call is the oracle your hand-built windows are checked against, and once they agree you understand what the fast path does. The reason fan-in matters is initialization: each output filter reads `in_channels * kh * kw` values, and the weight scale must shrink as that count grows so outputs do not blow up — the same rule as `in_features` for a linear layer. The number of output channels sets how many filters are learned, not how many values each one reads.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nimport torch.nn.functional as F\nx=t.arange(1.,16.).reshape(1,1,3,5)\nw=t.ones(1,1,2,3)\nprint(F.conv2d(x,w).shape)\n# Hidden checks\nassert F.conv2d(x,w).shape==(1,1,2,3)\n', globals()), end='')


We run the same ones-kernel over a ones-image with stride `2` and padding `1`, where the padding shows up as smaller sums at the edges. A `3×5` image padded to `5×7` with a `3×3` kernel at stride `2` gives `2` rows and `3` columns.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nimport torch.nn.functional as F\nx=t.ones(1,1,3,5)\nw=t.ones(1,1,3,3)\ny=F.conv2d(x,w,stride=2,padding=1)\nprint(y.shape)\n# Hidden checks\nassert y.shape==(1,1,2,3)\n', globals()), end='')




Corner windows overlap the padding on two sides and see four real pixels; edge windows see six. The interior of a bigger image would see nine.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(y)\n# Hidden checks\nassert y.tolist()==[[[[4.,6.,4.],[4.,6.,4.]]]]\n', globals()), end='')


<!-- dd:dd-q1252 -->

### Problem 1252 · faded — your turn

Return the image convolution through the functional library API. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import torch.nn.functional as F

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1252)


In [ ]:
#@title 💡 Solution — Problem 1252
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import torch.nn.functional as F

def solve(x,w,s,p):
    return F.conv2d(x,w,stride=s,padding=p)


We run the same ones-kernel over a ones-image with stride `2` and padding `1`, where the padding shows up as smaller sums at the edges. A `3×5` image padded to `5×7` with a `3×3` kernel at stride `2` gives `2` rows and `3` columns.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nimport torch.nn.functional as F\nx=t.ones(1,1,3,5)\nw=t.ones(1,1,3,3)\ny=F.conv2d(x,w,stride=2,padding=1)\nprint(y.shape)\n# Hidden checks\nassert y.shape==(1,1,2,3)\n', globals()), end='')




Corner windows overlap the padding on two sides and see four real pixels; edge windows see six. The interior of a bigger image would see nine.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(y)\n# Hidden checks\nassert y.tolist()==[[[[4.,6.,4.],[4.,6.,4.]]]]\n', globals()), end='')


<!-- dd:dd-q1250 -->

### Problem 1250 · faded — your turn

Return the image convolution followed by ReLU. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1250)


In [ ]:
#@title 💡 Solution — Problem 1250
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    return y.clamp(min=0)


<!-- dd:dd-q1253 -->

### Problem 1253 · independent

Return the output height and width of the image convolution with stride s and padding p, as a tuple of two integers. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1253)


In [ ]:
#@title 💡 Solution — Problem 1253
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,s,p):
    return (1+(x.shape[2]+2*p-w.shape[2])//s, 1+(x.shape[3]+2*p-w.shape[3])//s)


<!-- dd:dd-q1254 -->

### Problem 1254 · independent

Return average convolution activation per example and output channel, shape (batch,out_channels). x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1254)


In [ ]:
#@title 💡 Solution — Problem 1254
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    return y.mean(dim=(2,3))


<!-- dd:dd-q1255 -->

### Problem 1255 · independent

Return image convolution with each filter adjusted to have zero total coefficient. Use stride s and padding p. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1255)


In [ ]:
#@title 💡 Solution — Problem 1255
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    w=w-w.mean(dim=(1,2,3),keepdim=True)
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    return y


<!-- dd:dd-q1256 -->

### Problem 1256 · independent

Return each output channel’s maximum response over all examples and positions, shape (out_channels,). x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1256)


In [ ]:
#@title 💡 Solution — Problem 1256
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    return y.max(dim=3)[0].max(dim=2)[0].max(dim=0)[0]


<!-- dd:dd-q1257 -->

### Problem 1257 · independent

Return convolution activation at the upper-left output position, shape (batch,out_channels). Include padding exactly as in the full operation. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1257)


In [ ]:
#@title 💡 Solution — Problem 1257
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    return y[:,:,0,0]


<!-- dd:dd-q1258 -->

### Problem 1258 · independent

Return convolution feature maps centred to zero spatial mean independently for each example and output channel. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1258)


In [ ]:
#@title 💡 Solution — Problem 1258
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    return y-y.mean(dim=(2,3),keepdim=True)


<!-- dd:dd-q1259 -->

### Problem 1259 · independent

Return shape (batch,out_channels,2), containing each response map’s spatial minimum followed by its maximum. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1259)


In [ ]:
#@title 💡 Solution — Problem 1259
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    low=y.min(dim=3)[0].min(dim=2)[0]
    high=y.max(dim=3)[0].max(dim=2)[0]
    return t.stack((low,high),dim=-1)


<!-- dd:dd-q1260 -->

### Problem 1260 · independent

Return per-example total energy in each output channel, divided by total energy over all channels. Zero-energy examples return zeros. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1260)


In [ ]:
#@title 💡 Solution — Problem 1260
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    energy=(y*y).sum(dim=(2,3))
    total=energy.sum(dim=1,keepdim=True)
    return energy/t.where(total>0,total,t.ones_like(total))


<!-- dd:dd-q1261 -->

### Problem 1261 · independent

Return the response of locally mean-centred image patches to the filters. Patch means include padding and all input channels. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1261)


In [ ]:
#@title 💡 Solution — Problem 1261
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    win=win-win.mean(dim=(1,4,5),keepdim=True)
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    return y


#### Common mistakes

- **Stride scales the kernel strides too.** Only the output-position strides are multiplied by `S`; kernel strides stay one image step.
- **A square test image is enough.** It hides a row/column swap; use rectangular images and kernels.
- **`F.conv2d` flips the kernel.** It does not; it is the windowed sum with the weight as given.
- **More output channels means a bigger fan-in.** Fan-in is `in_channels * kh * kw`; output channels only set how many filters exist.


<!-- dd:dd-kp-cnn-pooling -->

## Pooling

`cnn.pooling`


<!-- dd:dd-seg-cnn-pooling-0 -->

### A spatial summary keeps examples and channels apart


Global average pooling replaces each channel's whole image with a single number — its mean over height and width. On `(batch, channels, height, width)` that is `x.mean(dim=(2, 3))`, and the result is `(batch, channels)`: one row per example, one value per channel, ready for a linear classifier head. The reduction names the two spatial axes and nothing else, so examples are never averaged together and channels are never mixed.

The reason to reduce only the spatial axes is what each axis means. Averaging over the batch would blend different images; averaging over channels would blend different feature detectors. The reason the mean is the usual choice at the end of a network is that it asks "how much of this feature appeared across the image", which is robust to where it appeared; a global maximum instead asks "did this feature appear strongly anywhere". Neither has a learned weight — pooling is a fixed summary, which is why it costs nothing to train.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[[[1.,3.],[5.,7.]],[[2.,2.],[4.,4.]]]])\nprint(x.mean(dim=(2,3)))\n# Hidden checks\nassert x.mean(dim=(2,3)).tolist()==[[4.,3.]]\n', globals()), end='')


We pool a batch of two one-channel images: one flat, one with a single bright pixel. The mean sees the total brightness; the maximum sees the peak. Predict both before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[[[2.,2.],[2.,2.]]],[[[0.,0.],[0.,8.]]]])\nprint(x.mean(dim=(2,3)))\n# Hidden checks\nassert x.mean(dim=(2,3)).tolist()==[[2.],[2.]]\n', globals()), end='')




The two images have the same mean but very different maxima. Reducing `dim=3` then `dim=2` takes the max over columns and then rows, and `[0]` picks the values rather than the indices.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(x.max(dim=3)[0].max(dim=2)[0])\n# Hidden checks\nassert x.max(dim=3)[0].max(dim=2)[0].tolist()==[[2.],[8.]]\n', globals()), end='')


<!-- dd:dd-q1265 -->

### Problem 1265 · faded — your turn

Return global spatial average, shape (batch,channels). x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1265)


In [ ]:
#@title 💡 Solution — Problem 1265
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.mean(dim=(2,3))


We pool a batch of two one-channel images: one flat, one with a single bright pixel. The mean sees the total brightness; the maximum sees the peak. Predict both before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[[[2.,2.],[2.,2.]]],[[[0.,0.],[0.,8.]]]])\nprint(x.mean(dim=(2,3)))\n# Hidden checks\nassert x.mean(dim=(2,3)).tolist()==[[2.],[2.]]\n', globals()), end='')




The two images have the same mean but very different maxima. Reducing `dim=3` then `dim=2` takes the max over columns and then rows, and `[0]` picks the values rather than the indices.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(x.max(dim=3)[0].max(dim=2)[0])\n# Hidden checks\nassert x.max(dim=3)[0].max(dim=2)[0].tolist()==[[2.],[8.]]\n', globals()), end='')


<!-- dd:dd-q1268 -->

### Problem 1268 · faded — your turn

Return each channel’s spatial maximum, shape (batch,channels). x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1268)


In [ ]:
#@title 💡 Solution — Problem 1268
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.max(dim=3)[0].max(dim=2)[0]


<!-- dd:dd-seg-cnn-pooling-1 -->

### Local maxima need a neutral boundary


Local max pooling keeps coarse spatial layout: each output is the largest value in one `k×k` window, with windows placed by a stride, exactly as convolution places them. The window view is the six-axis one from convolution, `(batch, channels, out_h, out_w, k, k)`, and the reduction is a maximum over the last two axes — `win.max(dim=5)[0].max(dim=4)[0]` — so channels are never combined. Output size follows the same `1 + (H + 2P - k) // S` rule.

The reason padding must be `-inf` here, not zero, is that the maximum is not neutral to zero: a window lying entirely over negative activations should return its largest negative value, and a zero pad would wrongly win. `t.full(shape, float("-inf"))` makes a boundary that can never be the maximum, and writing `x` into its interior gives the padded tensor whose strides the view reads. The reason max pooling is used between convolutions is that it halves the resolution while keeping the strongest response in each neighbourhood, so later layers see a wider context per pixel.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([-4.,-2.])\npadded=t.full((4,),float("-inf"))\npadded[1:3]=x\nprint(padded.as_strided((3,2),(1,1)).max(dim=1)[0])\n# Hidden checks\nassert padded.as_strided((3,2),(1,1)).max(dim=1)[0].tolist()==[-4.,-2.,-2.]\n', globals()), end='')


We pool a `4×4` image with `2×2` windows at stride `2`, which tiles it into four blocks. The view's output-position strides are twice the image strides; the kernel strides are the image strides themselves.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.arange(16.).reshape(1,1,4,4)\nbs,cs,hs,ws=x.stride()\nwin=x.as_strided((1,1,2,2,2,2),(bs,cs,2*hs,2*ws,hs,ws))\nprint(win[0,0,0,1])\n# Hidden checks\nassert win[0,0,0,1].tolist()==[[2.,3.],[6.,7.]]\n', globals()), end='')




Each block's maximum is its bottom-right value, because the image increases along both axes. The result is a `2×2` map: the same layout, half the resolution.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(win.max(dim=5)[0].max(dim=4)[0])\n# Hidden checks\nassert win.max(dim=5)[0].max(dim=4)[0].tolist()==[[[[5.,7.],[13.,15.]]]]\n', globals()), end='')


<!-- dd:dd-q1266 -->

### Problem 1266 · faded — your turn

Return local max pooling with stride s and padding p. x: float (batch,channels,height,width); k: square kernel size; s: stride; p: zero padding on every side, 0 ≤ p ≤ k//2. The kernel fits the padded image.


In [ ]:
import torch as t

def solve(x,k,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1266)


In [ ]:
#@title 💡 Solution — Problem 1266
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,k,s,p):
    b,c,h,w=x.shape
    xx=t.full((b,c,h+2*p,w+2*p),float("-inf"),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+w]=x
    bs,cs,hs,ws=xx.stride()
    win=xx.as_strided((b,c,1+(h+2*p-k)//s,1+(w+2*p-k)//s,k,k),(bs,cs,s*hs,s*ws,hs,ws))
    y=win.max(dim=5)[0].max(dim=4)[0]
    return y


We pool a `4×4` image with `2×2` windows at stride `2`, which tiles it into four blocks. The view's output-position strides are twice the image strides; the kernel strides are the image strides themselves.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.arange(16.).reshape(1,1,4,4)\nbs,cs,hs,ws=x.stride()\nwin=x.as_strided((1,1,2,2,2,2),(bs,cs,2*hs,2*ws,hs,ws))\nprint(win[0,0,0,1])\n# Hidden checks\nassert win[0,0,0,1].tolist()==[[2.,3.],[6.,7.]]\n', globals()), end='')




Each block's maximum is its bottom-right value, because the image increases along both axes. The result is a `2×2` map: the same layout, half the resolution.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(win.max(dim=5)[0].max(dim=4)[0])\n# Hidden checks\nassert win.max(dim=5)[0].max(dim=4)[0].tolist()==[[[[5.,7.],[13.,15.]]]]\n', globals()), end='')


<!-- dd:dd-q1267 -->

### Problem 1267 · faded — your turn

Return positive parts of locally pooled values. x: float (batch,channels,height,width); k: square kernel size; s: stride; p: zero padding on every side, 0 ≤ p ≤ k//2. The kernel fits the padded image.


In [ ]:
import torch as t

def solve(x,k,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1267)


In [ ]:
#@title 💡 Solution — Problem 1267
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,k,s,p):
    b,c,h,w=x.shape
    xx=t.full((b,c,h+2*p,w+2*p),float("-inf"),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+w]=x
    bs,cs,hs,ws=xx.stride()
    win=xx.as_strided((b,c,1+(h+2*p-k)//s,1+(w+2*p-k)//s,k,k),(bs,cs,s*hs,s*ws,hs,ws))
    y=win.max(dim=5)[0].max(dim=4)[0]
    return y.clamp(min=0)


<!-- dd:dd-q1269 -->

### Problem 1269 · independent

Return the global average features with the mean across channels removed for each example, shape (batch,channels). x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1269)


In [ ]:
#@title 💡 Solution — Problem 1269
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    g=x.mean(dim=(2,3))
    return g-g.mean(dim=1,keepdim=True)


<!-- dd:dd-q1270 -->

### Problem 1270 · independent

Return local MIN pooling with kernel k, stride s and padding p: the smallest value in every window, where padded positions never win. x: float (batch,channels,height,width); k: square kernel size; s: stride; p: zero padding on every side, 0 ≤ p ≤ k//2. The kernel fits the padded image.


In [ ]:
import torch as t

def solve(x,k,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1270)


In [ ]:
#@title 💡 Solution — Problem 1270
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,k,s,p):
    b,c,h,w=x.shape
    xx=t.full((b,c,h+2*p,w+2*p),float("inf"),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+w]=x
    bs,cs,hs,ws=xx.stride()
    win=xx.as_strided((b,c,1+(h+2*p-k)//s,1+(w+2*p-k)//s,k,k),(bs,cs,s*hs,s*ws,hs,ws))
    return win.min(dim=5)[0].min(dim=4)[0]


<!-- dd:dd-q1271 -->

### Problem 1271 · independent

Return the spatial range of each channel, shape (batch,channels). x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1271)


In [ ]:
#@title 💡 Solution — Problem 1271
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.max(dim=3)[0].max(dim=2)[0]-x.min(dim=3)[0].min(dim=2)[0]


<!-- dd:dd-q1272 -->

### Problem 1272 · independent

Return the difference between global max and global mean of each channel. x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1272)


In [ ]:
#@title 💡 Solution — Problem 1272
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.max(dim=3)[0].max(dim=2)[0]-x.mean(dim=(2,3))


<!-- dd:dd-q1273 -->

### Problem 1273 · independent

Return the fraction of positive pixels in each example/channel, shape (batch,channels). x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1273)


In [ ]:
#@title 💡 Solution — Problem 1273
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return (x>0).to(x.dtype).mean(dim=(2,3))


<!-- dd:dd-q1274 -->

### Problem 1274 · independent

Return global averages of locally max-pooled maps, shape (batch,channels). x: float (batch,channels,height,width); k: square kernel size; s: stride; p: zero padding on every side, 0 ≤ p ≤ k//2. The kernel fits the padded image.


In [ ]:
import torch as t

def solve(x,k,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1274)


In [ ]:
#@title 💡 Solution — Problem 1274
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,k,s,p):
    b,c,h,w=x.shape
    xx=t.full((b,c,h+2*p,w+2*p),float("-inf"),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+w]=x
    bs,cs,hs,ws=xx.stride()
    win=xx.as_strided((b,c,1+(h+2*p-k)//s,1+(w+2*p-k)//s,k,k),(bs,cs,s*hs,s*ws,hs,ws))
    y=win.max(dim=5)[0].max(dim=4)[0]
    return y.mean(dim=(2,3))


<!-- dd:dd-q1275 -->

### Problem 1275 · independent

Return each channel’s spatial maximum location as a flattened row-major index, shape (batch,channels). Ties choose first. x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1275)


In [ ]:
#@title 💡 Solution — Problem 1275
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.reshape(x.shape[0],x.shape[1],-1).argmax(dim=2)


<!-- dd:dd-q1276 -->

### Problem 1276 · independent

Return unit-length global-average feature vectors, shape (batch,channels). A zero vector stays zero. x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1276)


In [ ]:
#@title 💡 Solution — Problem 1276
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    y=x.mean(dim=(2,3))
    n=y.norm(dim=1,keepdim=True)
    return y/t.where(n>0,n,t.ones_like(n))


<!-- dd:dd-q1277 -->

### Problem 1277 · independent

Return root-mean-square activation per example/channel, shape (batch,channels). x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1277)


In [ ]:
#@title 💡 Solution — Problem 1277
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return t.sqrt((x*x).mean(dim=(2,3)))


#### Common mistakes

- **Global pooling returns one number per image.** It returns one per channel per image, shape `(batch, channels)`.
- **Zero padding is fine for max pooling.** A zero beats every negative activation; pad with `-inf`.
- **Pooling reduces the channel axis.** It never does; channels are distinct features and stay separate.
- **`x.max(dim=(2, 3))` works like `mean`.** `max` takes one axis at a time; reduce twice and index `[0]` for the values.


<!-- dd:dd-kp-cnn-batch-normalization -->

## Batch normalization

`cnn.batch-normalization`


<!-- dd:dd-seg-cnn-batch-normalization-0 -->

### One distribution per channel


BatchNorm treats each channel as its own distribution. For a batch `x` of shape `(batch, channels, height, width)`, the statistics of channel `c` are taken over every example and every pixel — axes `0`, `2` and `3` — while channels are never mixed. The procedure is: compute the per-channel mean, subtract it, compute the per-channel variance of the centred values, divide by `sqrt(var + eps)`, then apply a learned scale `g` and shift `b` per channel. Every per-channel vector has shape `(channels,)` and must be broadcast back as `[None, :, None, None]` to line up with the channel axis.

The reason the reduction spans batch AND pixels is that a channel is one feature detector; its output at any position on any image is a sample from the same distribution. The variance is the *population* variance — the plain mean of squared deviations, with no `n-1` correction — because the batch is not a sample of some larger truth here; it is the thing being normalized. `eps` sits inside the square root so that a constant channel, whose variance is exactly zero, divides by `sqrt(eps)` rather than by zero and comes out as all zeros, which then equals the shift `b`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[[[1.,3.]],[[2.,6.]]]])\nmean=x.mean(dim=(0,2,3),keepdim=True)\nvar=((x-mean)**2).mean(dim=(0,2,3),keepdim=True)\nz=(x-mean)/t.sqrt(var+1e-5)\nprint(z)\n# Hidden checks\nassert t.allclose(z,t.tensor([[[[-1.,1.]],[[-1.,1.]]]]),atol=1e-3)\n', globals()), end='')


We normalize a batch of two images with one channel and two pixels each, so the channel's four values are `5, 7, 1, 3`. Their mean is `4`, and the squared deviations are `1, 9, 9, 1`, so the population variance is `5`. Predict the sign of each normalized value before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[[[5.,7.]]],[[[1.,3.]]]])\nmean=x.mean(dim=(0,2,3))\nvar=((x-mean[None,:,None,None])**2).mean(dim=(0,2,3))\nprint(mean, var)\n# Hidden checks\nassert mean.tolist()==[4.] and var.tolist()==[5.]\n', globals()), end='')




Dividing by `sqrt(var + eps)` leaves values roughly between `-1.4` and `1.4`; the learned scale and shift then move them wherever the next layer wants. With `g = 2` and `b = 1` the pixel that was at the mean becomes exactly `1`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('g=t.tensor([2.]); b=t.tensor([1.]); eps=1e-5\nz=(x-mean[None,:,None,None])/t.sqrt(var[None,:,None,None]+eps)\ny=z*g[None,:,None,None]+b[None,:,None,None]\nprint(y)\n# Hidden checks\nassert t.allclose(y[0,0,0],t.tensor([1.894,3.683]),atol=1e-2)\n', globals()), end='')


<!-- dd:dd-q1297 -->

### Problem 1297 · faded — your turn

Return per-channel batch means, shape (channels,). x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1297)


In [ ]:
#@title 💡 Solution — Problem 1297
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    mean=x.mean(dim=(0,2,3))
    return mean


We normalize a batch of two images with one channel and two pixels each, so the channel's four values are `5, 7, 1, 3`. Their mean is `4`, and the squared deviations are `1, 9, 9, 1`, so the population variance is `5`. Predict the sign of each normalized value before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[[[5.,7.]]],[[[1.,3.]]]])\nmean=x.mean(dim=(0,2,3))\nvar=((x-mean[None,:,None,None])**2).mean(dim=(0,2,3))\nprint(mean, var)\n# Hidden checks\nassert mean.tolist()==[4.] and var.tolist()==[5.]\n', globals()), end='')




Dividing by `sqrt(var + eps)` leaves values roughly between `-1.4` and `1.4`; the learned scale and shift then move them wherever the next layer wants. With `g = 2` and `b = 1` the pixel that was at the mean becomes exactly `1`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('g=t.tensor([2.]); b=t.tensor([1.]); eps=1e-5\nz=(x-mean[None,:,None,None])/t.sqrt(var[None,:,None,None]+eps)\ny=z*g[None,:,None,None]+b[None,:,None,None]\nprint(y)\n# Hidden checks\nassert t.allclose(y[0,0,0],t.tensor([1.894,3.683]),atol=1e-2)\n', globals()), end='')


<!-- dd:dd-q1298 -->

### Problem 1298 · faded — your turn

Return training-mode normalized and affine-transformed x. x: float (batch,channels,height,width); g: scale (channels,); b: shift (channels,); eps > 0. Variance is the population variance.


In [ ]:
import torch as t

def solve(x,g,b,eps):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1298)


In [ ]:
#@title 💡 Solution — Problem 1298
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,g,b,eps):
    mean=x.mean(dim=(0,2,3))
    var=((x-mean[None,:,None,None])**2).mean(dim=(0,2,3))
    z=(x-mean[None,:,None,None])/t.sqrt(var[None,:,None,None]+eps)
    y=z*g[None,:,None,None]+b[None,:,None,None]
    return y


<!-- dd:dd-seg-cnn-batch-normalization-1 -->

### Running statistics remember past batches


At evaluation time there may be no batch to take statistics from — a single image, say — so BatchNorm keeps a *running* mean and variance per channel and updates them during training. The update is an exponential moving average: `running = (1 - momentum) * running + momentum * batch_stat`. With `momentum = 0.1` each new batch nudges the estimate a tenth of the way toward itself; with `momentum = 1` the estimate is simply the last batch.

The update is bookkeeping, not computation the network learns from, so it is done in place on the buffer and wrapped in `with t.no_grad():`. `with` opens a block in which a context is active and closes it when the block ends; `t.no_grad()` is the context in which autograd stops recording. The reason it is needed is that the batch mean was computed from `x`, which may carry a gradient graph; writing it into the buffer without `no_grad` would splice that graph into the buffer's history, and the next training step would try to backpropagate into the previous batch. Wrapping only the update, not the normalization, keeps the gradient path through the output intact.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nrunning=t.tensor([2.,6.])\nbatch_mean=t.tensor([4.,2.])\nmomentum=.25\nwith t.no_grad():\n    running.copy_((1-momentum)*running+momentum*batch_mean)\nprint(running)\n# Hidden checks\nassert running.tolist()==[2.5,5.]\n', globals()), end='')


We watch the running mean converge toward a repeated batch. The buffer starts at `0`, the batch mean is `8`, and `momentum = 0.5` moves halfway each step. After one update the estimate is `4`; predict the second.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nrunning=t.tensor([0.])\nbatch_mean=t.tensor([8.])\nmomentum=.5\nwith t.no_grad():\n    running.copy_((1-momentum)*running+momentum*batch_mean)\nprint(running)\n# Hidden checks\nassert running.tolist()==[4.]\n', globals()), end='')




The second step starts from the updated value, not the original: `(1-0.5)*4 + 0.5*8 = 6`. Each application closes half the remaining gap, which is why the estimate never overshoots and never quite arrives.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('with t.no_grad():\n    running.copy_((1-momentum)*running+momentum*batch_mean)\nprint(running)\n# Hidden checks\nassert running.tolist()==[6.]\n', globals()), end='')


<!-- dd:dd-q1299 -->

### Problem 1299 · faded — your turn

Return the updated running mean after this batch. x: float (batch,channels,height,width); rm: running mean (channels,); momentum in (0,1]. Variance is the population variance.


In [ ]:
import torch as t

def solve(x,rm,momentum):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1299)


In [ ]:
#@title 💡 Solution — Problem 1299
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,rm,momentum):
    return (1-momentum)*rm+momentum*x.mean(dim=(0,2,3))


<!-- dd:dd-seg-cnn-batch-normalization-2 -->

### Training uses the batch; evaluation uses the memory


A module carries a flag, `self.training`, that `m.train()` sets to `True` and `m.eval()` sets to `False`. BatchNorm's forward branches on it: in training mode it normalizes with the current batch's statistics and updates the running buffers; in evaluation mode it normalizes with the running buffers and changes nothing. The learned scale and shift are applied identically in both modes.

The reason for two behaviours is what each mode is for. Training wants the actual distribution of the batch, so the network learns against what it will see. Evaluation wants each prediction to depend only on its own input: if the batch statistics were used, the output for one image would change depending on which other images happened to share its batch, and a model served one request at a time would divide by a variance of zero. So evaluation uses the statistics remembered from training, and the buffers are frozen — which is also why evaluation must not update them.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nm=t.nn.Module()\nm.eval()\nif m.training:\n    label="batch statistics"\nelse:\n    label="running statistics"\nprint(label)\n# Hidden checks\nassert label=="running statistics"\n', globals()), end='')


We normalize the same one-channel batch in both modes and compare. In evaluation mode the stored mean `1` and variance `4` are used, so a pixel at `1` maps to zero and a pixel at `3` maps to `2 / sqrt(4 + eps)`, just under `1`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[[[1.,3.]]]])\nrm=t.tensor([1.]); rv=t.tensor([4.]); eps=1e-5\nz_eval=(x-rm[None,:,None,None])/t.sqrt(rv[None,:,None,None]+eps)\nprint(z_eval)\n# Hidden checks\nassert t.allclose(z_eval,t.tensor([[[[0.,1.]]]]),atol=1e-3)\n', globals()), end='')




In training mode the batch's own statistics apply — mean `2`, variance `1` — so the same two pixels land symmetrically at `-1` and `1`. Same input, different output: that is the flag doing its job.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('mean=x.mean(dim=(0,2,3)); var=((x-mean[None,:,None,None])**2).mean(dim=(0,2,3))\nz_train=(x-mean[None,:,None,None])/t.sqrt(var[None,:,None,None]+eps)\nprint(z_train)\n# Hidden checks\nassert t.allclose(z_train,t.tensor([[[[-1.,1.]]]]),atol=1e-3)\n', globals()), end='')


<!-- dd:dd-q1300 -->

### Problem 1300 · faded — your turn

Return evaluation-mode normalized and affine-transformed x. x: float (batch,channels,height,width); g: scale (channels,); b: shift (channels,); rm: running mean (channels,); rv: running variance (channels,); eps > 0. Variance is the population variance.


In [ ]:
import torch as t

def solve(x,g,b,rm,rv,eps):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1300)


In [ ]:
#@title 💡 Solution — Problem 1300
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,g,b,rm,rv,eps):
    mean=rm
    var=rv
    z=(x-mean[None,:,None,None])/t.sqrt(var[None,:,None,None]+eps)
    y=z*g[None,:,None,None]+b[None,:,None,None]
    return y


<!-- dd:dd-q1301 -->

### Problem 1301 · independent

Return the training-mode BatchNorm output followed by ReLU, preserving the shape of x. x: float (batch,channels,height,width); g: scale (channels,); b: shift (channels,); eps > 0. Variance is the population variance.


In [ ]:
import torch as t

def solve(x,g,b,eps):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1301)


In [ ]:
#@title 💡 Solution — Problem 1301
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,g,b,eps):
    mean=x.mean(dim=(0,2,3))
    var=((x-mean[None,:,None,None])**2).mean(dim=(0,2,3))
    z=(x-mean[None,:,None,None])/t.sqrt(var[None,:,None,None]+eps)
    y=z*g[None,:,None,None]+b[None,:,None,None]
    return y.clamp(min=0)


<!-- dd:dd-q1302 -->

### Problem 1302 · independent

Return the evaluation-mode BatchNorm output averaged over both spatial axes, shape (batch,channels). x: float (batch,channels,height,width); g: scale (channels,); b: shift (channels,); rm: running mean (channels,); rv: running variance (channels,); eps > 0. Variance is the population variance.


In [ ]:
import torch as t

def solve(x,g,b,rm,rv,eps):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1302)


In [ ]:
#@title 💡 Solution — Problem 1302
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,g,b,rm,rv,eps):
    z=(x-rm[None,:,None,None])/t.sqrt(rv[None,:,None,None]+eps)
    y=z*g[None,:,None,None]+b[None,:,None,None]
    return y.mean(dim=(2,3))


<!-- dd:dd-q1303 -->

### Problem 1303 · independent

Return population variance of each input channel, shape (channels,). x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1303)


In [ ]:
#@title 💡 Solution — Problem 1303
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    mean=x.mean(dim=(0,2,3))
    var=((x-mean[None,:,None,None])**2).mean(dim=(0,2,3))
    return var


<!-- dd:dd-q1304 -->

### Problem 1304 · independent

Return the updated running variance after this batch. x: float (batch,channels,height,width); rv: running variance (channels,); momentum in (0,1]. Variance is the population variance.


In [ ]:
import torch as t

def solve(x,rv,momentum):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1304)


In [ ]:
#@title 💡 Solution — Problem 1304
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,rv,momentum):
    mean=x.mean(dim=(0,2,3))
    var=((x-mean[None,:,None,None])**2).mean(dim=(0,2,3))
    return (1-momentum)*rv+momentum*var


<!-- dd:dd-q1305 -->

### Problem 1305 · independent

Return training output minus evaluation output for the same input. x: float (batch,channels,height,width); g: scale (channels,); b: shift (channels,); rm: running mean (channels,); rv: running variance (channels,); eps > 0. Variance is the population variance.


In [ ]:
import torch as t

def solve(x,g,b,rm,rv,eps):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1305)


In [ ]:
#@title 💡 Solution — Problem 1305
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,g,b,rm,rv,eps):
    mean=x.mean(dim=(0,2,3))
    var=((x-mean[None,:,None,None])**2).mean(dim=(0,2,3))
    z=(x-mean[None,:,None,None])/t.sqrt(var[None,:,None,None]+eps)
    y=z*g[None,:,None,None]+b[None,:,None,None]
    training=y
    mean=rm
    var=rv
    z=(x-mean[None,:,None,None])/t.sqrt(var[None,:,None,None]+eps)
    y=z*g[None,:,None,None]+b[None,:,None,None]
    return training-y


<!-- dd:dd-q1306 -->

### Problem 1306 · independent

Return the running mean after observing this same batch twice. Each update uses the result of the previous update. x: float (batch,channels,height,width); rm: running mean (channels,); momentum in (0,1]. Variance is the population variance.


In [ ]:
import torch as t

def solve(x,rm,momentum):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1306)


In [ ]:
#@title 💡 Solution — Problem 1306
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,rm,momentum):
    mean=x.mean(dim=(0,2,3))
    first=(1-momentum)*rm+momentum*mean
    return (1-momentum)*first+momentum*mean


<!-- dd:dd-q1307 -->

### Problem 1307 · independent

Return the effective scale per channel for evaluation: the multiplier that converts each input value into output before adding a constant offset. g: scale (channels,); rv: running variance (channels,); eps > 0.


In [ ]:
import torch as t

def solve(g,rv,eps):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1307)


In [ ]:
#@title 💡 Solution — Problem 1307
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(g,rv,eps):
    return g/t.sqrt(rv+eps)


<!-- dd:dd-q1308 -->

### Problem 1308 · independent

Return the effective offset per channel for evaluation, so output equals input times effective scale plus this offset. g: scale (channels,); b: shift (channels,); rm: running mean (channels,); rv: running variance (channels,); eps > 0.


In [ ]:
import torch as t

def solve(g,b,rm,rv,eps):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1308)


In [ ]:
#@title 💡 Solution — Problem 1308
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(g,b,rm,rv,eps):
    return b-rm*g/t.sqrt(rv+eps)


<!-- dd:dd-q1309 -->

### Problem 1309 · independent

Return training-mode channel means and population variances after the affine transform, shape (2,channels), means first. x: float (batch,channels,height,width); g: scale (channels,); b: shift (channels,); eps > 0. Variance is the population variance.


In [ ]:
import torch as t

def solve(x,g,b,eps):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1309)


In [ ]:
#@title 💡 Solution — Problem 1309
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,g,b,eps):
    mean=x.mean(dim=(0,2,3))
    var=((x-mean[None,:,None,None])**2).mean(dim=(0,2,3))
    z=(x-mean[None,:,None,None])/t.sqrt(var[None,:,None,None]+eps)
    y=z*g[None,:,None,None]+b[None,:,None,None]
    ym=y.mean(dim=(0,2,3))
    yv=((y-ym[None,:,None,None])**2).mean(dim=(0,2,3))
    return t.stack((ym,yv))


#### Common mistakes

- **Statistics are per image.** They are per channel, pooled over every example in the batch and every pixel; a per-image mean would erase the difference between a bright image and a dark one.
- **Variance uses `n-1`.** BatchNorm uses the population variance; `x.var()`'s default correction gives a different number.
- **Evaluation mode still updates the buffers.** It reads them and leaves them alone; only training mode writes.
- **The running update is part of the gradient path.** It is bookkeeping done under `no_grad`; the normalized output, not the buffer write, is what gradients flow through.


<!-- dd:dd-kp-cnn-sequential -->

## Chaining modules in order

`cnn.sequential`


<!-- dd:dd-seg-cnn-sequential-0 -->

### Registered children run in registration order


A module can own other modules the way it owns parameters. `self.add_module(name, mod)` registers `mod` as a child under a string name, and the module keeps every child in `self._modules`, a dictionary from name to child that remembers the order of insertion. A chain's `forward` walks the names in `self._modules` in that order, looks each child up and feeds each output into the next input, so the first module registered is the first one applied. Registering by name is the one mechanism behind both `self.layer = mod` and `add_module`; the explicit form is what you reach for when the children arrive in a list and their names are just their positions.

The reason to register rather than keep a plain Python list is the same reason a `Parameter` beats a plain tensor: only registered children are visible to `parameters()`, `state_dict()` and `to()`. A list attribute holding modules would compute correctly and then hand the optimizer nothing to train.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass Double(t.nn.Module):\n    def forward(self,x):\n        return x*2\nclass AddOne(t.nn.Module):\n    def forward(self,x):\n        return x+1\nclass Chain(t.nn.Module):\n    def __init__(self,mods):\n        super().__init__()\n        i=0\n        for mod in mods:\n            self.add_module(str(i),mod)\n            i+=1\n    def forward(self,x):\n        for name in self._modules:\n            x=self._modules[name](x)\n        return x\nprint(list(Chain([Double(),AddOne()])._modules))\n# Hidden checks\nassert list(Chain([Double(),AddOne()])._modules)==["0","1"]\n', globals()), end='')


We chain `Double` then `AddOne` and apply the result to `3`. Registration order is application order, so the value is doubled first and then incremented: `3*2+1 = 7`. Predict what the reversed chain gives before running the second cell.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('m=Chain([Double(),AddOne()])\nprint(list(m._modules), m(t.tensor(3.)))\n# Hidden checks\nassert list(m._modules)==["0","1"] and m(t.tensor(3.)).item()==7.\n', globals()), end='')




Reversing the list reverses the computation: `(3+1)*2 = 8`. Same two children, different order, different function — the chain is the order, not the set.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('r=Chain([AddOne(),Double()])\nprint(r(t.tensor(3.)))\n# Hidden checks\nassert r(t.tensor(3.)).item()==8.\n', globals()), end='')


<!-- dd:dd-q1374 -->

### Problem 1374 · faded — your turn

Return a module that applies each module in the list `mods` in order, feeding every output into the next module. Register the children under the names "0", "1", … so that the module's registered children come out in application order. For an empty list the forward returns its input unchanged.


In [ ]:
import torch as t

def solve(mods):
    class Chain(t.nn.Module):
        def __init__(self):
            super().__init__()
            i=0
            for mod in mods:
                self._____(str(i),mod)
                i+=1
        def forward(self,x):
            for name in self._____:
                x=self._____[name](x)
            return x
    return Chain()


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1374)


In [ ]:
#@title 💡 Solution — Problem 1374
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(mods):
    class Chain(t.nn.Module):
        def __init__(self):
            super().__init__()
            i=0
            for mod in mods:
                self.add_module(str(i),mod)
                i+=1
        def forward(self,x):
            for name in self._modules:
                x=self._modules[name](x)
            return x
    return Chain()


We chain `Double` then `AddOne` and apply the result to `3`. Registration order is application order, so the value is doubled first and then incremented: `3*2+1 = 7`. Predict what the reversed chain gives before running the second cell.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('m=Chain([Double(),AddOne()])\nprint(list(m._modules), m(t.tensor(3.)))\n# Hidden checks\nassert list(m._modules)==["0","1"] and m(t.tensor(3.)).item()==7.\n', globals()), end='')




Reversing the list reverses the computation: `(3+1)*2 = 8`. Same two children, different order, different function — the chain is the order, not the set.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('r=Chain([AddOne(),Double()])\nprint(r(t.tensor(3.)))\n# Hidden checks\nassert r(t.tensor(3.)).item()==8.\n', globals()), end='')


<!-- dd:dd-q1375 -->

### Problem 1375 · faded — your turn

Without building a module, return the result of applying the modules in the list `mods` to the tensor `x` in order: the first module sees `x`, each later module sees the previous output. An empty list returns `x`.


In [ ]:
import torch as t

def solve(mods,x):
    for mod in mods:
        x=_____
    return _____


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1375)


In [ ]:
#@title 💡 Solution — Problem 1375
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(mods,x):
    for mod in mods:
        x=mod(x)
    return x


<!-- dd:dd-seg-cnn-sequential-1 -->

### The library chain is `t.nn.Sequential`


`t.nn.Sequential(a, b, c)` is the ready-made version of that chain: its children are registered under `"0"`, `"1"`, `"2"` in the order given, `forward` applies them in that order, `len(m)` counts them and `m[i]` returns the `i`-th child, with negative indices counting from the end like a list. An empty `t.nn.Sequential()` is a chain with no steps whose forward returns its input, and `add_module` still works on it, so a chain can be built up in a loop. A chain can also be walked directly: `for mod in m` visits the children in order.

A step in a chain does not need parameters. `t.nn.ReLU()` is a module whose forward is `x.clamp(min=0)`, and packaging the nonlinearity as a module is what lets it sit between two layers as an ordinary step rather than as a special case inside somebody's `forward`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass Double(t.nn.Module):\n    def forward(self,x):\n        return x*2\nclass AddOne(t.nn.Module):\n    def forward(self,x):\n        return x+1\nm=t.nn.Sequential(Double(),t.nn.ReLU(),AddOne())\nprint(len(m), m[1], m[-1])\n# Hidden checks\nassert len(m)==3 and isinstance(m[1],t.nn.ReLU) and isinstance(m[-1],AddOne)\n', globals()), end='')


We push `[-1, 2]` through the chain. Doubling gives `[-2, 4]`, the ReLU clamps the negative entry to `0`, and adding one gives `[1, 5]`. Predict what happens to `-1` if the ReLU is removed.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('x=t.tensor([-1.,2.])\nprint(m(x))\n# Hidden checks\nassert m(x).tolist()==[1.,5.]\n', globals()), end='')




Without the clamp the negative entry survives: `-2+1 = -1`. The two chains agree on the positive entry and disagree on the negative one, which is exactly the nonlinearity's job.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('n=t.nn.Sequential()\nn.add_module("0",Double())\nn.add_module("1",AddOne())\nprint(n(x), len(n))\n# Hidden checks\nassert n(x).tolist()==[-1.,5.] and len(n)==2\n', globals()), end='')


<!-- dd:dd-q1376 -->

### Problem 1376 · faded — your turn

Return a library Sequential chain that applies the modules in the list `mods` in order with a fresh ReLU module inserted immediately after EACH of them, so the chain has twice as many children as `mods` has entries. Start from an empty chain and register the children under the names "0", "1", … in application order.


In [ ]:
import torch as t

def solve(mods):
    seq=t.nn._____()
    i=0
    for mod in mods:
        seq._____(str(i),mod)
        seq._____(str(i+1),t.nn._____())
        i+=2
    return seq


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1376)


In [ ]:
#@title 💡 Solution — Problem 1376
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(mods):
    seq=t.nn.Sequential()
    i=0
    for mod in mods:
        seq.add_module(str(i),mod)
        seq.add_module(str(i+1),t.nn.ReLU())
        i+=2
    return seq


We push `[-1, 2]` through the chain. Doubling gives `[-2, 4]`, the ReLU clamps the negative entry to `0`, and adding one gives `[1, 5]`. Predict what happens to `-1` if the ReLU is removed.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('x=t.tensor([-1.,2.])\nprint(m(x))\n# Hidden checks\nassert m(x).tolist()==[1.,5.]\n', globals()), end='')




Without the clamp the negative entry survives: `-2+1 = -1`. The two chains agree on the positive entry and disagree on the negative one, which is exactly the nonlinearity's job.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('n=t.nn.Sequential()\nn.add_module("0",Double())\nn.add_module("1",AddOne())\nprint(n(x), len(n))\n# Hidden checks\nassert n(x).tolist()==[-1.,5.] and len(n)==2\n', globals()), end='')


<!-- dd:dd-q1377 -->

### Problem 1377 · faded — your turn

Given a Sequential chain `m`, a tensor `x` and an integer `k` between 0 and the number of children, return the result of applying only the FIRST `k` children of `m` to `x` in order. A `k` of 0 returns `x` itself; a `k` equal to the number of children gives the same as the full chain.


In [ ]:
import torch as t

def solve(m,x,k):
    for mod in _____:
        x=_____
    return x


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1377)


In [ ]:
#@title 💡 Solution — Problem 1377
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x,k):
    for mod in list(m)[:k]:
        x=mod(x)
    return x


<!-- dd:dd-seg-cnn-sequential-2 -->

### A layer is an object with its weight inside


`t.nn.Linear(in_features, out_features)` is the affine map from the linear-layer lesson packaged as a module: it owns `.weight`, a parameter of shape `(out_features, in_features)`, and `.bias` of shape `(out_features,)`, both filled with random values at construction, and its forward computes `x @ weight.T + bias` on the last axis. Passing `bias=False` leaves the bias out. Because the weight is a parameter, setting it by hand is an in-place write under `t.no_grad()`, the same way the batch-norm buffers were updated.

The reason the layer carries its own weight is that a chain has nothing to pass alongside `x`: each step receives one tensor and returns one tensor, so any state a step needs must live inside the step. That is what makes `Sequential(Linear, ReLU, Linear)` a complete network rather than a recipe that still wants its weights handed in.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nlin=t.nn.Linear(2,1,bias=False)\nprint(lin.weight.shape)\nwith t.no_grad():\n    lin.weight.copy_(t.tensor([[1.,-1.]]))\nprint(lin(t.tensor([3.,1.])))\n# Hidden checks\nassert lin.weight.shape==(1,2) and lin(t.tensor([3.,1.])).tolist()==[2.]\n', globals()), end='')


We build a one-layer network with a ReLU after it and feed two examples. The layer computes `3-1 = 2` for the first row and `1-3 = -2` for the second; the ReLU keeps `2` and clamps `-2` to `0`. Predict the output shape before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('net=t.nn.Sequential(lin,t.nn.ReLU())\nx=t.tensor([[3.,1.],[1.,3.]])\nprint(net(x))\n# Hidden checks\nassert net(x).detach().tolist()==[[2.],[0.]]\n', globals()), end='')




The weight is a real parameter: it requires a gradient and it is the only entry the chain would hand an optimizer. The ReLU contributes nothing, which is why the count is one.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(net[0].weight.requires_grad, len(list(net.state_dict())))\n# Hidden checks\nassert net[0].weight.requires_grad and len(list(net.state_dict()))==1\n', globals()), end='')


<!-- dd:dd-q1378 -->

### Problem 1378 · faded — your turn

Given a 2-D float tensor `w` of shape `(out, in)`, return a library Linear layer with no bias whose weight parameter equals `w`. Its forward maps a tensor of shape `(..., in)` to `(..., out)` by multiplying with the transpose of `w`.


In [ ]:
import torch as t

def solve(w):
    lin=t.nn._____(w.shape[1],w.shape[0],bias=False)
    with t.no_grad():
        lin._____.copy_(w)
    return lin


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1378)


In [ ]:
#@title 💡 Solution — Problem 1378
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w):
    lin=t.nn.Linear(w.shape[1],w.shape[0],bias=False)
    with t.no_grad():
        lin.weight.copy_(w)
    return lin


We build a one-layer network with a ReLU after it and feed two examples. The layer computes `3-1 = 2` for the first row and `1-3 = -2` for the second; the ReLU keeps `2` and clamps `-2` to `0`. Predict the output shape before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('net=t.nn.Sequential(lin,t.nn.ReLU())\nx=t.tensor([[3.,1.],[1.,3.]])\nprint(net(x))\n# Hidden checks\nassert net(x).detach().tolist()==[[2.],[0.]]\n', globals()), end='')




The weight is a real parameter: it requires a gradient and it is the only entry the chain would hand an optimizer. The ReLU contributes nothing, which is why the count is one.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(net[0].weight.requires_grad, len(list(net.state_dict())))\n# Hidden checks\nassert net[0].weight.requires_grad and len(list(net.state_dict()))==1\n', globals()), end='')


<!-- dd:dd-q1379 -->

### Problem 1379 · faded — your turn

Given `w1` of shape `(hidden, in)` and `w2` of shape `(out, hidden)`, return a Sequential chain of three children: a bias-free Linear layer whose weight is `w1`, a ReLU module, and a bias-free Linear layer whose weight is `w2`. Applied to a `(batch, in)` input it returns `(batch, out)`.


In [ ]:
import torch as t

def solve(w1,w2):
    a=t.nn._____(w1.shape[1],w1.shape[0],bias=False)
    b=t.nn._____(w2.shape[1],w2.shape[0],bias=False)
    with t.no_grad():
        a._____.copy_(w1)
        b._____.copy_(w2)
    return t.nn._____(a,t.nn._____(),b)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1379)


In [ ]:
#@title 💡 Solution — Problem 1379
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w1,w2):
    a=t.nn.Linear(w1.shape[1],w1.shape[0],bias=False)
    b=t.nn.Linear(w2.shape[1],w2.shape[0],bias=False)
    with t.no_grad():
        a.weight.copy_(w1)
        b.weight.copy_(w2)
    return t.nn.Sequential(a,t.nn.ReLU(),b)


<!-- dd:dd-q1380 -->

### Problem 1380 · independent

Given a Sequential chain `m`, return a NEW chain of the same library type containing the same child modules in reverse order, registered under the names "0", "1", … . `m` itself must be left unchanged.


In [ ]:
import torch as t

def solve(m):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1380)


In [ ]:
#@title 💡 Solution — Problem 1380
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m):
    rev=t.nn.Sequential()
    i=0
    for mod in list(m)[::-1]:
        rev.add_module(str(i),mod)
        i+=1
    return rev


<!-- dd:dd-q1381 -->

### Problem 1381 · independent

Given a module `mod` and an integer `k` of at least 1, return a Sequential chain that applies `mod` `k` times in a row: the same module object registered `k` times under the names "0" … "k-1".


In [ ]:
import torch as t

def solve(mod,k):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1381)


In [ ]:
#@title 💡 Solution — Problem 1381
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(mod,k):
    seq=t.nn.Sequential()
    i=0
    for step in [mod]*k:
        seq.add_module(str(i),step)
        i+=1
    return seq


<!-- dd:dd-q1382 -->

### Problem 1382 · independent

Given a Sequential chain `m`, return the number of its direct children whose type is exactly the library ReLU module, as a Python int.


In [ ]:
import torch as t

def solve(m):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1382)


In [ ]:
#@title 💡 Solution — Problem 1382
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m):
    n=0
    for mod in m:
        if type(mod) is t.nn.ReLU:
            n+=1
    return n


<!-- dd:dd-q1383 -->

### Problem 1383 · independent

Given a Sequential chain `m` that contains at least one library Linear layer among its direct children, return the weight tensor of the LAST such child, shape `(out, in)`.


In [ ]:
import torch as t

def solve(m):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1383)


In [ ]:
#@title 💡 Solution — Problem 1383
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m):
    for mod in list(m)[::-1]:
        if type(mod) is t.nn.Linear:
            return mod.weight


<!-- dd:dd-q1384 -->

### Problem 1384 · independent

Given a Sequential chain `m`, an index `i` (possibly negative), a module `mod` and a tensor `x`: replace the child at position `i` of `m` by `mod` using index assignment, and return the output of the modified chain for `x`.


In [ ]:
import torch as t

def solve(m,i,mod,x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1384)


In [ ]:
#@title 💡 Solution — Problem 1384
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,i,mod,x):
    m[i]=mod
    return m(x)


<!-- dd:dd-q1385 -->

### Problem 1385 · independent

Given a Sequential chain `m` and a tensor `x`, return the list of intermediate results: element `i` is the tensor after the first `i+1` children have been applied, so the list has one entry per child and its last entry equals the full chain's output. Return an empty list for an empty chain.


In [ ]:
import torch as t

def solve(m,x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1385)


In [ ]:
#@title 💡 Solution — Problem 1385
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x):
    outs=[]
    for mod in m:
        x=mod(x)
        outs.append(x)
    return outs


#### Common mistakes

- **A Python list of modules is a chain.** Only registered children are seen by `parameters()` and `state_dict()`; a list attribute computes but cannot be trained or saved.
- **`Sequential` applies modules to the original input.** Each step receives the previous step's output; only the first sees `x`.
- **A `ReLU` module has parameters.** It is a parameter-free step; it exists as a module so it can sit in a chain.
- **`t.nn.Linear(a, b)` has a weight of shape `(a, b)`.** The weight is `(out, in)`, so `(b, a)`; the forward multiplies by its transpose.


<!-- dd:dd-kp-cnn-residual-block -->

## The residual block

`cnn.residual-block`


<!-- dd:dd-seg-cnn-residual-block-0 -->

### The left branch is conv, norm, ReLU, conv, norm


The convolution and batch-norm lessons built both operations by hand; the library packages them as layers. `t.nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=s, padding=1, bias=False)` owns a weight of shape `(out, in, 3, 3)` and maps `(batch, in, h, w)` to `(batch, out, h', w')`, where padding `1` with a `3×3` kernel keeps the spatial size at stride `1` and halves it (rounding up) at stride `2`. `t.nn.BatchNorm2d(out)` owns the per-channel scale, shift and running statistics of the batch-norm lesson and keeps the shape. The bias is left out of the convolution because the batch-norm that follows subtracts the channel mean, which in training mode would cancel a per-channel bias exactly (the batch norm's own shift plays that role instead).

A residual block's left branch is the chain conv → norm → ReLU → conv → norm, with the stride applied only in the first convolution. The reason the second convolution has stride `1` and `out` input channels is that the first has already done the resizing: after it, the tensor is at the block's output shape, and everything that follows works at that shape.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nconv=t.nn.Conv2d(3,8,kernel_size=3,stride=2,padding=1,bias=False)\nbn=t.nn.BatchNorm2d(8)\nx=t.zeros(2,3,6,6)\nprint(conv.weight.shape, bn(conv(x)).shape)\n# Hidden checks\nassert conv.weight.shape==(8,3,3,3) and bn(conv(x)).shape==(2,8,3,3)\n', globals()), end='')


We give a one-channel `3×3` convolution all-ones weights and feed a `3×3` image of ones with padding `1`. Each output pixel counts how many of its nine neighbours lie inside the image: `9` at the centre, `6` on an edge, `4` in a corner. Predict the corner value before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nconv=t.nn.Conv2d(1,1,kernel_size=3,stride=1,padding=1,bias=False)\nwith t.no_grad():\n    conv.weight.copy_(t.ones(1,1,3,3))\nx=t.ones(1,1,3,3)\nprint(conv(x)[0,0])\n# Hidden checks\nassert conv(x)[0,0].detach().tolist()==[[4.,6.,4.],[6.,9.,6.],[4.,6.,4.]]\n', globals()), end='')




In evaluation mode a fresh batch-norm uses its running mean `0` and running variance `1`, so it leaves the values essentially unchanged, and the ReLU after it changes nothing because every count is positive. The left branch of a block is this convolution–norm pair twice over, with a ReLU between the two pairs and none after the second.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('bn=t.nn.BatchNorm2d(1)\nbn.eval()\nleft=t.nn.Sequential(conv,bn,t.nn.ReLU())\nprint(left(x)[0,0,1,1])\n# Hidden checks\nassert abs(left(x)[0,0,1,1].item()-9.)<1e-3\n', globals()), end='')


<!-- dd:dd-q1389 -->

### Problem 1389 · faded — your turn

Given channel counts `i`, `o` and a stride `s`, return the LEFT branch of a residual block as a Sequential chain of five children: a `3×3` convolution from `i` to `o` channels with stride `s`, padding 1 and no bias; a 2-D batch-norm over `o` channels; a ReLU; a `3×3` convolution from `o` to `o` channels with stride 1, padding 1 and no bias; and a second batch-norm over `o` channels. Applied to `(batch, i, h, w)` it returns `(batch, o, h', w')` with each spatial size divided by `s` and rounded up.


In [ ]:
import torch as t

def solve(i,o,s):
    return t.nn.Sequential(t.nn._____(i,o,kernel_size=3,stride=s,padding=1,bias=False),t.nn._____(o),t.nn.ReLU(),t.nn._____(o,o,kernel_size=3,stride=1,padding=1,bias=False),t.nn._____(o))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1389)


In [ ]:
#@title 💡 Solution — Problem 1389
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(i,o,s):
    return t.nn.Sequential(t.nn.Conv2d(i,o,kernel_size=3,stride=s,padding=1,bias=False),t.nn.BatchNorm2d(o),t.nn.ReLU(),t.nn.Conv2d(o,o,kernel_size=3,stride=1,padding=1,bias=False),t.nn.BatchNorm2d(o))


We give a one-channel `3×3` convolution all-ones weights and feed a `3×3` image of ones with padding `1`. Each output pixel counts how many of its nine neighbours lie inside the image: `9` at the centre, `6` on an edge, `4` in a corner. Predict the corner value before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nconv=t.nn.Conv2d(1,1,kernel_size=3,stride=1,padding=1,bias=False)\nwith t.no_grad():\n    conv.weight.copy_(t.ones(1,1,3,3))\nx=t.ones(1,1,3,3)\nprint(conv(x)[0,0])\n# Hidden checks\nassert conv(x)[0,0].detach().tolist()==[[4.,6.,4.],[6.,9.,6.],[4.,6.,4.]]\n', globals()), end='')




In evaluation mode a fresh batch-norm uses its running mean `0` and running variance `1`, so it leaves the values essentially unchanged, and the ReLU after it changes nothing because every count is positive. The left branch of a block is this convolution–norm pair twice over, with a ReLU between the two pairs and none after the second.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('bn=t.nn.BatchNorm2d(1)\nbn.eval()\nleft=t.nn.Sequential(conv,bn,t.nn.ReLU())\nprint(left(x)[0,0,1,1])\n# Hidden checks\nassert abs(left(x)[0,0,1,1].item()-9.)<1e-3\n', globals()), end='')


<!-- dd:dd-q1390 -->

### Problem 1390 · faded — your turn

Given a float image batch `x` of shape `(batch, c, h, w)` and a stride `s`, return the output of a `3×3` convolution layer with padding 1, stride `s`, no bias, `c` input channels and ONE output channel, whose every weight equals `1/(9*c)` — so each output pixel is the mean of the `3×3` neighbourhood across all channels, with zeros outside the image. Output shape `(batch, 1, h', w')` with each spatial size divided by `s` and rounded up.


In [ ]:
import torch as t

def solve(x,s):
    c=x.shape[1]
    conv=t.nn._____(c,1,kernel_size=3,stride=s,padding=1,bias=False)
    with t.no_grad():
        conv.weight.copy_(t.ones(1,c,3,3)/(9*c))
    return conv(x)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1390)


In [ ]:
#@title 💡 Solution — Problem 1390
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,s):
    c=x.shape[1]
    conv=t.nn.Conv2d(c,1,kernel_size=3,stride=s,padding=1,bias=False)
    with t.no_grad():
        conv.weight.copy_(t.ones(1,c,3,3)/(9*c))
    return conv(x)


<!-- dd:dd-seg-cnn-residual-block-1 -->

### The shortcut must match the left branch, then the two add


A residual block returns `relu(left(x) + right(x))`: the left branch computes a change and the right branch, the *shortcut*, carries `x` across so the block only has to learn the difference from its input: while the left branch is small the block is close to the identity. That only adds up if both branches have the same shape. When the block keeps its shape — stride `1` and `in == out`, which is the single test `(s, i) != (1, o)` failing — the shortcut is `t.nn.Identity()`, a module whose forward returns its input untouched. When the block changes the channel count or the stride, the shortcut is a `1×1` convolution with the same stride, followed by a batch-norm, so that it resizes `x` the same way the left branch did.

The reason for a `1×1` kernel rather than `3×3` is that the shortcut is not supposed to look at neighbours; it only re-mixes channels and, through its stride, keeps every `s`-th pixel. The reason the ReLU comes after the sum rather than at the end of the left branch is that a ReLU before the sum would let the block only ever add to `x`, never subtract.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\ndef right_branch(i,o,s):\n    if (s,i)!=(1,o):\n        return t.nn.Sequential(t.nn.Conv2d(i,o,kernel_size=1,stride=s,bias=False),t.nn.BatchNorm2d(o))\n    return t.nn.Identity()\nprint(type(right_branch(4,4,1)), len(right_branch(4,8,2)))\n# Hidden checks\nassert type(right_branch(4,4,1)) is t.nn.Identity and len(right_branch(4,8,2))==2\n', globals()), end='')


We build a shape-preserving block on a one-channel `3×3` image of ones, with every convolution weight set to one and both batch-norms in evaluation mode. The first convolution gives the neighbour counts from before; the second convolution of that map has centre `4+6+4+6+9+6+4+6+4 = 49`. Predict the block's centre value: the shortcut adds the input's `1`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass Block(t.nn.Module):\n    def __init__(self,i,o,s):\n        super().__init__()\n        self.left=t.nn.Sequential(t.nn.Conv2d(i,o,kernel_size=3,stride=s,padding=1,bias=False),t.nn.BatchNorm2d(o),t.nn.ReLU(),t.nn.Conv2d(o,o,kernel_size=3,stride=1,padding=1,bias=False),t.nn.BatchNorm2d(o))\n        self.right=right_branch(i,o,s)\n        self.relu=t.nn.ReLU()\n    def forward(self,x):\n        a=self.left(x)\n        b=self.right(x)\n        return self.relu(a+b)\nb=Block(1,1,1)\nprint(type(b.right))\n# Hidden checks\nassert type(b.right) is t.nn.Identity\n', globals()), end='')




The sum is `49 + 1 = 50` at the centre; at a corner the second convolution sees `4+6+6+9 = 25` and on an edge `4+6+4+6+9+6 = 35`, and the shortcut adds `1` to each. Every number is positive so the final ReLU is invisible here.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('with t.no_grad():\n    for p in b.parameters():\n        if p.ndim==4:\n            p.copy_(t.ones(p.shape))\nb.eval()\nx=t.ones(1,1,3,3)\nprint(b(x)[0,0])\n# Hidden checks\nassert t.allclose(b(x)[0,0].detach(),t.tensor([[26.,36.,26.],[36.,50.,36.],[26.,36.,26.]]),atol=1e-2)\n', globals()), end='')


<!-- dd:dd-q1391 -->

### Problem 1391 · faded — your turn

Given channel counts `i`, `o` and a stride `s`, return the SHORTCUT (right branch) of a residual block: an Identity module when `s` is 1 and `i` equals `o`; otherwise a Sequential chain of a `1×1` convolution from `i` to `o` channels with stride `s`, no padding and no bias, followed by a 2-D batch-norm over `o` channels.


In [ ]:
import torch as t

def solve(i,o,s):
    if (s,i)!=(1,o):
        return t.nn.Sequential(t.nn._____(i,o,kernel_size=1,stride=s,bias=False),t.nn._____(o))
    return t.nn._____()


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1391)


In [ ]:
#@title 💡 Solution — Problem 1391
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(i,o,s):
    if (s,i)!=(1,o):
        return t.nn.Sequential(t.nn.Conv2d(i,o,kernel_size=1,stride=s,bias=False),t.nn.BatchNorm2d(o))
    return t.nn.Identity()


We build a shape-preserving block on a one-channel `3×3` image of ones, with every convolution weight set to one and both batch-norms in evaluation mode. The first convolution gives the neighbour counts from before; the second convolution of that map has centre `4+6+4+6+9+6+4+6+4 = 49`. Predict the block's centre value: the shortcut adds the input's `1`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass Block(t.nn.Module):\n    def __init__(self,i,o,s):\n        super().__init__()\n        self.left=t.nn.Sequential(t.nn.Conv2d(i,o,kernel_size=3,stride=s,padding=1,bias=False),t.nn.BatchNorm2d(o),t.nn.ReLU(),t.nn.Conv2d(o,o,kernel_size=3,stride=1,padding=1,bias=False),t.nn.BatchNorm2d(o))\n        self.right=right_branch(i,o,s)\n        self.relu=t.nn.ReLU()\n    def forward(self,x):\n        a=self.left(x)\n        b=self.right(x)\n        return self.relu(a+b)\nb=Block(1,1,1)\nprint(type(b.right))\n# Hidden checks\nassert type(b.right) is t.nn.Identity\n', globals()), end='')




The sum is `49 + 1 = 50` at the centre; at a corner the second convolution sees `4+6+6+9 = 25` and on an edge `4+6+4+6+9+6 = 35`, and the shortcut adds `1` to each. Every number is positive so the final ReLU is invisible here.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('with t.no_grad():\n    for p in b.parameters():\n        if p.ndim==4:\n            p.copy_(t.ones(p.shape))\nb.eval()\nx=t.ones(1,1,3,3)\nprint(b(x)[0,0])\n# Hidden checks\nassert t.allclose(b(x)[0,0].detach(),t.tensor([[26.,36.,26.],[36.,50.,36.],[26.,36.,26.]]),atol=1e-2)\n', globals()), end='')


<!-- dd:dd-q1392 -->

### Problem 1392 · faded — your turn

Return a residual block module for channel counts `i`, `o` and stride `s` with three registered children named `left`, `right` and `relu`: `left` is the five-step branch (`3×3` conv with stride `s`, padding 1, no bias; batch-norm; ReLU; `3×3` conv stride 1, padding 1, no bias; batch-norm), `right` is the Identity when the shape is preserved and otherwise a `1×1` stride-`s` bias-free conv followed by a batch-norm, and the forward returns the ReLU of the sum of both branches.


In [ ]:
import torch as t

def solve(i,o,s):
    class Block(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.left=t.nn.Sequential(t.nn._____(i,o,kernel_size=3,stride=s,padding=1,bias=False),t.nn._____(o),t.nn.ReLU(),t.nn._____(o,o,kernel_size=3,stride=1,padding=1,bias=False),t.nn._____(o))
            if (s,i)!=(1,o):
                self.right=t.nn.Sequential(t.nn._____(i,o,kernel_size=1,stride=s,bias=False),t.nn._____(o))
            else:
                self.right=t.nn._____()
            self.relu=t.nn.ReLU()
        def forward(self,x):
            return self.relu(self.left(x)+self.right(x))
    return Block()


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1392)


In [ ]:
#@title 💡 Solution — Problem 1392
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(i,o,s):
    class Block(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.left=t.nn.Sequential(t.nn.Conv2d(i,o,kernel_size=3,stride=s,padding=1,bias=False),t.nn.BatchNorm2d(o),t.nn.ReLU(),t.nn.Conv2d(o,o,kernel_size=3,stride=1,padding=1,bias=False),t.nn.BatchNorm2d(o))
            if (s,i)!=(1,o):
                self.right=t.nn.Sequential(t.nn.Conv2d(i,o,kernel_size=1,stride=s,bias=False),t.nn.BatchNorm2d(o))
            else:
                self.right=t.nn.Identity()
            self.relu=t.nn.ReLU()
        def forward(self,x):
            return self.relu(self.left(x)+self.right(x))
    return Block()


<!-- dd:dd-q1393 -->

### Problem 1393 · independent

Without building any module, return the output shape of a residual block as a tuple of four Python ints: the block takes `(b, i, h, w)`, has `o` output channels and stride `s`, and every spatial size is rounded UP when divided by `s`.


In [ ]:
import torch as t

def solve(b,i,o,h,w,s):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1393)


In [ ]:
#@title 💡 Solution — Problem 1393
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(b,i,o,h,w,s):
    return (b,o,(h+s-1)//s,(w+s-1)//s)


<!-- dd:dd-q1394 -->

### Problem 1394 · independent

Given two modules `left` and `right` that return tensors of the same shape for `x`, return the residual combination: the ReLU of the sum of the two branch outputs, computed without defining a class.


In [ ]:
import torch as t

def solve(left,right,x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1394)


In [ ]:
#@title 💡 Solution — Problem 1394
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(left,right,x):
    return t.nn.ReLU()(left(x)+right(x))


<!-- dd:dd-q1395 -->

### Problem 1395 · independent

Given a module `f` that preserves its input's shape, return a module wrapping it as an identity-shortcut residual step: the forward returns the ReLU of the sum of the input and the output of `f`, and `f` is registered as the child named `f`.


In [ ]:
import torch as t

def solve(f):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1395)


In [ ]:
#@title 💡 Solution — Problem 1395
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(f):
    class Res(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.f=f
            self.relu=t.nn.ReLU()
        def forward(self,x):
            return self.relu(x+self.f(x))
    return Res()


<!-- dd:dd-q1396 -->

### Problem 1396 · independent

Given a float batch `x` of shape `(batch, c, h, w)` and a stride `s`, return the identity-shortcut downsample as a convolution: build a `1×1` convolution layer with stride `s`, no padding, no bias and `c` input and output channels, set its weight so that each output channel copies the same-index input channel, and return its output. The result equals keeping every `s`-th row and column of `x`.


In [ ]:
import torch as t

def solve(x,s):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1396)


In [ ]:
#@title 💡 Solution — Problem 1396
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,s):
    c=x.shape[1]
    conv=t.nn.Conv2d(c,c,kernel_size=1,stride=s,bias=False)
    with t.no_grad():
        conv.weight.copy_(t.eye(c).reshape(c,c,1,1))
    return conv(x)


<!-- dd:dd-q1397 -->

### Problem 1397 · independent

Return a residual block for channel counts `i`, `o` and stride `s` whose shortcut is ALWAYS a projection: the child `right` is a `1×1` stride-`s` bias-free convolution from `i` to `o` channels followed by a batch-norm, even when the shape is preserved. The child `left` is the usual five-step branch and `relu` is the final ReLU applied to the sum.


In [ ]:
import torch as t

def solve(i,o,s):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1397)


In [ ]:
#@title 💡 Solution — Problem 1397
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(i,o,s):
    class Block(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.left=t.nn.Sequential(t.nn.Conv2d(i,o,kernel_size=3,stride=s,padding=1,bias=False),t.nn.BatchNorm2d(o),t.nn.ReLU(),t.nn.Conv2d(o,o,kernel_size=3,stride=1,padding=1,bias=False),t.nn.BatchNorm2d(o))
            self.right=t.nn.Sequential(t.nn.Conv2d(i,o,kernel_size=1,stride=s,bias=False),t.nn.BatchNorm2d(o))
            self.relu=t.nn.ReLU()
        def forward(self,x):
            return self.relu(self.left(x)+self.right(x))
    return Block()


<!-- dd:dd-q1398 -->

### Problem 1398 · independent

Return a SHALLOW residual block for `i`, `o`, `s`: the child `left` is only conv → batch-norm (one `3×3` bias-free convolution with stride `s` and padding 1, then a batch-norm over `o` channels), `right` is the usual shortcut (Identity when shape-preserving, otherwise `1×1` stride-`s` bias-free conv plus batch-norm), and the forward returns the ReLU of the sum.


In [ ]:
import torch as t

def solve(i,o,s):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1398)


In [ ]:
#@title 💡 Solution — Problem 1398
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(i,o,s):
    class Block(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.left=t.nn.Sequential(t.nn.Conv2d(i,o,kernel_size=3,stride=s,padding=1,bias=False),t.nn.BatchNorm2d(o))
            if (s,i)!=(1,o):
                self.right=t.nn.Sequential(t.nn.Conv2d(i,o,kernel_size=1,stride=s,bias=False),t.nn.BatchNorm2d(o))
            else:
                self.right=t.nn.Identity()
            self.relu=t.nn.ReLU()
        def forward(self,x):
            return self.relu(self.left(x)+self.right(x))
    return Block()


#### Common mistakes

- **The shortcut is always the identity.** Only when the shape is preserved; a stride or a channel change needs a `1×1` convolution and a batch-norm so the two branches can add.
- **Both convolutions in the left branch carry the stride.** Only the first does; the second works at the output shape with stride `1`.
- **The ReLU goes at the end of the left branch.** It goes after the sum; a ReLU before the sum would make the block unable to subtract from `x`.
- **A convolution followed by batch-norm needs its bias.** The norm subtracts the channel mean, which cancels the bias exactly; it is dead weight.


<!-- dd:dd-kp-cnn-block-group -->

## A group of residual blocks

`cnn.block-group`


### Only the first block of a group resizes


A ResNet is built from *groups* of residual blocks. Within a group every block has the same output channel count `o`, and the whole group's change of shape happens in its first block: that block takes the group's input channels `i` and the group's stride `s`, so it is the only block that can need a projection shortcut (it needs one exactly when `s` is not `1` or `i` differs from `o`); every later block is `(o, o, 1)` — same channels in and out, stride `1`, identity shortcut. A group of `n` blocks is therefore a chain of one resizing block followed by `n - 1` shape-preserving ones, and its output is `(batch, o, h', w')` with the spatial size divided by `s` once, rounded up.

The reason the resizing is done once, up front, is that the group is meant to work at one resolution: the first block moves the tensor to that resolution and channel count, and the remaining blocks refine it there. Doing the stride in every block would shrink the image `n` times; and once the first block has produced `o` channels every later block must accept `o` — a later block built for `i` inputs would fail on a channel mismatch whenever `i` differs from `o`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass Block(t.nn.Module):\n    def __init__(self,i,o,s):\n        super().__init__()\n        self.left=t.nn.Sequential(t.nn.Conv2d(i,o,kernel_size=3,stride=s,padding=1,bias=False),t.nn.BatchNorm2d(o),t.nn.ReLU(),t.nn.Conv2d(o,o,kernel_size=3,stride=1,padding=1,bias=False),t.nn.BatchNorm2d(o))\n        self.right=t.nn.Identity() if (s,i)==(1,o) else t.nn.Sequential(t.nn.Conv2d(i,o,kernel_size=1,stride=s,bias=False),t.nn.BatchNorm2d(o))\n        self.relu=t.nn.ReLU()\n    def forward(self,x):\n        a=self.left(x)\n        b=self.right(x)\n        return self.relu(a+b)\nprint(Block(4,8,2)(t.zeros(1,4,6,6)).shape)\n# Hidden checks\nassert Block(4,8,2)(t.zeros(1,4,6,6)).shape==(1,8,3,3)\n', globals()), end='')


We build a group of three blocks that takes `4` channels to `8` at stride `2`. The first block is `Block(4, 8, 2)`; the other two are `Block(8, 8, 1)`. Predict how many of the three blocks have an Identity shortcut before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('class Group(t.nn.Module):\n    def __init__(self,n,i,o,s):\n        super().__init__()\n        self.blocks=t.nn.Sequential()\n        self.blocks.add_module("0",Block(i,o,s))\n        k=1\n        for extra in [o]*(n-1):\n            self.blocks.add_module(str(k),Block(o,o,1))\n            k+=1\n    def forward(self,x):\n        return self.blocks(x)\ng=Group(3,4,8,2)\nprint(len(g.blocks), [type(b.right) is t.nn.Identity for b in g.blocks])\n# Hidden checks\nassert len(g.blocks)==3 and [type(b.right) is t.nn.Identity for b in g.blocks]==[False,True,True]\n', globals()), end='')




Two of the three: only the first block changes shape, so only the first needs a projection. A `6×6` input comes out `3×3` with `8` channels, halved exactly once.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('x=t.zeros(2,4,6,6)\nprint(g(x).shape)\n# Hidden checks\nassert g(x).shape==(2,8,3,3)\n', globals()), end='')


<!-- dd:dd-q1402 -->

### Problem 1402 · faded — your turn

Return a block-group module for `n` blocks, input channels `i`, output channels `o` and first stride `s`: its single child `blocks` is a Sequential chain of `n` residual blocks registered under the names "0", "1", …, where only the first block changes the channel count (from `i` to `o`) and applies the stride `s`, and every later block keeps `o` channels at stride 1; the forward applies the chain. A class `ResidualBlock` taking (in channels, out channels, stride) is provided.


In [ ]:
import torch as t

def solve(n,i,o,s):
    class BlockGroup(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.blocks=t.nn.Sequential()
            self.blocks.add_module("0",ResidualBlock(_____,_____,_____))
            k=1
            for extra in [o]*(n-1):
                self.blocks.add_module(str(k),ResidualBlock(_____,_____,_____))
                k+=1
        def forward(self,x):
            return self.blocks(x)
    return BlockGroup()


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1402)


In [ ]:
#@title 💡 Solution — Problem 1402
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n,i,o,s):
    class BlockGroup(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.blocks=t.nn.Sequential()
            self.blocks.add_module("0",ResidualBlock(i,o,s))
            k=1
            for extra in [o]*(n-1):
                self.blocks.add_module(str(k),ResidualBlock(o,o,1))
                k+=1
        def forward(self,x):
            return self.blocks(x)
    return BlockGroup()


We build a group of three blocks that takes `4` channels to `8` at stride `2`. The first block is `Block(4, 8, 2)`; the other two are `Block(8, 8, 1)`. Predict how many of the three blocks have an Identity shortcut before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('class Group(t.nn.Module):\n    def __init__(self,n,i,o,s):\n        super().__init__()\n        self.blocks=t.nn.Sequential()\n        self.blocks.add_module("0",Block(i,o,s))\n        k=1\n        for extra in [o]*(n-1):\n            self.blocks.add_module(str(k),Block(o,o,1))\n            k+=1\n    def forward(self,x):\n        return self.blocks(x)\ng=Group(3,4,8,2)\nprint(len(g.blocks), [type(b.right) is t.nn.Identity for b in g.blocks])\n# Hidden checks\nassert len(g.blocks)==3 and [type(b.right) is t.nn.Identity for b in g.blocks]==[False,True,True]\n', globals()), end='')




Two of the three: only the first block changes shape, so only the first needs a projection. A `6×6` input comes out `3×3` with `8` channels, halved exactly once.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('x=t.zeros(2,4,6,6)\nprint(g(x).shape)\n# Hidden checks\nassert g(x).shape==(2,8,3,3)\n', globals()), end='')


<!-- dd:dd-q1403 -->

### Problem 1403 · faded — your turn

Without building any module, return the list of `(in, out, stride)` argument triples for the `n` blocks of a group with input channels `i`, output channels `o` and first stride `s`: the first triple is `(i, o, s)` and each of the other `n - 1` triples is `(o, o, 1)`. Return a list of tuples.


In [ ]:
import torch as t

def solve(n,i,o,s):
    return [(_____,_____,_____)]+[(_____,_____,_____)]*(n-1)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1403)


In [ ]:
#@title 💡 Solution — Problem 1403
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n,i,o,s):
    return [(i,o,s)]+[(o,o,1)]*(n-1)


<!-- dd:dd-q1404 -->

### Problem 1404 · independent

Given the first group's input channel count `c0` and three equal-length lists `n_blocks`, `out_feats` and `strides`, return the list of `(n, i, o, s)` argument tuples for building each group in order: `i` is `c0` for the first group and the previous group's `o` for every later group.


In [ ]:
import torch as t

def solve(c0,n_blocks,out_feats,strides):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1404)


In [ ]:
#@title 💡 Solution — Problem 1404
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(c0,n_blocks,out_feats,strides):
    args=[]
    i=c0
    k=0
    for n in n_blocks:
        args.append((n,i,out_feats[k],strides[k]))
        i=out_feats[k]
        k+=1
    return args


<!-- dd:dd-q1405 -->

### Problem 1405 · independent

Without building any module, return the output shape of a group of `n` residual blocks applied to `(b, i, h, w)`, with output channels `o` and first stride `s`, as a tuple of four Python ints. Spatial sizes divide by the stride once and round up.


In [ ]:
import torch as t

def solve(b,i,o,h,w,n,s):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1405)


In [ ]:
#@title 💡 Solution — Problem 1405
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(b,i,o,h,w,n,s):
    return (b,o,(h+s-1)//s,(w+s-1)//s)


<!-- dd:dd-q1406 -->

### Problem 1406 · independent

Return a group module whose child `blocks` chains `n` blocks with the stride in the LAST block instead of the first: the first block takes `i` channels to `o` at stride 1, the middle ones keep `o` channels at stride 1, and the last block keeps `o` channels at stride `s`. When there is only one block it takes `i` to `o` at stride `s`. The forward applies the chain. A class `ResidualBlock` taking (in channels, out channels, stride) is provided.


In [ ]:
import torch as t

def solve(n,i,o,s):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1406)


In [ ]:
#@title 💡 Solution — Problem 1406
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n,i,o,s):
    class Group(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.blocks=t.nn.Sequential()
            if n==1:
                self.blocks.add_module("0",ResidualBlock(i,o,s))
            else:
                self.blocks.add_module("0",ResidualBlock(i,o,1))
                k=1
                for extra in [o]*(n-2):
                    self.blocks.add_module(str(k),ResidualBlock(o,o,1))
                    k+=1
                self.blocks.add_module(str(k),ResidualBlock(o,o,s))
        def forward(self,x):
            return self.blocks(x)
    return Group()


<!-- dd:dd-q1407 -->

### Problem 1407 · independent

Return a group module whose child `blocks` applies ONE shared shape-preserving block (channels `o` to `o`, stride 1) `n` times in a row (the same block object registered `n` times under "0", "1", …), so the group has exactly one block's worth of parameters. The forward applies the chain. A class `ResidualBlock` taking (in channels, out channels, stride) is provided.


In [ ]:
import torch as t

def solve(n,o):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1407)


In [ ]:
#@title 💡 Solution — Problem 1407
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n,o):
    class Group(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.blocks=t.nn.Sequential()
            block=ResidualBlock(o,o,1)
            k=0
            for step in [block]*n:
                self.blocks.add_module(str(k),step)
                k+=1
        def forward(self,x):
            return self.blocks(x)
    return Group()


<!-- dd:dd-q1408 -->

### Problem 1408 · independent

Without building any module, return the number of trainable parameters in a group of `n` residual blocks with input channels `i`, output channels `o` and first stride `s`, as a Python int. Count: each `3×3` bias-free convolution from `a` to `b` channels has `b*a*9` weights, each batch-norm over `b` channels has `2*b` (scale and shift), and a projection shortcut (`1×1` conv `i`→`o` plus batch-norm) is present in the first block exactly when the stride is not 1 or `i` differs from `o`.


In [ ]:
import torch as t

def solve(n,i,o,s):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1408)


In [ ]:
#@title 💡 Solution — Problem 1408
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n,i,o,s):
    first=o*i*9+2*o+o*o*9+2*o
    if (s,i)!=(1,o):
        first+=o*i+2*o
    rest=(o*o*9+2*o)*2
    return first+rest*(n-1)


<!-- dd:dd-q1409 -->

### Problem 1409 · independent

Return a group module whose child `blocks` chains `n` blocks that DOUBLE the channels at every block: block `k` maps `i*2**k` channels to `i*2**(k+1)`, and only the first block uses stride `s` (the rest stride 1). The forward applies the chain. A class `ResidualBlock` taking (in channels, out channels, stride) is provided.


In [ ]:
import torch as t

def solve(n,i,s):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1409)


In [ ]:
#@title 💡 Solution — Problem 1409
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(n,i,s):
    class Group(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.blocks=t.nn.Sequential()
            c=i
            k=0
            for step in [0]*n:
                self.blocks.add_module(str(k),ResidualBlock(c,2*c,s if k==0 else 1))
                c=2*c
                k+=1
        def forward(self,x):
            return self.blocks(x)
    return Group()


#### Common mistakes

- **Every block in a group carries the stride.** Only the first; a stride in each of `n` blocks would shrink the image `n` times.
- **Every block in a group maps `i` to `o`.** Only the first takes `i`; the rest are `(o, o, 1)`.
- **The group needs its own ReLU or norm after the blocks.** Each block already ends in a ReLU; the group is just the chain.
- **A group with `n = 1` is a plain block.** It is a chain of one resizing block — still wrapped, so groups compose uniformly.


<!-- dd:dd-kp-cnn-resnet34 -->

## Assembling ResNet34

`cnn.resnet34`


<!-- dd:dd-seg-cnn-resnet34-0 -->

### The stem shrinks the image four-fold before any block


ResNet34 opens with a *stem*: one wide convolution (`7×7` window, stride `2`, padding `3`, `3` input channels to `64`), a batch norm, a ReLU, and a max-pool with a `3×3` window, stride `2`, padding `1`. Both strided steps halve the spatial size, rounding up, so a `224×224` image reaches the first residual block as `64` channels of `56×56`. The stem is the only place in the network where a `7×7` window or a max-pool appears; everything after it is residual blocks, until the average-pool and linear head at the end.

The `padding = 3` on a `7×7` window and `padding = 1` on a `3×3` window are the "same" paddings: with stride `1` they would leave the image size unchanged, so with stride `2` each step gives exactly `ceil(size / 2)`. A max-pool module takes the same window, stride and padding arguments as a convolution but has no parameters — it keeps the largest value in each window per channel.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nstem=t.nn.Sequential(t.nn.Conv2d(3,64,kernel_size=7,stride=2,padding=3),t.nn.BatchNorm2d(64),t.nn.ReLU(),t.nn.MaxPool2d(kernel_size=3,stride=2,padding=1))\nprint(stem(t.zeros(1,3,224,224)).shape)\nprint(t.nn.MaxPool2d(kernel_size=3,stride=2,padding=1)(t.arange(16.).reshape(1,1,4,4)))\n# Hidden checks\nassert stem(t.zeros(1,3,224,224)).shape==(1,64,56,56)\nassert t.nn.MaxPool2d(kernel_size=3,stride=2,padding=1)(t.arange(16.).reshape(1,1,4,4)).tolist()==[[[[5.,7.],[13.,15.]]]]\n', globals()), end='')


We push a batch of two `3×30×30` images through a stem with `8` output channels. Predict the shape after the convolution and after the pool before running: each strided step gives `ceil(30 / 2) = 15`, then `ceil(15 / 2) = 8`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('stem=t.nn.Sequential(t.nn.Conv2d(3,8,kernel_size=7,stride=2,padding=3),t.nn.BatchNorm2d(8),t.nn.ReLU(),t.nn.MaxPool2d(kernel_size=3,stride=2,padding=1))\nx=t.zeros(2,3,30,30)\nprint(stem[0](x).shape, stem(x).shape)\n# Hidden checks\nassert stem[0](x).shape==(2,8,15,15) and stem(x).shape==(2,8,8,8)\n', globals()), end='')




The pool has no parameters, so the stem's parameter count is the convolution's plus the batch norm's:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(sum([p.numel() for p in stem.parameters()]), 8*3*49+8, 2*8)\n# Hidden checks\nassert sum([p.numel() for p in stem.parameters()])==8*3*49+8+2*8\n', globals()), end='')


<!-- dd:dd-q1413 -->

### Problem 1413 · faded — your turn

Return the ResNet stem for `c0` output channels as a Sequential chain of four modules: a convolution from `3` channels to `c0` with a `7×7` window, stride `2` and padding `3` (with bias), a batch norm over `c0` channels, a ReLU, and a max-pool with a `3×3` window, stride `2` and padding `1`.


In [ ]:
import torch as t

def solve(c0):
    return t.nn.Sequential(t.nn.Conv2d(3,c0,kernel_size=7,stride=2,padding=3),t.nn.BatchNorm2d(c0),t.nn.ReLU(),t.nn._____(kernel_size=3,stride=2,padding=1))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1413)


In [ ]:
#@title 💡 Solution — Problem 1413
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(c0):
    return t.nn.Sequential(t.nn.Conv2d(3,c0,kernel_size=7,stride=2,padding=3),t.nn.BatchNorm2d(c0),t.nn.ReLU(),t.nn.MaxPool2d(kernel_size=3,stride=2,padding=1))


We push a batch of two `3×30×30` images through a stem with `8` output channels. Predict the shape after the convolution and after the pool before running: each strided step gives `ceil(30 / 2) = 15`, then `ceil(15 / 2) = 8`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('stem=t.nn.Sequential(t.nn.Conv2d(3,8,kernel_size=7,stride=2,padding=3),t.nn.BatchNorm2d(8),t.nn.ReLU(),t.nn.MaxPool2d(kernel_size=3,stride=2,padding=1))\nx=t.zeros(2,3,30,30)\nprint(stem[0](x).shape, stem(x).shape)\n# Hidden checks\nassert stem[0](x).shape==(2,8,15,15) and stem(x).shape==(2,8,8,8)\n', globals()), end='')




The pool has no parameters, so the stem's parameter count is the convolution's plus the batch norm's:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(sum([p.numel() for p in stem.parameters()]), 8*3*49+8, 2*8)\n# Hidden checks\nassert sum([p.numel() for p in stem.parameters()])==8*3*49+8+2*8\n', globals()), end='')


<!-- dd:dd-q1414 -->

### Problem 1414 · faded — your turn

Without building any module, return the shape of the stem's output for a batch `(b, 3, h, w)` and stem width `c0`, as a tuple of four Python ints. The `7×7` stride-`2` padding-`3` convolution and the `3×3` stride-`2` padding-`1` pool each halve a spatial size, rounding up.


In [ ]:
import torch as t

def solve(b,c0,h,w):
    return (b,_____,_____,_____)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1414)


In [ ]:
#@title 💡 Solution — Problem 1414
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(b,c0,h,w):
    return (b,c0,(h+3)//4,(w+3)//4)


<!-- dd:dd-seg-cnn-resnet34-1 -->

### Stem, four stages, then average-pool and a linear head


The whole network is three Sequential chains in a row. `in_layers` is the stem. `residual_layers` is a chain of four block groups, with `[3, 4, 6, 3]` blocks, `[64, 128, 256, 512]` output channels and first strides `[1, 2, 2, 2]`; each group's input width is the previous group's output width, and the first group's is the stem's `64`. `out_layers` averages every channel over its remaining `7×7` spatial positions, leaving one number per channel, and a linear layer maps those `512` numbers to `n_classes` logits. Summing the block counts gives `3 + 4 + 6 + 3 = 16` blocks of two convolutions each, plus the stem convolution and the head: the "34" in the name.

Averaging over the spatial axes with `x.mean((2, 3))` turns `(batch, 512, h, w)` into `(batch, 512)` whatever `h` and `w` are, which is why the network accepts images of any spatial size, as long as every stride still leaves at least one position. It is wrapped in a tiny module so it can sit inside a Sequential chain next to the linear layer.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass ResidualBlock(t.nn.Module):\n    def __init__(self,i,o,s):\n        super().__init__()\n        self.left=t.nn.Sequential(t.nn.Conv2d(i,o,kernel_size=3,stride=s,padding=1,bias=False),t.nn.BatchNorm2d(o),t.nn.ReLU(),t.nn.Conv2d(o,o,kernel_size=3,stride=1,padding=1,bias=False),t.nn.BatchNorm2d(o))\n        self.right=t.nn.Identity() if (s,i)==(1,o) else t.nn.Sequential(t.nn.Conv2d(i,o,kernel_size=1,stride=s,bias=False),t.nn.BatchNorm2d(o))\n        self.relu=t.nn.ReLU()\n    def forward(self,x):\n        return self.relu(self.left(x)+self.right(x))\nclass BlockGroup(t.nn.Module):\n    def __init__(self,n,i,o,s):\n        super().__init__()\n        self.blocks=t.nn.Sequential()\n        self.blocks.add_module("0",ResidualBlock(i,o,s))\n        k=1\n        for extra in [o]*(n-1):\n            self.blocks.add_module(str(k),ResidualBlock(o,o,1))\n            k+=1\n    def forward(self,x):\n        return self.blocks(x)\n# Hidden checks\nassert BlockGroup(2,4,8,2)(t.zeros(1,4,8,8)).shape==(1,8,4,4)\n', globals()), end='')


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('class AveragePool(t.nn.Module):\n    def forward(self,x):\n        return x.mean((2,3))\nclass ResNet(t.nn.Module):\n    def __init__(self,c0,n_blocks,out_feats,strides,n_classes):\n        super().__init__()\n        self.in_layers=t.nn.Sequential(t.nn.Conv2d(3,c0,kernel_size=7,stride=2,padding=3),t.nn.BatchNorm2d(c0),t.nn.ReLU(),t.nn.MaxPool2d(kernel_size=3,stride=2,padding=1))\n        self.residual_layers=t.nn.Sequential()\n        i=c0\n        k=0\n        for n in n_blocks:\n            self.residual_layers.add_module(str(k),BlockGroup(n,i,out_feats[k],strides[k]))\n            i=out_feats[k]\n            k+=1\n        self.out_layers=t.nn.Sequential(AveragePool(),t.nn.Linear(out_feats[-1],n_classes))\n    def forward(self,x):\n        return self.out_layers(self.residual_layers(self.in_layers(x)))\n# Hidden checks\nassert ResNet(8,[1,1],[8,16],[1,2],5)(t.zeros(2,3,32,32)).shape==(2,5)\n', globals()), end='')


A scaled-down network: stem width `8`, two groups of one block with widths `[8, 16]` and strides `[1, 2]`, `5` classes. For a `3×32×32` image the stem gives `8×8×8`, the first group keeps `8×8`, the second halves it to `4×4` with `16` channels, the average-pool leaves `16` numbers, and the head gives `5` logits. Predict the three intermediate shapes before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('net=ResNet(8,[1,1],[8,16],[1,2],5)\nx=t.zeros(2,3,32,32)\na=net.in_layers(x)\nb=net.residual_layers(a)\nprint(a.shape, b.shape, net.out_layers(b).shape)\n# Hidden checks\nassert a.shape==(2,8,8,8) and b.shape==(2,16,4,4) and net.out_layers(b).shape==(2,5)\n', globals()), end='')




The real ResNet34 is the same class with the paper's numbers, and its last feature map for a `224×224` input is `512` channels of `7×7`:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('big=ResNet(64,[3,4,6,3],[64,128,256,512],[1,2,2,2],1000)\nprint(sum([len(g.blocks) for g in big.residual_layers]), big.residual_layers(big.in_layers(t.zeros(1,3,224,224))).shape)\n# Hidden checks\nassert sum([len(g.blocks) for g in big.residual_layers])==16\nassert big.residual_layers(big.in_layers(t.zeros(1,3,224,224))).shape==(1,512,7,7)\n', globals()), end='')


<!-- dd:dd-q1415 -->

### Problem 1415 · faded — your turn

Return a ResNet module with three children: `in_layers` (the stem for `c0` channels: `7×7` stride-`2` padding-`3` convolution from `3` channels with bias, batch norm, ReLU, `3×3` stride-`2` padding-`1` max-pool), `residual_layers` (a Sequential chain of one group per entry of `n_blocks`, with the matching entries of `out_feats` and `strides`, each group's input width being the previous group's output width and `c0` for the first), and `out_layers` (a module averaging over the two spatial axes followed by a linear layer from the last output width to `n_classes`). The forward applies the three in order. Classes `ResidualBlock` and `BlockGroup` (arguments: block count, in channels, out channels, first stride) are provided.


In [ ]:
import torch as t

def solve(c0,n_blocks,out_feats,strides,n_classes):
    class AveragePool(t.nn.Module):
        def forward(self,x):
            return x.mean((2,3))
    class ResNet(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.in_layers=t.nn.Sequential(t.nn.Conv2d(3,c0,kernel_size=7,stride=2,padding=3),t.nn.BatchNorm2d(c0),t.nn.ReLU(),t.nn._____(kernel_size=3,stride=2,padding=1))
            self.residual_layers=t.nn.Sequential()
            i=c0
            k=0
            for n in n_blocks:
                self.residual_layers.add_module(str(k),BlockGroup(_____,_____,_____,_____))
                i=_____
                k+=1
            self.out_layers=t.nn.Sequential(AveragePool(),t.nn.Linear(out_feats[-1],n_classes))
        def forward(self,x):
            return self.out_layers(self.residual_layers(self.in_layers(x)))
    return ResNet()


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1415)


In [ ]:
#@title 💡 Solution — Problem 1415
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(c0,n_blocks,out_feats,strides,n_classes):
    class AveragePool(t.nn.Module):
        def forward(self,x):
            return x.mean((2,3))
    class ResNet(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.in_layers=t.nn.Sequential(t.nn.Conv2d(3,c0,kernel_size=7,stride=2,padding=3),t.nn.BatchNorm2d(c0),t.nn.ReLU(),t.nn.MaxPool2d(kernel_size=3,stride=2,padding=1))
            self.residual_layers=t.nn.Sequential()
            i=c0
            k=0
            for n in n_blocks:
                self.residual_layers.add_module(str(k),BlockGroup(n,i,out_feats[k],strides[k]))
                i=out_feats[k]
                k+=1
            self.out_layers=t.nn.Sequential(AveragePool(),t.nn.Linear(out_feats[-1],n_classes))
        def forward(self,x):
            return self.out_layers(self.residual_layers(self.in_layers(x)))
    return ResNet()


A scaled-down network: stem width `8`, two groups of one block with widths `[8, 16]` and strides `[1, 2]`, `5` classes. For a `3×32×32` image the stem gives `8×8×8`, the first group keeps `8×8`, the second halves it to `4×4` with `16` channels, the average-pool leaves `16` numbers, and the head gives `5` logits. Predict the three intermediate shapes before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('net=ResNet(8,[1,1],[8,16],[1,2],5)\nx=t.zeros(2,3,32,32)\na=net.in_layers(x)\nb=net.residual_layers(a)\nprint(a.shape, b.shape, net.out_layers(b).shape)\n# Hidden checks\nassert a.shape==(2,8,8,8) and b.shape==(2,16,4,4) and net.out_layers(b).shape==(2,5)\n', globals()), end='')




The real ResNet34 is the same class with the paper's numbers, and its last feature map for a `224×224` input is `512` channels of `7×7`:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('big=ResNet(64,[3,4,6,3],[64,128,256,512],[1,2,2,2],1000)\nprint(sum([len(g.blocks) for g in big.residual_layers]), big.residual_layers(big.in_layers(t.zeros(1,3,224,224))).shape)\n# Hidden checks\nassert sum([len(g.blocks) for g in big.residual_layers])==16\nassert big.residual_layers(big.in_layers(t.zeros(1,3,224,224))).shape==(1,512,7,7)\n', globals()), end='')


<!-- dd:dd-q1416 -->

### Problem 1416 · faded — your turn

Return the classification head as a module: given a feature map `(batch, c, h, w)` it averages every channel over the two spatial axes to `(batch, c)` and applies a linear layer (child `lin`) from `c` to `n_classes`, returning `(batch, n_classes)` logits.


In [ ]:
import torch as t

def solve(c,n_classes):
    class Head(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.lin=t.nn.Linear(_____,_____)
        def forward(self,x):
            return self.lin(_____)
    return Head()


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1416)


In [ ]:
#@title 💡 Solution — Problem 1416
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(c,n_classes):
    class Head(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.lin=t.nn.Linear(c,n_classes)
        def forward(self,x):
            return self.lin(x.mean((2,3)))
    return Head()


<!-- dd:dd-q1417 -->

### Problem 1417 · independent

Without building any module, return the shape of the feature map that leaves the residual stages, for an input batch `(b, 3, h, w)`, group output widths `out_feats` and group first strides `strides`, as a tuple of four Python ints. The stem quarters each spatial size (rounding up at each of its two halvings), then each group divides by its first stride, rounding up.


In [ ]:
import torch as t

def solve(b,h,w,out_feats,strides):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1417)


In [ ]:
#@title 💡 Solution — Problem 1417
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(b,h,w,out_feats,strides):
    h=(h+1)//2
    w=(w+1)//2
    h=(h+1)//2
    w=(w+1)//2
    for s in strides:
        h=(h+s-1)//s
        w=(w+s-1)//s
    return (b,out_feats[-1],h,w)


<!-- dd:dd-q1418 -->

### Problem 1418 · independent

Two halvings throw away most of a small image's detail, so return a small-image stem for `c0` output channels: a Sequential chain of a convolution from `3` channels to `c0` with a `3×3` window, stride `1`, padding `1` and bias, a batch norm over `c0` channels, and a ReLU — no pooling, so the spatial size is unchanged.


In [ ]:
import torch as t

def solve(c0):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1418)


In [ ]:
#@title 💡 Solution — Problem 1418
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(c0):
    return t.nn.Sequential(t.nn.Conv2d(3,c0,kernel_size=3,stride=1,padding=1),t.nn.BatchNorm2d(c0),t.nn.ReLU())


<!-- dd:dd-q1419 -->

### Problem 1419 · independent

Return a feature network with children `in_layers` (the standard stem for `c0` channels: `7×7` stride-`2` padding-`3` convolution from `3` channels with bias, batch norm, ReLU, `3×3` stride-`2` padding-`1` max-pool) and `residual_layers` (the chained groups from `n_blocks`, `out_feats`, `strides`, input widths chained from `c0`), and NO head: the forward returns the spatially averaged features, shape `(batch, last output width)`. Classes `ResidualBlock` and `BlockGroup` (block count, in channels, out channels, first stride) are provided.


In [ ]:
import torch as t

def solve(c0,n_blocks,out_feats,strides):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1419)


In [ ]:
#@title 💡 Solution — Problem 1419
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(c0,n_blocks,out_feats,strides):
    class Features(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.in_layers=t.nn.Sequential(t.nn.Conv2d(3,c0,kernel_size=7,stride=2,padding=3),t.nn.BatchNorm2d(c0),t.nn.ReLU(),t.nn.MaxPool2d(kernel_size=3,stride=2,padding=1))
            self.residual_layers=t.nn.Sequential()
            i=c0
            k=0
            for n in n_blocks:
                self.residual_layers.add_module(str(k),BlockGroup(n,i,out_feats[k],strides[k]))
                i=out_feats[k]
                k+=1
        def forward(self,x):
            return self.residual_layers(self.in_layers(x)).mean((2,3))
    return Features()


<!-- dd:dd-q1420 -->

### Problem 1420 · independent

Without building any module, return the parameter count of the stem plus the head as a Python int: the stem is a `7×7` convolution from `3` channels to `c0` channels WITH bias followed by a batch norm (which has an affine weight and bias per channel) and a parameter-free pool; the head is a linear layer from `c` features to `n_classes` outputs with bias. Count every weight and bias value.


In [ ]:
import torch as t

def solve(c0,c,n_classes):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1420)


In [ ]:
#@title 💡 Solution — Problem 1420
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(c0,c,n_classes):
    return c0*3*49+c0+2*c0+c*n_classes+n_classes


<!-- dd:dd-q1421 -->

### Problem 1421 · independent

Given a ResNet-style module `net` already in evaluation mode and an image batch `x` of shape `(batch, 3, h, w)`, return the predicted class index of every image: the position of the largest logit along the class axis, as a 1-D integer tensor of length `batch`.


In [ ]:
import torch as t

def solve(net,x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1421)


In [ ]:
#@title 💡 Solution — Problem 1421
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(net,x):
    return net(x).argmax(dim=1)


<!-- dd:dd-q1422 -->

### Problem 1422 · independent

Return the stem for images with `cin` channels (for example `1` for grayscale) and `c0` output channels: a Sequential chain of a `7×7` stride-`2` padding-`3` convolution with bias from `cin` to `c0`, a batch norm over `c0` channels, a ReLU, and a `3×3` stride-`2` padding-`1` max-pool.


In [ ]:
import torch as t

def solve(cin,c0):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1422)


In [ ]:
#@title 💡 Solution — Problem 1422
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(cin,c0):
    return t.nn.Sequential(t.nn.Conv2d(cin,c0,kernel_size=7,stride=2,padding=3),t.nn.BatchNorm2d(c0),t.nn.ReLU(),t.nn.MaxPool2d(kernel_size=3,stride=2,padding=1))


#### Common mistakes

- **The stem's max-pool has weights.** It has none; the stem's parameters are the convolution's and the batch norm's.
- **Every group starts from `64` channels.** Only the first; each group's input width is the previous group's output width.
- **The head flattens the `7×7` map into `512 * 49` features.** It averages each channel to one number, so the head is `512 → n_classes` and the input size is free.
- **"34" counts the residual blocks.** It counts convolutions plus the head: `16` blocks of two, one stem convolution, one linear layer.


<!-- dd:dd-kp-cnn-feature-extraction -->

## Pretrained weights and feature extraction

`cnn.feature-extraction`


<!-- dd:dd-seg-cnn-feature-extraction-0 -->

### A module's state is an ordered mapping of named tensors


Every module can hand you its *state dict*: a mapping from a dotted name to a tensor, one entry per parameter **and** per (persistent) buffer, in registration order. A Sequential chain of a convolution and a batch norm therefore has six entries — the convolution's weight, the batch norm's weight and bias, and the batch norm's three buffers (running mean, running variance, batch counter) — even though only three of them are parameters. The names are the path through the children: `"1.running_mean"` is the `running_mean` of child `"1"`.

Two models built the same way — the same layers registered in the same order — have state dicts of the same length whose entries line up position by position, whatever the names. Equal lengths and shapes alone do not prove that: the copy is only right when you know the two registration orders match. Python's `zip` walks two sequences side by side, yielding a pair at each step, which is exactly how you line up "my name" with "their name".


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nm=t.nn.Sequential(t.nn.Conv2d(1,2,kernel_size=3,bias=False),t.nn.BatchNorm2d(2))\nsd=m.state_dict()\nprint(list(sd))\nprint(sd["1.running_var"])\nprint(list(zip(["a","b","c"],[1,2,3])))\n# Hidden checks\nassert list(sd)==["0.weight","1.weight","1.bias","1.running_mean","1.running_var","1.num_batches_tracked"]\nassert sd["1.running_var"].tolist()==[1.,1.] and list(zip(["a","b","c"],[1,2,3]))==[("a",1),("b",2),("c",3)]\n', globals()), end='')


Two chains with the same layers but different child names. Predict, before running, how many entries each state dict has and which entries pair up.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('mine=t.nn.Sequential(t.nn.Conv2d(1,2,kernel_size=3,bias=False),t.nn.BatchNorm2d(2))\ntheirs=t.nn.Sequential()\ntheirs.add_module("conv",t.nn.Conv2d(1,2,kernel_size=3,bias=False))\ntheirs.add_module("bn",t.nn.BatchNorm2d(2))\npairs=list(zip(list(mine.state_dict()),list(theirs.state_dict())))\nprint(len(mine.state_dict()), len(theirs.state_dict()))\nprint(pairs[:2])\n# Hidden checks\nassert len(mine.state_dict())==6==len(theirs.state_dict())\nassert pairs[:2]==[("0.weight","conv.weight"),("1.weight","bn.weight")]\n', globals()), end='')




Six and six, and the pairing is by position: `"0.weight"` lines up with `"conv.weight"` although the names share nothing.


<!-- dd:dd-q1426 -->

### Problem 1426 · faded — your turn

Given a module `m`, return a list with one `(name, shape)` tuple per entry of its state mapping, in the mapping's own order, where `shape` is a tuple of Python ints.


In [ ]:
import torch as t

def solve(m):
    sd=m._____()
    return [(k,tuple(sd[k].shape)) for k in sd]


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1426)


In [ ]:
#@title 💡 Solution — Problem 1426
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m):
    sd=m.state_dict()
    return [(k,tuple(sd[k].shape)) for k in sd]


Two chains with the same layers but different child names. Predict, before running, how many entries each state dict has and which entries pair up.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('mine=t.nn.Sequential(t.nn.Conv2d(1,2,kernel_size=3,bias=False),t.nn.BatchNorm2d(2))\ntheirs=t.nn.Sequential()\ntheirs.add_module("conv",t.nn.Conv2d(1,2,kernel_size=3,bias=False))\ntheirs.add_module("bn",t.nn.BatchNorm2d(2))\npairs=list(zip(list(mine.state_dict()),list(theirs.state_dict())))\nprint(len(mine.state_dict()), len(theirs.state_dict()))\nprint(pairs[:2])\n# Hidden checks\nassert len(mine.state_dict())==6==len(theirs.state_dict())\nassert pairs[:2]==[("0.weight","conv.weight"),("1.weight","bn.weight")]\n', globals()), end='')




Six and six, and the pairing is by position: `"0.weight"` lines up with `"conv.weight"` although the names share nothing.


<!-- dd:dd-q1427 -->

### Problem 1427 · faded — your turn

Given two modules `m` and `src` whose state mappings have equally many entries (their names, and even the shapes at a position, may differ), return a list of `(my_name, their_name, same_shape)` tuples pairing the entries of the two state mappings by position, where `same_shape` is whether the two tensors at that position have equal shapes.


In [ ]:
import torch as t

def solve(m,src):
    sd=m._____()
    ps=src._____()
    return [(a,b,sd[a].shape==ps[b].shape) for a,b in _____(list(sd),list(ps))]


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1427)


In [ ]:
#@title 💡 Solution — Problem 1427
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,src):
    sd=m.state_dict()
    ps=src.state_dict()
    return [(a,b,sd[a].shape==ps[b].shape) for a,b in zip(list(sd),list(ps))]


<!-- dd:dd-seg-cnn-feature-extraction-1 -->

### Copying weights by position, and what a parameter is


Loading a state dict writes every tensor in the mapping into the module's tensor of the same name. To copy a pretrained model whose names differ from yours, take your own state dict, replace each of its values with the pretrained value at the same position, and load the result back. The shapes must agree entry by entry — that is the check that the two architectures really match.

A module's *parameters* are the subset of its state that is trained: its weights and biases, not the batch-norm running statistics. Iterating a module's parameters yields those tensors in registration order, which is how you count them or inspect them.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nmine=t.nn.Sequential(t.nn.Conv2d(1,2,kernel_size=3,bias=False),t.nn.BatchNorm2d(2))\ntheirs=t.nn.Sequential()\ntheirs.add_module("conv",t.nn.Conv2d(1,2,kernel_size=3,bias=False))\ntheirs.add_module("bn",t.nn.BatchNorm2d(2))\nsd=mine.state_dict()\nps=theirs.state_dict()\nfor a,b in zip(list(sd),list(ps)):\n    sd[a]=ps[b]\nmine.load_state_dict(sd)\nprint(len(list(mine.parameters())), len(sd))\n# Hidden checks\nassert (mine[0].weight-theirs[0].weight).abs().sum().item()==0\nassert len(list(mine.parameters()))==3 and len(sd)==6\n', globals()), end='')


Fill the pretrained chain with known numbers, copy into a fresh chain, and check that both give the same output. Predict how many parameters the count reports before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('with t.no_grad():\n    for p in theirs.parameters():\n        p.copy_(t.linspace(-1.,1.,p.numel()).reshape(p.shape))\nsd=mine.state_dict()\nps=theirs.state_dict()\nfor a,b in zip(list(sd),list(ps)):\n    sd[a]=ps[b]\nmine.load_state_dict(sd)\nprint(mine[0].weight.reshape(-1)[:3])\n# Hidden checks\nassert (mine[0].weight-theirs[0].weight).abs().sum().item()==0\n', globals()), end='')




The fresh chain now carries the filled numbers, so the two chains agree on any input, and the parameter count is the convolution's `18` weights plus the batch norm's `2 + 2`:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('x=t.arange(16.).reshape(1,1,4,4)\nprint((mine.eval()(x)-theirs.eval()(x)).abs().sum().item(), sum([p.numel() for p in mine.parameters()]))\n# Hidden checks\nassert (mine.eval()(x)-theirs.eval()(x)).abs().sum().item()==0\nassert sum([p.numel() for p in mine.parameters()])==18+2+2\n', globals()), end='')




The difference is zero and the count is `22`.


<!-- dd:dd-q1428 -->

### Problem 1428 · faded — your turn

Copy the weights of `src` into `m`, two modules with the same architecture but possibly different child names: take `m`'s state mapping, replace each of its values by the value of `src`'s state mapping at the same position, load the result into `m`, and return `m`.


In [ ]:
import torch as t

def solve(m,src):
    sd=m._____()
    ps=src._____()
    for a,b in _____(list(sd),list(ps)):
        sd[a]=ps[b]
    m._____(sd)
    return m


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1428)


In [ ]:
#@title 💡 Solution — Problem 1428
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,src):
    sd=m.state_dict()
    ps=src.state_dict()
    for a,b in zip(list(sd),list(ps)):
        sd[a]=ps[b]
    m.load_state_dict(sd)
    return m


Fill the pretrained chain with known numbers, copy into a fresh chain, and check that both give the same output. Predict how many parameters the count reports before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('with t.no_grad():\n    for p in theirs.parameters():\n        p.copy_(t.linspace(-1.,1.,p.numel()).reshape(p.shape))\nsd=mine.state_dict()\nps=theirs.state_dict()\nfor a,b in zip(list(sd),list(ps)):\n    sd[a]=ps[b]\nmine.load_state_dict(sd)\nprint(mine[0].weight.reshape(-1)[:3])\n# Hidden checks\nassert (mine[0].weight-theirs[0].weight).abs().sum().item()==0\n', globals()), end='')




The fresh chain now carries the filled numbers, so the two chains agree on any input, and the parameter count is the convolution's `18` weights plus the batch norm's `2 + 2`:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('x=t.arange(16.).reshape(1,1,4,4)\nprint((mine.eval()(x)-theirs.eval()(x)).abs().sum().item(), sum([p.numel() for p in mine.parameters()]))\n# Hidden checks\nassert (mine.eval()(x)-theirs.eval()(x)).abs().sum().item()==0\nassert sum([p.numel() for p in mine.parameters()])==18+2+2\n', globals()), end='')




The difference is zero and the count is `22`.


<!-- dd:dd-q1429 -->

### Problem 1429 · faded — your turn

Given a module `m`, return a tuple of three Python ints: how many parameter tensors it has, how many entries its state mapping has, and the total number of values across its parameters.


In [ ]:
import torch as t

def solve(m):
    ps=list(m._____())
    return (len(ps), len(m._____()), sum([p.numel() for p in ps]))


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1429)


In [ ]:
#@title 💡 Solution — Problem 1429
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m):
    ps=list(m.parameters())
    return (len(ps), len(m.state_dict()), sum([p.numel() for p in ps]))


<!-- dd:dd-seg-cnn-feature-extraction-2 -->

### Freeze everything, then replace the head


Feature extraction keeps a pretrained network's body fixed and trains only a new final layer. Every parameter carries a `requires_grad` flag saying whether gradients should flow into it; calling a module's `requires_grad_(False)` clears the flag on every parameter below it in one go. Replacing the last child of the head with a freshly constructed linear layer then gives you a layer whose parameters are new — and new parameters are created with the flag set — so after the two steps exactly the new head is trainable. Freezing stops gradient updates to those parameters; it does not stop a batch norm's running statistics from moving while the module is in training mode, so a frozen body is normally kept in evaluation mode as well.

The order matters: freeze first, replace second. Freezing after replacing would freeze the new head too.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass AveragePool(t.nn.Module):\n    def forward(self,x):\n        return x.mean((2,3))\nnet=t.nn.Sequential(t.nn.Conv2d(3,4,kernel_size=3,padding=1,bias=False),t.nn.BatchNorm2d(4),t.nn.ReLU(),AveragePool(),t.nn.Linear(4,10))\nnet.requires_grad_(False)\nnet[-1]=t.nn.Linear(4,3)\nprint([p.requires_grad for p in net.parameters()])\nprint(net(t.zeros(2,3,8,8)).shape)\n# Hidden checks\nassert [p.requires_grad for p in net.parameters()]==[False,False,False,True,True]\nassert net(t.zeros(2,3,8,8)).shape==(2,3)\n', globals()), end='')


Count trainable versus frozen parameter values after the freeze-and-replace. Predict both numbers before running: the body holds `4*3*9 + 2*4 = 116` values, the new head `4*3 + 3 = 15`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('trainable=[p for p in net.parameters() if p.requires_grad]\nfrozen=[p for p in net.parameters() if not p.requires_grad]\nprint(sum([p.numel() for p in trainable]), sum([p.numel() for p in frozen]))\n# Hidden checks\nassert sum([p.numel() for p in trainable])==15 and sum([p.numel() for p in frozen])==116\n', globals()), end='')




In ResNet34 the same two lines are `requires_grad_(False)` on the whole network and a new linear layer written into `out_layers[-1]`, from `512` features to the new class count.


<!-- dd:dd-q1430 -->

### Problem 1430 · faded — your turn

Return an instance of a subclass of it that builds the network with `1000` classes, freezes every parameter, and then replaces the last module of its `out_layers` child (through the children mapping by name) with a new linear layer from the last output width to `n_classes`, so only the new head is trainable. The lesson's `ResNet` class (arguments: stem width, block counts, output widths, first strides, class count) is provided.


In [ ]:
import torch as t

def solve(c0,n_blocks,out_feats,strides,n_classes):
    class FeatureExtractor(ResNet):
        def __init__(self):
            super().__init__(c0,n_blocks,out_feats,strides,1000)
            self._____(False)
            self._modules["out_layers"][-1]=t.nn.Linear(out_feats[-1],n_classes)
    return FeatureExtractor()


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1430)


In [ ]:
#@title 💡 Solution — Problem 1430
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(c0,n_blocks,out_feats,strides,n_classes):
    class FeatureExtractor(ResNet):
        def __init__(self):
            super().__init__(c0,n_blocks,out_feats,strides,1000)
            self.requires_grad_(False)
            self._modules["out_layers"][-1]=t.nn.Linear(out_feats[-1],n_classes)
    return FeatureExtractor()


Count trainable versus frozen parameter values after the freeze-and-replace. Predict both numbers before running: the body holds `4*3*9 + 2*4 = 116` values, the new head `4*3 + 3 = 15`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('trainable=[p for p in net.parameters() if p.requires_grad]\nfrozen=[p for p in net.parameters() if not p.requires_grad]\nprint(sum([p.numel() for p in trainable]), sum([p.numel() for p in frozen]))\n# Hidden checks\nassert sum([p.numel() for p in trainable])==15 and sum([p.numel() for p in frozen])==116\n', globals()), end='')




In ResNet34 the same two lines are `requires_grad_(False)` on the whole network and a new linear layer written into `out_layers[-1]`, from `512` features to the new class count.


<!-- dd:dd-q1431 -->

### Problem 1431 · faded — your turn

Given a module `m`, return a tuple of two Python ints: the number of values across its trainable parameters (gradient flag set) and across its frozen parameters (flag cleared).


In [ ]:
import torch as t

def solve(m):
    a=sum([p.numel() for p in m._____() if p._____])
    b=sum([p.numel() for p in m._____()])-a
    return (a,b)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1431)


In [ ]:
#@title 💡 Solution — Problem 1431
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m):
    a=sum([p.numel() for p in m.parameters() if p.requires_grad])
    b=sum([p.numel() for p in m.parameters()])-a
    return (a,b)


<!-- dd:dd-q1432 -->

### Problem 1432 · independent

Given a module `m` whose children are registered in order, freeze every parameter of `m` and then make only the parameters of its LAST child trainable again; return `m`.


In [ ]:
import torch as t

def solve(m):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1432)


In [ ]:
#@title 💡 Solution — Problem 1432
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m):
    m.requires_grad_(False)
    names=list(m._modules)
    m._modules[names[-1]].requires_grad_(True)
    return m


<!-- dd:dd-q1433 -->

### Problem 1433 · independent

Copy weights from `src` into `m` by position, but only where the two tensors at a position have the same shape, leaving every other entry of `m` unchanged; return the number of entries copied as a Python int.


In [ ]:
import torch as t

def solve(m,src):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1433)


In [ ]:
#@title 💡 Solution — Problem 1433
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,src):
    sd=m.state_dict()
    ps=src.state_dict()
    n=0
    for a,b in zip(list(sd),list(ps)):
        if sd[a].shape==ps[b].shape:
            sd[a]=ps[b]
            n+=1
    m.load_state_dict(sd)
    return n


<!-- dd:dd-q1434 -->

### Problem 1434 · independent

Two modules `m` and `src` have identical architecture AND identical state names. Copy from `src` into `m` only the state entries whose name starts with the string `prefix` (compare the leading characters of the name), leave the rest of `m` untouched, and return `m`.


In [ ]:
import torch as t

def solve(m,src,prefix):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1434)


In [ ]:
#@title 💡 Solution — Problem 1434
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,src,prefix):
    sd=m.state_dict()
    ps=src.state_dict()
    for k in sd:
        if k[:len(prefix)]==prefix:
            sd[k]=ps[k]
    m.load_state_dict(sd)
    return m


<!-- dd:dd-q1435 -->

### Problem 1435 · independent

Given a network `m` built by the lesson's `ResNet` class (children `in_layers`, `residual_layers`, `out_layers`, the last being a Sequential chain ending in the linear head), freeze all of `m`, then replace the last module of the `out_layers` child — reached from OUTSIDE the class through the children mapping by name — with a new linear layer from that head's input width to `n`; return `m`.


In [ ]:
import torch as t

def solve(m,n):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1435)


In [ ]:
#@title 💡 Solution — Problem 1435
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,n):
    m.requires_grad_(False)
    head=m._modules["out_layers"]
    head[-1]=t.nn.Linear(head[-1].weight.shape[1],n)
    return m


<!-- dd:dd-q1436 -->

### Problem 1436 · independent

Given a module `m`, return a list with one `(child_name, trainable_values)` tuple per direct child, in registration order, where `trainable_values` is the number of values across that child's parameters whose gradient flag is set (a Python int; `0` for a child with no trainable parameters).


In [ ]:
import torch as t

def solve(m):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1436)


In [ ]:
#@title 💡 Solution — Problem 1436
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m):
    out=[]
    for name in m._modules:
        child=m._modules[name]
        out.append((name,sum([p.numel() for p in child.parameters() if p.requires_grad])))
    return out


<!-- dd:dd-q1437 -->

### Problem 1437 · independent

Given a network `m` built by the lesson's `ResNet` class, freeze everything, then make trainable again the last `k` groups of its `residual_layers` child (k may be `0`) and the whole `out_layers` child; return `m`. Reach the children from outside the class through the children mapping by name.


In [ ]:
import torch as t

def solve(m,k):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1437)


In [ ]:
#@title 💡 Solution — Problem 1437
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,k):
    m.requires_grad_(False)
    groups=list(m._modules["residual_layers"])
    for g in groups[len(groups)-k:]:
        g.requires_grad_(True)
    m._modules["out_layers"].requires_grad_(True)
    return m


#### Common mistakes

- **The state dict holds only the parameters.** It also holds buffers such as batch-norm running statistics; parameters are the trained subset.
- **Names must match to copy weights.** Loading matches by name, so you rebuild the mapping with YOUR names and THEIR values, paired by position.
- **Replace the head, then freeze.** Freezing afterwards freezes the new head too; freeze first, replace second.
- **Freezing removes the parameters.** They stay in the state dict and still run in the forward pass; only their gradient flag is cleared.


<!-- dd:dd-kp-cnn-training-loop -->

## The training loop

`cnn.training-loop`


<!-- dd:dd-seg-cnn-training-loop-0 -->

### Backward fills every parameter's gradient


A loss is a single number computed from the model's output. Calling `backward()` on it walks the computation back to every parameter that took part and writes the derivative of the loss with respect to that parameter into the parameter's `grad` attribute — a tensor of the same shape, `None` before the first backward. One step of gradient descent then moves every parameter a little way against its gradient: `p -= lr * p.grad`, done under `no_grad` because the update itself must not be recorded as part of any computation.

The reason the gradient has the parameter's shape is that it holds one derivative per weight value. The reason the update must be wrapped in `no_grad` is that a parameter is a leaf that requires a gradient; changing it in place while recording is an error.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nlin=t.nn.Linear(2,1,bias=False)\nwith t.no_grad():\n    lin.weight.copy_(t.tensor([[1.,2.]]))\nx=t.tensor([[1.,1.]])\nloss=(lin(x)**2).mean()\nloss.backward()\nprint(lin.weight.grad)\nwith t.no_grad():\n    lin.weight-=0.1*lin.weight.grad\nprint(lin.weight)\n# Hidden checks\nassert lin.weight.grad.tolist()==[[6.,6.]] and (lin.weight-t.tensor([[0.4,1.4]])).abs().max().item()<1e-6\n', globals()), end='')


The output is `1*1 + 2*1 = 3`, the loss is `3² = 9`, and the derivative of `out²` with respect to each weight is `2 * out * x = 6`. Predict the loss after the step before running: the new output is `0.4 + 1.4 = 1.8`, so the loss should be `3.24`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(((lin(x)**2).mean()).item())\n# Hidden checks\nassert abs(((lin(x)**2).mean()).item()-3.24)<1e-5\n', globals()), end='')


<!-- dd:dd-q1441 -->

### Problem 1441 · faded — your turn

Given a module `m`, an input batch `x`, a `target` tensor of the same shape as the module's output and a learning rate `lr`, take ONE step of plain gradient descent on the mean squared error between output and target: compute the loss, run backward, and subtract `lr` times each parameter's gradient from that parameter in place without recording the update. Return `m`.


In [ ]:
import torch as t

def solve(m,x,target,lr):
    loss=((m(x)-target)**2).mean()
    loss._____()
    with t.no_grad():
        for p in m.parameters():
            p-=lr*p._____
    return m


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1441)


In [ ]:
#@title 💡 Solution — Problem 1441
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x,target,lr):
    loss=((m(x)-target)**2).mean()
    loss.backward()
    with t.no_grad():
        for p in m.parameters():
            p-=lr*p.grad
    return m


The output is `1*1 + 2*1 = 3`, the loss is `3² = 9`, and the derivative of `out²` with respect to each weight is `2 * out * x = 6`. Predict the loss after the step before running: the new output is `0.4 + 1.4 = 1.8`, so the loss should be `3.24`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(((lin(x)**2).mean()).item())\n# Hidden checks\nassert abs(((lin(x)**2).mean()).item()-3.24)<1e-5\n', globals()), end='')


<!-- dd:dd-q1442 -->

### Problem 1442 · faded — your turn

Given a module `m`, an input batch `x` and a `target` tensor of the output's shape, compute the mean squared error between output and target, run backward, and return the gradient tensor of the module's FIRST parameter (in registration order).


In [ ]:
import torch as t

def solve(m,x,target):
    loss=((m(x)-target)**2).mean()
    loss._____()
    return list(m.parameters())[0]._____


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1442)


In [ ]:
#@title 💡 Solution — Problem 1442
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x,target):
    loss=((m(x)-target)**2).mean()
    loss.backward()
    return list(m.parameters())[0].grad


<!-- dd:dd-seg-cnn-training-loop-1 -->

### An optimizer owns the parameters and applies the step


Writing the update by hand works but scales badly and misses the better update rules. An optimizer object is constructed with the parameters it may change and a learning rate; after `backward()` has filled the gradients, its `step()` applies the rule to every one of those parameters at once. Adam is the rule ARENA uses throughout: it scales each parameter's step by a running estimate of its gradient size, so on the very first step every value with a nonzero gradient moves by almost exactly the learning rate (the step is `lr` times `g / (|g| + eps)`), against the sign of its gradient; a zero gradient moves nothing.

The reason to hand the optimizer the parameters up front is that it keeps per-parameter running statistics between steps; the reason the first Adam step is `±lr` is that those statistics start empty, so the normalised gradient is just its sign.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nlin=t.nn.Linear(2,1,bias=False)\nwith t.no_grad():\n    lin.weight.copy_(t.tensor([[1.,2.]]))\nopt=t.optim.Adam(lin.parameters(),lr=0.1)\nx=t.tensor([[1.,1.]])\nloss=(lin(x)**2).mean()\nloss.backward()\nopt.step()\nprint(lin.weight)\n# Hidden checks\nassert (lin.weight-t.tensor([[0.9,1.9]])).abs().max().item()<1e-4\n', globals()), end='')


The gradient is `[6, 6]` as before, but Adam's first step ignores its size: both weights move by the learning rate `0.1`. Predict the loss after the step before running: output `0.9 + 1.9 = 2.8`, loss `7.84`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(((lin(x)**2).mean()).item())\n# Hidden checks\nassert abs(((lin(x)**2).mean()).item()-7.84)<1e-3\n', globals()), end='')


<!-- dd:dd-q1443 -->

### Problem 1443 · faded — your turn

Given a module `m`, an input batch `x`, a `target` tensor of the output's shape and a learning rate `lr`, build an Adam optimizer over the module's parameters with that learning rate, compute the mean squared error between output and target, run backward, and apply ONE optimizer step. Return `m`.


In [ ]:
import torch as t

def solve(m,x,target,lr):
    opt=t.optim._____(m.parameters(),lr=lr)
    loss=((m(x)-target)**2).mean()
    loss._____()
    opt._____()
    return m


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1443)


In [ ]:
#@title 💡 Solution — Problem 1443
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x,target,lr):
    opt=t.optim.Adam(m.parameters(),lr=lr)
    loss=((m(x)-target)**2).mean()
    loss.backward()
    opt.step()
    return m


The gradient is `[6, 6]` as before, but Adam's first step ignores its size: both weights move by the learning rate `0.1`. Predict the loss after the step before running: output `0.9 + 1.9 = 2.8`, loss `7.84`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(((lin(x)**2).mean()).item())\n# Hidden checks\nassert abs(((lin(x)**2).mean()).item()-7.84)<1e-3\n', globals()), end='')


<!-- dd:dd-q1444 -->

### Problem 1444 · faded — your turn

Given `m`, `x`, a `target` of the output's shape and a learning rate `lr`, return a tuple of two Python floats: the mean squared error before an Adam step and the mean squared error after it (recompute the output after the step).


In [ ]:
import torch as t

def solve(m,x,target,lr):
    opt=t.optim._____(m.parameters(),lr=lr)
    loss=((m(x)-target)**2).mean()
    loss._____()
    opt._____()
    after=((m(x)-target)**2).mean()
    return (loss.item(),after.item())


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1444)


In [ ]:
#@title 💡 Solution — Problem 1444
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x,target,lr):
    opt=t.optim.Adam(m.parameters(),lr=lr)
    loss=((m(x)-target)**2).mean()
    loss.backward()
    opt.step()
    after=((m(x)-target)**2).mean()
    return (loss.item(),after.item())


<!-- dd:dd-seg-cnn-training-loop-2 -->

### Clear the gradients each step, and use the classification loss


`backward()` *adds* to `grad`; it never overwrites. Two backward passes on the same batch without clearing leave twice the gradient, so a loop must call the optimizer's `zero_grad()` once per step — after the step, or before the backward, either is fine as long as it happens every iteration. For classification the loss is cross-entropy: given a `(batch, classes)` matrix of logits and a `(batch,)` tensor of integer labels, it computes the log-sum-exp of each row minus the logit of the true class and averages over the batch — the same formula as the evaluation lesson, in one library call.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nlin=t.nn.Linear(2,1,bias=False)\nwith t.no_grad():\n    lin.weight.copy_(t.tensor([[1.,2.]]))\nopt=t.optim.Adam(lin.parameters(),lr=0.1)\nx=t.tensor([[1.,1.]])\n(lin(x)**2).mean().backward()\n(lin(x)**2).mean().backward()\nprint(lin.weight.grad)\nopt.zero_grad()\nprint(lin.weight.grad)\n# Hidden checks\nassert lin.weight.grad is None or lin.weight.grad.abs().sum().item()==0\n', globals()), end='')


Logits `[1, 2, 3]` with true class `2`: the loss is `log(e¹ + e² + e³) - 3 ≈ 0.4076`. Predict which of the two batches has the larger loss before running: the second row's true class is the smallest logit.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('logits=t.tensor([[1.,2.,3.],[1.,2.,3.]])\ny=t.tensor([2,0])\nprint(t.nn.functional.cross_entropy(logits[:1],y[:1]).item())\nprint(t.nn.functional.cross_entropy(logits,y).item())\n# Hidden checks\nassert abs(t.nn.functional.cross_entropy(logits[:1],y[:1]).item()-0.4076)<1e-3\nassert abs(t.nn.functional.cross_entropy(logits,y).item()-(0.4076+2.4076)/2)<1e-3\n', globals()), end='')




The batch loss is the mean of the two rows, `(0.4076 + 2.4076) / 2`.


<!-- dd:dd-q1445 -->

### Problem 1445 · faded — your turn

Given a classifier `m`, a batch `x`, integer labels `y`, a learning rate `lr` and a step count `k`, build an Adam optimizer and take `k` steps on the cross-entropy loss between the logits and the labels, clearing the gradients every step. Return the list of `k` loss values as Python floats, in order.


In [ ]:
import torch as t

def solve(m,x,y,lr,k):
    opt=t.optim._____(m.parameters(),lr=lr)
    losses=[]
    for step in [0]*k:
        loss=t.nn.functional._____(m(x),y)
        loss._____()
        opt._____()
        opt._____()
        losses.append(loss.item())
    return losses


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1445)


In [ ]:
#@title 💡 Solution — Problem 1445
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x,y,lr,k):
    opt=t.optim.Adam(m.parameters(),lr=lr)
    losses=[]
    for step in [0]*k:
        loss=t.nn.functional.cross_entropy(m(x),y)
        loss.backward()
        opt.step()
        opt.zero_grad()
        losses.append(loss.item())
    return losses


Logits `[1, 2, 3]` with true class `2`: the loss is `log(e¹ + e² + e³) - 3 ≈ 0.4076`. Predict which of the two batches has the larger loss before running: the second row's true class is the smallest logit.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('logits=t.tensor([[1.,2.,3.],[1.,2.,3.]])\ny=t.tensor([2,0])\nprint(t.nn.functional.cross_entropy(logits[:1],y[:1]).item())\nprint(t.nn.functional.cross_entropy(logits,y).item())\n# Hidden checks\nassert abs(t.nn.functional.cross_entropy(logits[:1],y[:1]).item()-0.4076)<1e-3\nassert abs(t.nn.functional.cross_entropy(logits,y).item()-(0.4076+2.4076)/2)<1e-3\n', globals()), end='')




The batch loss is the mean of the two rows, `(0.4076 + 2.4076) / 2`.


<!-- dd:dd-q1446 -->

### Problem 1446 · faded — your turn

Given a classifier `m`, a batch `x` and integer labels `y`, compute the cross-entropy loss and run backward TWICE (recomputing the loss the second time) without clearing anything in between, and return the resulting gradient tensor of the module's first parameter.


In [ ]:
import torch as t

def solve(m,x,y):
    t.nn.functional._____(m(x),y)._____()
    t.nn.functional._____(m(x),y)._____()
    return list(m.parameters())[0]._____


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1446)


In [ ]:
#@title 💡 Solution — Problem 1446
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x,y):
    t.nn.functional.cross_entropy(m(x),y).backward()
    t.nn.functional.cross_entropy(m(x),y).backward()
    return list(m.parameters())[0].grad


<!-- dd:dd-seg-cnn-training-loop-3 -->

### Train mode for the update, eval mode for the measurement


A module has a mode flag. In *train* mode a batch norm normalises with the current batch's statistics and updates its running averages; in *eval* mode it uses the stored running averages and changes nothing. `train()` and `eval()` set the flag on the module and every child. The loop therefore switches to train mode before the epoch's updates and to eval mode before measuring accuracy, and the measurement runs under `no_grad` because no update follows it.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nnet=t.nn.Sequential(t.nn.Conv2d(1,2,kernel_size=3,padding=1,bias=False),t.nn.BatchNorm2d(2),t.nn.ReLU(),t.nn.Flatten(),t.nn.Linear(32,3))\ndef evaluate(m,test_batches):\n    m.eval()\n    correct=0\n    total=0\n    with t.no_grad():\n        for x,y in test_batches:\n            correct+=(m(x).argmax(dim=1)==y).sum().item()\n            total+=len(y)\n    return correct/total\nx=t.linspace(-1.,1.,2*16).reshape(2,1,4,4)\nprint(net.training, net.train()(x).sum().item(), net.eval()(x).sum().item(), net.training)\n# Hidden checks\nassert net.training is False\n', globals()), end='')


The full loop, one epoch at a time: train mode, then for every training batch compute logits, loss, backward, step, zero the gradients and record the loss; then eval mode and one accuracy number for the epoch. Predict how many losses and how many accuracies two epochs over three training batches produce before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('def train(m,train_batches,test_batches,lr,epochs):\n    opt=t.optim.Adam(m.parameters(),lr=lr)\n    loss_list=[]\n    acc_list=[]\n    for epoch in [0]*epochs:\n        m.train()\n        for x,y in train_batches:\n            loss=t.nn.functional.cross_entropy(m(x),y)\n            loss.backward()\n            opt.step()\n            opt.zero_grad()\n            loss_list.append(loss.item())\n        acc_list.append(evaluate(m,test_batches))\n    return loss_list,acc_list\nte=[(t.linspace(0.,1.,4*16).reshape(4,1,4,4),t.tensor([0,1,2,0]))]\nprint(evaluate(net,te))\n# Hidden checks\nassert 0<=evaluate(net,te)<=1\n', globals()), end='')




The untrained accuracy is whatever the random head happens to guess. Now three training batches, two epochs:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('tr=[(t.linspace(-1.,1.,4*16).reshape(4,1,4,4)*(k+1)/3,(t.arange(4)+k)%3) for k in [0,1,2]]\nlosses,accs=train(net,tr,te,0.05,2)\nprint(len(losses), len(accs), accs)\n# Hidden checks\nassert len(losses)==6 and len(accs)==2\n', globals()), end='')




Six losses (one per batch per epoch) and two accuracies (one per epoch).


<!-- dd:dd-q1447 -->

### Problem 1447 · faded — your turn

Given a classifier `m` and `test_batches`, a list of `(x, y)` pairs, put the module in evaluation mode and, without recording gradients, count how many predictions (largest logit per example) equal their label across every batch. Return the accuracy `correct / total` as a Python float.


In [ ]:
import torch as t

def solve(m,test_batches):
    m._____()
    correct=0
    total=0
    with t.no_grad():
        for x,y in test_batches:
            correct+=(m(x).argmax(dim=1)==y).sum().item()
            total+=len(y)
    return correct/total


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1447)


In [ ]:
#@title 💡 Solution — Problem 1447
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,test_batches):
    m.eval()
    correct=0
    total=0
    with t.no_grad():
        for x,y in test_batches:
            correct+=(m(x).argmax(dim=1)==y).sum().item()
            total+=len(y)
    return correct/total


The full loop, one epoch at a time: train mode, then for every training batch compute logits, loss, backward, step, zero the gradients and record the loss; then eval mode and one accuracy number for the epoch. Predict how many losses and how many accuracies two epochs over three training batches produce before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('def train(m,train_batches,test_batches,lr,epochs):\n    opt=t.optim.Adam(m.parameters(),lr=lr)\n    loss_list=[]\n    acc_list=[]\n    for epoch in [0]*epochs:\n        m.train()\n        for x,y in train_batches:\n            loss=t.nn.functional.cross_entropy(m(x),y)\n            loss.backward()\n            opt.step()\n            opt.zero_grad()\n            loss_list.append(loss.item())\n        acc_list.append(evaluate(m,test_batches))\n    return loss_list,acc_list\nte=[(t.linspace(0.,1.,4*16).reshape(4,1,4,4),t.tensor([0,1,2,0]))]\nprint(evaluate(net,te))\n# Hidden checks\nassert 0<=evaluate(net,te)<=1\n', globals()), end='')




The untrained accuracy is whatever the random head happens to guess. Now three training batches, two epochs:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('tr=[(t.linspace(-1.,1.,4*16).reshape(4,1,4,4)*(k+1)/3,(t.arange(4)+k)%3) for k in [0,1,2]]\nlosses,accs=train(net,tr,te,0.05,2)\nprint(len(losses), len(accs), accs)\n# Hidden checks\nassert len(losses)==6 and len(accs)==2\n', globals()), end='')




Six losses (one per batch per epoch) and two accuracies (one per epoch).


<!-- dd:dd-q1448 -->

### Problem 1448 · faded — your turn

Train the classifier `m` for `epochs` epochs and return its batch losses and per-epoch accuracies. Given lists of `(x, y)` pairs `train_batches` and `test_batches` (logits of shape `(B, C)`, labels of shape `(B,)`) and a learning rate `lr`, build an Adam optimizer over the module's parameters and, for each epoch: switch to train mode; for every training batch compute the cross-entropy loss of the logits against the labels, run backward, step, clear the gradients and record the loss as a float; then switch to evaluation mode and, without recording gradients, record the accuracy over all test batches. Return the tuple `(loss_list, acc_list)`.


In [ ]:
import torch as t

def solve(m,train_batches,test_batches,lr,epochs):
    opt=t.optim._____(m.parameters(),lr=lr)
    loss_list=[]
    acc_list=[]
    for epoch in [0]*epochs:
        m._____()
        for x,y in train_batches:
            loss=t.nn.functional._____(m(x),y)
            loss._____()
            opt._____()
            opt._____()
            loss_list.append(loss.item())
        m._____()
        correct=0
        total=0
        with t.no_grad():
            for x,y in test_batches:
                correct+=(m(x).argmax(dim=1)==y).sum().item()
                total+=len(y)
        acc_list.append(correct/total)
    return (loss_list,acc_list)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1448)


In [ ]:
#@title 💡 Solution — Problem 1448
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,train_batches,test_batches,lr,epochs):
    opt=t.optim.Adam(m.parameters(),lr=lr)
    loss_list=[]
    acc_list=[]
    for epoch in [0]*epochs:
        m.train()
        for x,y in train_batches:
            loss=t.nn.functional.cross_entropy(m(x),y)
            loss.backward()
            opt.step()
            opt.zero_grad()
            loss_list.append(loss.item())
        m.eval()
        correct=0
        total=0
        with t.no_grad():
            for x,y in test_batches:
                correct+=(m(x).argmax(dim=1)==y).sum().item()
                total+=len(y)
        acc_list.append(correct/total)
    return (loss_list,acc_list)


<!-- dd:dd-q1449 -->

### Problem 1449 · independent

Without any optimizer object, take `k` steps of plain gradient descent (learning rate `lr`) on the mean squared error between the output of `m` on `x` and `target`, clearing the module's gradients after each in-place update. Return `m`.


In [ ]:
import torch as t

def solve(m,x,target,lr,k):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1449)


In [ ]:
#@title 💡 Solution — Problem 1449
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x,target,lr,k):
    for step in [0]*k:
        loss=((m(x)-target)**2).mean()
        loss.backward()
        with t.no_grad():
            for p in m.parameters():
                p-=lr*p.grad
        m.zero_grad()
    return m


<!-- dd:dd-q1450 -->

### Problem 1450 · independent

Run ONE epoch of training in train mode: an Adam optimizer over the parameters of `m`, and for each `(x, y)` in `train_batches` the cross-entropy loss, backward, step and clearing of the gradients. Return the list of per-batch losses as Python floats.


In [ ]:
import torch as t

def solve(m,train_batches,lr):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1450)


In [ ]:
#@title 💡 Solution — Problem 1450
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,train_batches,lr):
    opt=t.optim.Adam(m.parameters(),lr=lr)
    m.train()
    losses=[]
    for x,y in train_batches:
        loss=t.nn.functional.cross_entropy(m(x),y)
        loss.backward()
        opt.step()
        opt.zero_grad()
        losses.append(loss.item())
    return losses


<!-- dd:dd-q1451 -->

### Problem 1451 · independent

Given a classifier `m`, a batch `x` and integer labels `y`, run backward on the cross-entropy loss once and return a list with the SUM of each parameter's gradient as a Python float, one entry per parameter in registration order.


In [ ]:
import torch as t

def solve(m,x,y):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1451)


In [ ]:
#@title 💡 Solution — Problem 1451
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x,y):
    t.nn.functional.cross_entropy(m(x),y).backward()
    return [p.grad.sum().item() for p in m.parameters()]


<!-- dd:dd-q1452 -->

### Problem 1452 · independent

Given a classifier `m` and `test_batches`, a list of `(x, y)` pairs, return a list with the accuracy of each batch on its own as a Python float, measured in evaluation mode without recording gradients.


In [ ]:
import torch as t

def solve(m,test_batches):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1452)


In [ ]:
#@title 💡 Solution — Problem 1452
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,test_batches):
    m.eval()
    accs=[]
    with t.no_grad():
        for x,y in test_batches:
            hits=(m(x).argmax(dim=1)==y).to(t.float32)
            accs.append(hits.mean().item())
    return accs


<!-- dd:dd-q1453 -->

### Problem 1453 · independent

Given a module `m` containing a batch norm and a batch `x`, return a tuple of two tensors computed without recording gradients: the output in train mode, then the output in evaluation mode.


In [ ]:
import torch as t

def solve(m,x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1453)


In [ ]:
#@title 💡 Solution — Problem 1453
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x):
    with t.no_grad():
        a=m.train()(x)
        b=m.eval()(x)
    return (a,b)


<!-- dd:dd-q1454 -->

### Problem 1454 · independent

Given a classifier `m` in evaluation mode, a batch `x` and integer labels `y`, return the cross-entropy loss of the logits against the labels as a Python float, computed without recording gradients.


In [ ]:
import torch as t

def solve(m,x,y):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1454)


In [ ]:
#@title 💡 Solution — Problem 1454
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x,y):
    with t.no_grad():
        return t.nn.functional.cross_entropy(m(x),y).item()


#### Common mistakes

- **Backward overwrites the gradient.** It accumulates; without a zero-grad every step, step `n` uses the sum of `n` gradients.
- **The optimizer computes the gradient.** Backward does; the optimizer only applies the update rule to the gradients it finds.
- **Adam's first step is proportional to the gradient.** Every value with a nonzero gradient moves by almost exactly the learning rate whatever the gradient's size; the running statistics only shape later steps.
- **Eval mode disables gradients.** It only changes how batch norm and similar layers behave; `no_grad` is a separate switch.


<!-- dd:dd-kp-cnn-forward-hooks -->

## Forward hooks

`cnn.forward-hooks`


<!-- dd:dd-seg-cnn-forward-hooks-0 -->

### A forward hook runs after a module's forward, and its handle removes it


A *forward hook* is a function you attach to a module; every time the module finishes its forward pass, the hook is called with three arguments — the module, the tuple of inputs it received, and the output it produced — and it can inspect any of them. Registering returns a *handle*; calling the handle's `remove()` detaches the hook again. Hooks stack: register two and both fire, and a hook you forget to remove keeps firing on every later call.

If the hook returns a value, that value **replaces** the module's output for the caller — the way to patch a layer without editing its forward. A hook that only records things returns nothing, and the output passes through unchanged.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nnet=t.nn.Sequential(t.nn.Linear(4,3),t.nn.ReLU(),t.nn.Linear(3,2))\nshapes=[]\ndef record(module,inputs,output):\n    shapes.append(tuple(output.shape))\nhandles=[child.register_forward_hook(record) for child in net]\nnet(t.zeros(5,4))\nprint(shapes)\nfor h in handles:\n    h.remove()\nnet(t.zeros(5,4))\nprint(len(shapes))\n# Hidden checks\nassert shapes==[(5,3),(5,3),(5,2)] and len(shapes)==3\n', globals()), end='')


Attach a hook that doubles the output of the ReLU. Predict, before running, how the final output compares with and without the hook: the second linear layer sees doubled activations, so its bias-free part doubles.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('with t.no_grad():\n    for p in net.parameters():\n        p.copy_(t.linspace(-1.,1.,p.numel()).reshape(p.shape))\nx=t.ones(1,4)\nbefore=net(x)\ndef double(module,inputs,output):\n    return output*2\nh=net[1].register_forward_hook(double)\nduring=net(x)\nprint(before, during)\n# Hidden checks\nassert (during-before).abs().max().item()>0\n', globals()), end='')




The hook's returned tensor became the ReLU's output, so the final logits changed. Remove the handle and the chain is back to normal:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('h.remove()\nprint(net(x))\n# Hidden checks\nassert (net(x)-before).abs().max().item()==0\n', globals()), end='')




The output equals the first print: once the handle is removed, the hook is gone.


<!-- dd:dd-q1458 -->

### Problem 1458 · faded — your turn

Given a Sequential chain `m` and an input `x`, attach a forward hook to EACH direct child that appends the child's output shape (a tuple of Python ints) to a list; run the chain on `x` once; detach every hook through its handle; run the chain once more; and return the list — one entry per child.


In [ ]:
import torch as t

def solve(m,x):
    shapes=[]
    def record(module,inputs,output):
        shapes.append(tuple(output.shape))
    handles=[child._____(record) for child in m]
    m(x)
    for h in handles:
        h._____()
    m(x)
    return shapes


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1458)


In [ ]:
#@title 💡 Solution — Problem 1458
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x):
    shapes=[]
    def record(module,inputs,output):
        shapes.append(tuple(output.shape))
    handles=[child.register_forward_hook(record) for child in m]
    m(x)
    for h in handles:
        h.remove()
    m(x)
    return shapes


Attach a hook that doubles the output of the ReLU. Predict, before running, how the final output compares with and without the hook: the second linear layer sees doubled activations, so its bias-free part doubles.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('with t.no_grad():\n    for p in net.parameters():\n        p.copy_(t.linspace(-1.,1.,p.numel()).reshape(p.shape))\nx=t.ones(1,4)\nbefore=net(x)\ndef double(module,inputs,output):\n    return output*2\nh=net[1].register_forward_hook(double)\nduring=net(x)\nprint(before, during)\n# Hidden checks\nassert (during-before).abs().max().item()>0\n', globals()), end='')




The hook's returned tensor became the ReLU's output, so the final logits changed. Remove the handle and the chain is back to normal:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('h.remove()\nprint(net(x))\n# Hidden checks\nassert (net(x)-before).abs().max().item()==0\n', globals()), end='')




The output equals the first print: once the handle is removed, the hook is gone.


<!-- dd:dd-q1459 -->

### Problem 1459 · faded — your turn

Given a Sequential chain `m`, an input `x` and an index `k`, capture the OUTPUT tensor of the `k`-th direct child (position `k` in the chain) with a forward hook while running the chain on `x`, detach the hook, and return the captured tensor.


In [ ]:
import torch as t

def solve(m,x,k):
    captured=[]
    def grab(module,inputs,output):
        captured.append(output)
    h=list(m)[k]._____(grab)
    m(x)
    h._____()
    return captured[0]


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1459)


In [ ]:
#@title 💡 Solution — Problem 1459
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x,k):
    captured=[]
    def grab(module,inputs,output):
        captured.append(output)
    h=list(m)[k].register_forward_hook(grab)
    m(x)
    h.remove()
    return captured[0]


<!-- dd:dd-seg-cnn-forward-hooks-1 -->

### `apply` visits every module in the tree, including the root


Attaching a hook to each child by hand misses nested children — a block inside a group inside a network. A module's `apply(fn)` calls `fn` on every module in the tree — each child's own subtree is finished before that child, siblings go in registration order, and the module itself comes last — so one small function can register a hook on all of them at once. The standard use is a NaN detector: a hook that checks its output with `isnan` (elementwise, `True` where a value is not-a-number) and records the module when any entry is NaN. NaN usually propagates, so modules downstream of the culprit tend to be flagged too; the first record is the earliest module whose OUTPUT held a NaN, and comparing its input tells you whether it produced the NaN or merely passed one on.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass NanModule(t.nn.Module):\n    def forward(self,x):\n        return x*float("nan")\nnet=t.nn.Sequential(t.nn.Identity(),t.nn.Sequential(NanModule(),t.nn.Identity()),t.nn.Identity())\ncount=[0]\ndef tally(module):\n    count[0]+=1\nnet.apply(tally)\nprint(count[0])\nprint(t.isnan(t.tensor([1.,float("nan"),3.])))\n# Hidden checks\nassert count[0]==6 and t.isnan(t.tensor([1.,float("nan"),3.])).tolist()==[False,True,False]\n', globals()), end='')


Six modules: three top-level children, the two inside the nested chain, and the root. Now the detector. Predict how many modules flag NaN before running: the NaN module, the identity after it inside the nested chain, the nested chain itself, the outer identity after it, and the root — five.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('flagged=[]\ndef check(module,inputs,output):\n    if t.isnan(output).any():\n        flagged.append(module)\nhandles=[]\ndef add_hook(module):\n    handles.append(module.register_forward_hook(check))\nnet.apply(add_hook)\nnet(t.ones(3))\nprint(len(flagged), type(flagged[0]) is NanModule)\n# Hidden checks\nassert len(flagged)==5 and type(flagged[0]) is NanModule\n', globals()), end='')




Five, and the first flagged module is the source. Keeping the handles in a list is what lets you remove every hook afterwards — a second run then records nothing new:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('for h in handles:\n    h.remove()\nnet(t.ones(3))\nprint(len(flagged))\n# Hidden checks\nassert len(flagged)==5\n', globals()), end='')


<!-- dd:dd-q1460 -->

### Problem 1460 · faded — your turn

Given a module `m`, return the number of modules `apply` visits — every module in the tree, nested ones included, plus `m` itself — as a Python int.


In [ ]:
import torch as t

def solve(m):
    count=[0]
    def tally(module):
        count[0]+=1
    m._____(tally)
    return count[0]


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1460)


In [ ]:
#@title 💡 Solution — Problem 1460
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m):
    count=[0]
    def tally(module):
        count[0]+=1
    m.apply(tally)
    return count[0]


Six modules: three top-level children, the two inside the nested chain, and the root. Now the detector. Predict how many modules flag NaN before running: the NaN module, the identity after it inside the nested chain, the nested chain itself, the outer identity after it, and the root — five.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('flagged=[]\ndef check(module,inputs,output):\n    if t.isnan(output).any():\n        flagged.append(module)\nhandles=[]\ndef add_hook(module):\n    handles.append(module.register_forward_hook(check))\nnet.apply(add_hook)\nnet(t.ones(3))\nprint(len(flagged), type(flagged[0]) is NanModule)\n# Hidden checks\nassert len(flagged)==5 and type(flagged[0]) is NanModule\n', globals()), end='')




Five, and the first flagged module is the source. Keeping the handles in a list is what lets you remove every hook afterwards — a second run then records nothing new:


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('for h in handles:\n    h.remove()\nnet(t.ones(3))\nprint(len(flagged))\n# Hidden checks\nassert len(flagged)==5\n', globals()), end='')


<!-- dd:dd-q1461 -->

### Problem 1461 · faded — your turn

Given a module `m` and an input `x`, use `apply` to attach to EVERY module in the tree a forward hook that records the module when any entry of its OUTPUT is not-a-number; run the model on `x`; return how many modules were recorded, as a Python int.


In [ ]:
import torch as t

def solve(m,x):
    flagged=[]
    def check(module,inputs,output):
        if t._____(output).any():
            flagged.append(module)
    def add_hook(module):
        module._____(check)
    m._____(add_hook)
    m(x)
    return len(flagged)


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1461)


In [ ]:
#@title 💡 Solution — Problem 1461
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x):
    flagged=[]
    def check(module,inputs,output):
        if t.isnan(output).any():
            flagged.append(module)
    def add_hook(module):
        module.register_forward_hook(check)
    m.apply(add_hook)
    m(x)
    return len(flagged)


<!-- dd:dd-q1462 -->

### Problem 1462 · independent

Given a Sequential chain `m` and an input `x`, return a list with one `(min, max)` tuple of Python floats per direct child, the smallest and largest value of that child's output, captured with forward hooks that are detached before returning.


In [ ]:
import torch as t

def solve(m,x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1462)


In [ ]:
#@title 💡 Solution — Problem 1462
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x):
    stats=[]
    def record(module,inputs,output):
        stats.append((output.min().item(),output.max().item()))
    handles=[child.register_forward_hook(record) for child in m]
    m(x)
    for h in handles:
        h.remove()
    return stats


<!-- dd:dd-q1463 -->

### Problem 1463 · independent

Given a Sequential chain `m`, an input `x` and an index `k`, attach a forward hook to the `k`-th direct child that appends its output to a list, run the chain on `x`, detach the hook through its handle, run the chain on `x` again, and return a tuple: the first captured tensor and the number of captures (a Python int).


In [ ]:
import torch as t

def solve(m,x,k):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1463)


In [ ]:
#@title 💡 Solution — Problem 1463
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x,k):
    captured=[]
    def grab(module,inputs,output):
        captured.append(output)
    h=list(m)[k].register_forward_hook(grab)
    m(x)
    h.remove()
    m(x)
    return (captured[0],len(captured))


<!-- dd:dd-q1464 -->

### Problem 1464 · independent

Return the output of the Sequential chain `m` on `x` while the direct child at position `k` has its output rescaled. Attach to that child a forward hook that replaces the child's output with the output multiplied by the scalar `c`; run the chain on `x`; detach the hook; return the tensor the chain produced while the hook was attached. Later children see the rescaled tensor, so the result is not simply the plain output times `c`.


In [ ]:
import torch as t

def solve(m,x,k,c):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1464)


In [ ]:
#@title 💡 Solution — Problem 1464
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x,k,c):
    def scale(module,inputs,output):
        return output*c
    h=list(m)[k].register_forward_hook(scale)
    y=m(x)
    h.remove()
    return y


<!-- dd:dd-q1465 -->

### Problem 1465 · independent

Given a module `m` and an input `x`, use `apply` to attach a forward hook to EVERY module in the tree that appends the output shape (a tuple of Python ints) to a list; run the model on `x`; return the list in the order the hooks fired.


In [ ]:
import torch as t

def solve(m,x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1465)


In [ ]:
#@title 💡 Solution — Problem 1465
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x):
    shapes=[]
    def record(module,inputs,output):
        shapes.append(tuple(output.shape))
    def add_hook(module):
        module.register_forward_hook(record)
    m.apply(add_hook)
    m(x)
    return shapes


<!-- dd:dd-q1466 -->

### Problem 1466 · independent

Given a Sequential chain `m` and an input `x`, return a list of Python booleans, one per direct child, saying whether that child's OUTPUT contains any not-a-number entry, captured with forward hooks that are detached before returning.


In [ ]:
import torch as t

def solve(m,x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1466)


In [ ]:
#@title 💡 Solution — Problem 1466
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m,x):
    flags=[]
    def check(module,inputs,output):
        flags.append(t.isnan(output).any().item())
    handles=[child.register_forward_hook(check) for child in m]
    m(x)
    for h in handles:
        h.remove()
    return flags


<!-- dd:dd-q1467 -->

### Problem 1467 · independent

Given a module `m`, return the number of modules strictly BELOW it in the tree (every nested module, at any depth, but not `m` itself), as a Python int, using `apply`.


In [ ]:
import torch as t

def solve(m):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1467)


In [ ]:
#@title 💡 Solution — Problem 1467
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(m):
    count=[0]
    def tally(module):
        count[0]+=1
    m.apply(tally)
    return count[0]-1


#### Common mistakes

- **A hook changes the output only if it edits it in place.** A returned value replaces the output; a hook that returns nothing leaves it unchanged.
- **Hooks vanish after one call.** They fire on every forward until their handle is removed.
- **`apply` visits only the direct children.** It walks the whole tree, children before parents, and ends with the module itself.
- **Only the NaN source is flagged.** NaN usually propagates, so downstream modules are flagged too; the first record is the first module whose output held a NaN, which is the source only if its input was clean.
